# PakFace — Face Recognition for the Pakistani Context
**Nemo's Deep Dive · AI Odyssey (TechFest '26) · Round 2**

End-to-end pipeline: **SCRFD detection (retry cascade) → 5-point alignment → quality assessment → adaptive
enhancement → AdaFace IR-101 + ArcFace R100 ensemble with flip TTA → cosine score fusion → F1-calibrated,
quality-binned thresholds**, plus fine-tuning on our own consented Pakistani data, full metrics, per-group
fairness, ablation, stress tests, 1:N identification and group-photo search.

**How to run (Google Colab):** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom.
Part 1 recreates the code, Part 2 installs and checks it, Part 3 loads data, Part 4 trains and evaluates,
Part 5 runs the hidden-benchmark predictions. After a runtime restart, re-run the **Setup** cell first.

> This notebook is generated from the project repository by `tools/build_notebook.py`, so the code here is
> identical to the code in the repo.

## 1. Strategy in one paragraph

On a hidden benchmark, the winning moves are (1) the strongest *pretrained* backbones for low-quality
faces, (2) a pipeline that never fails on ugly inputs, and (3) a threshold calibrated on data that looks
like the benchmark. Fine-tuning on a few thousand images is the *riskiest* step: it can easily make the
model worse on new people. Fine-tuning on our own collected data is a **required deliverable**, so we always
produce and report a fine-tuned model, but we only *ship* it if it beats the pretrained model on unseen
identities, and we always keep one backbone frozen in the ensemble as an anchor. Everything is
measured on **identity-disjoint** Pakistani test data from our own consented collection.

**Hidden benchmark format:** the rulebook's tagline ("pick a single face out of an entire school", "a sea of
faces, one to find") hints at 1:N identification, while the metric list (FAR/FRR/EER) is verification. We
cover both: `04_evaluate` reports verification metrics *and* rank-1/rank-5 + open-set identification, and
`predict.py` has `pairs`, `identify`, `search` and `one` modes. **Group photos** are handled automatically
(`--multi-face auto`): an image with 2+ faces is split into one entry per face, while single-person images
keep the normal single-face pipeline. `search` finds one person across group / classroom / CCTV photos and
can draw boxes on the matches (`--draw`), which also makes a good live demo.

**Ranking metric is F1** (organisers), but they also look at the other metrics. So thresholds are fitted to
**maximise F1 on validation pairs** (`--method max_f1`, the default), and every other metric is still reported.

| Stage | Choice | Why it helps (say this in the presentation) |
|---|---|---|
| Detection | SCRFD-10G (InsightFace `buffalo_l`), threshold 0.35 | RetinaFace-family detector with 5 landmarks, strong on small/occluded faces |
| Retry cascade | pad → 2× upscale → brighten → whole-image fallback | tight crops, tiny CCTV faces and dark images otherwise produce *no face* = guaranteed error |
| Alignment | similarity transform to ArcFace 112×112 template | backbones were trained on exactly this geometry |
| Quality | sharpness, brightness, contrast, face size, det score, yaw → score in [0,1] | routes enhancement, weights templates, picks threshold bin; never rejects a benchmark pair |
| Enhancement | adaptive gamma + CLAHE (+ mild denoise), **fused** with original embedding | recovers dark/backlit faces without throwing away the original signal; no GAN restoration (it hallucinates identity) |
| Backbone 1 | **AdaFace IR-101, WebFace12M** | quality-adaptive margin — built for blurry, low-res faces (best on IJB-S/TinyFace) |
| Backbone 2 | **ArcFace R100, Glint360K** (`antelopev2`) | different training data + loss → complementary errors |
| TTA | horizontal flip, norm-weighted | cheap, consistent +gain |
| Metric | cosine similarity, weighted score fusion | the angular-margin training objective optimises cosine; weights tuned on val |
| Threshold | max-accuracy threshold fitted on val, **per quality bin** (shrunk to global) | low-quality pairs have compressed genuine scores and need a lower threshold |
| Fine-tuning | AdaFace head, last 25 % unfrozen, frozen BN, distillation to original, heavy degradation + occlusion aug | adapts to Pakistani faces/conditions without forgetting |
| Anti-spoof (optional) | MiniFASNet ONNX wrapper | deployment story only — never applied to benchmark scoring |

**On "100 % accuracy":** aim for it, but don't promise it. Benchmarks with CCTV-quality, masked and
profile faces have a few genuinely ambiguous pairs; strong systems typically land in the high 90s on
mixed-quality Pakistani data. What separates teams is the last 1–3 %, which is exactly what the retry
cascade, ensemble and calibrated thresholds target.

## Setup

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
PROJECT = "/content/pakface" if IN_COLAB else ("/kaggle/working/pakface" if IN_KAGGLE else os.path.abspath("pakface_nb"))
for d in ['configs', 'docs', 'pakface', 'scripts', 'tests', 'webapp']:
    os.makedirs(os.path.join(PROJECT, d), exist_ok=True)
os.chdir(PROJECT)
print("environment:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local", "| project dir:", PROJECT)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"

# Part 1 — Source code
Each cell below writes one file of the project. Run them all once per session.

## Project files

### `requirements.txt`

In [ ]:
%%writefile requirements.txt
# Kaggle / Colab (GPU). On Kaggle: pip install -q -r requirements.txt
insightface==0.7.3
onnxruntime-gpu>=1.17      # use `onnxruntime` instead on a CPU-only machine
transformers>=4.40
huggingface_hub>=0.23
safetensors
omegaconf                  # some CVLFace wrappers import it; harmless otherwise
fvcore                     # imported by the CVLFace iresnet model code (AdaFace IR-101)
torch>=2.1
torchvision
opencv-python-headless>=4.8
numpy
pandas
scikit-learn
scipy
matplotlib
tqdm
tabulate
Pillow
gradio>=6.0                # demo web app (python app.py)

### `app.py`
Nemo's Deep Dive face-matching demo (Gradio web app).

In [ ]:
%%writefile app.py
"""Nemo's Deep Dive face-matching demo (Gradio web app).

    python app.py                 # http://localhost:7860 (webcam works on localhost)
    python app.py --share         # also a public https link: Colab / Kaggle, or a phone on another network
    python app.py --ctx -1        # force CPU (default: GPU 0 if one is available)

Loads the same frozen pipeline as scripts/predict.py (configs/pipeline.json + results/calibration.json).
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from webapp.engine import FaceEngine  # noqa: E402
from webapp.ui import CSS, HEAD, build, theme  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/pipeline.json")
    ap.add_argument("--calibration", default="results/calibration.json")
    ap.add_argument("--ctx", type=int, default=None, help="GPU id, -1 for CPU (default: GPU 0 if available)")
    ap.add_argument("--share", action="store_true", help="create a public https link")
    ap.add_argument("--host", default="127.0.0.1", help="0.0.0.0 to allow other devices on the same network")
    ap.add_argument("--port", type=int, default=None, help="default: first free port from 7860")
    args = ap.parse_args()

    if args.ctx is None:
        import torch

        args.ctx = 0 if torch.cuda.is_available() else -1
    print(f"loading face models on {'GPU' if args.ctx >= 0 else 'CPU'} (first launch can take a minute)...")
    engine = FaceEngine(args.config, args.calibration, args.ctx)
    if not engine.calibrated:
        print(f"WARNING: {args.calibration} not found -> using the uncalibrated default threshold of 0.30")
    build(engine).launch(share=args.share, server_name=args.host, server_port=args.port, theme=theme(), css=CSS,
                         head=HEAD, inbrowser=not args.share, footer_links=[])


if __name__ == "__main__":
    main()

## Configuration

### `configs/pipeline.json`

In [ ]:
%%writefile configs/pipeline.json
{
  "detector_pack": "buffalo_l",
  "det_size": 640,
  "det_thresh": 0.35,
  "pad_retry": true,
  "upscale_retry": true,
  "enhance_for_detection": true,
  "use_alignment": true,
  "use_enhancement": true,
  "enhancement_fusion": true,
  "use_flip_tta": true,
  "models": [
    {
      "name": "adaface_ir101",
      "kind": "adaface_hf",
      "path": "models/adaface_ir101_webface12m",
      "repo_id": "minchul/cvlface_adaface_ir101_webface12m",
      "channel_order": "rgb",
      "weight": 1.0,
      "finetuned_ckpt": ""
    },
    {
      "name": "glint_r100",
      "kind": "insightface_onnx",
      "path": "~/.insightface/models/antelopev2/glintr100.onnx",
      "repo_id": "",
      "channel_order": "rgb",
      "weight": 1.0,
      "finetuned_ckpt": ""
    }
  ],
  "fusion": "weighted_mean",
  "quality_binned_threshold": true,
  "ctx_id": 0,
  "cache_dir": "cache"
}

## `pakface/` — the pipeline library

### `pakface/__init__.py`
PakFace: face recognition pipeline tuned for Pakistani faces (TechFest Nemo's Deep Dive, Round 2).

In [ ]:
%%writefile pakface/__init__.py
"""PakFace: face recognition pipeline tuned for Pakistani faces (TechFest Nemo's Deep Dive, Round 2)."""
from .config import ModelSpec, PipelineConfig, default_config  # noqa: F401

### `pakface/config.py`
Central configuration for the PakFace pipeline.

In [ ]:
%%writefile pakface/config.py
"""Central configuration for the PakFace pipeline.

Every pipeline stage can be switched on/off from here, which is what makes the
ablation study (scripts/05_ablation.py) a one-liner per variant.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path


@dataclass
class ModelSpec:
    """One recognition backbone in the ensemble."""

    name: str                      # unique key, e.g. "adaface_ir101"
    kind: str                      # "adaface_hf" | "insightface_onnx"
    path: str = ""                 # local dir (adaface_hf) or .onnx file (insightface_onnx)
    repo_id: str = ""              # HF repo for adaface_hf (auto-download)
    channel_order: str = "rgb"     # adaface_hf only: "rgb" or "bgr" (verify with scripts/00_sanity_check.py)
    weight: float = 1.0            # score-fusion weight (tune on val with 05_ablation.py --tune-weights)
    finetuned_ckpt: str = ""       # optional state_dict produced by scripts/03_finetune.py


@dataclass
class PipelineConfig:
    # ---------------- detection ----------------
    detector_pack: str = "buffalo_l"     # insightface model pack providing SCRFD/RetinaFace-style detector
    det_size: int = 640
    det_thresh: float = 0.35             # lower than default 0.5: we prefer a weak face over no face
    pad_retry: bool = True               # retry with border padding (tight crops kill detectors)
    upscale_retry: bool = True           # retry with 2x upscale for tiny images
    enhance_for_detection: bool = True   # retry detection on a brightened image when dark

    # ---------------- alignment ----------------
    use_alignment: bool = True           # 5-point similarity transform to the ArcFace 112x112 template

    # ---------------- quality + enhancement ----------------
    use_enhancement: bool = True         # adaptive low-light / contrast fix on the aligned crop
    enhancement_fusion: bool = True      # embed original AND enhanced crop, average (never loses info)

    # ---------------- embedding ----------------
    use_flip_tta: bool = True
    models: list[ModelSpec] = field(default_factory=list)

    # ---------------- matching ----------------
    fusion: str = "weighted_mean"        # score fusion across models
    quality_binned_threshold: bool = True

    # ---------------- runtime ----------------
    ctx_id: int = 0                      # GPU id, -1 for CPU
    cache_dir: str = "cache"

    # ------------------------------------------------------------------
    def to_json(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(asdict(self), indent=2))

    @classmethod
    def from_json(cls, path: str | Path) -> "PipelineConfig":
        raw = json.loads(Path(path).read_text())
        raw["models"] = [ModelSpec(**m) for m in raw.get("models", [])]
        return cls(**raw)

    def variant_key(self) -> str:
        """Key that changes whenever an option that affects embeddings changes (for caching)."""
        parts = [
            f"al{int(self.use_alignment)}",
            f"en{int(self.use_enhancement)}{int(self.enhancement_fusion)}",
            f"tta{int(self.use_flip_tta)}",
            f"pad{int(self.pad_retry)}up{int(self.upscale_retry)}ed{int(self.enhance_for_detection)}",
            f"dt{self.det_thresh}",
        ]
        return "_".join(parts)


def default_config() -> PipelineConfig:
    """The competition configuration: AdaFace IR-101 (low-quality specialist) + Glint360K R100 ensemble."""
    return PipelineConfig(
        models=[
            ModelSpec(
                name="adaface_ir101",
                kind="adaface_hf",
                repo_id="minchul/cvlface_adaface_ir101_webface12m",
                path="models/adaface_ir101_webface12m",
                channel_order="rgb",
                weight=1.0,
            ),
            ModelSpec(
                name="glint_r100",
                kind="insightface_onnx",
                path="~/.insightface/models/antelopev2/glintr100.onnx",
                weight=1.0,
            ),
        ]
    )

### `pakface/data.py`
Dataset handling: scanning, merging sources, de-duplication, identity-disjoint splits, pair generation.

In [ ]:
%%writefile pakface/data.py
"""Dataset handling: scanning, merging sources, de-duplication, identity-disjoint splits, pair generation.

Expected raw layout (one folder per person, any depth of source folders above it):

    data/raw/<source>/<identity>/*.jpg
    e.g. data/raw/own_collection/P0007/IMG_2231.jpg
         data/raw/halfface/P_012/p012_00m.jpg
         data/raw/imfdb/AamirKhan/....jpg

Optional metadata CSVs (used for the per-group fairness breakdown):
    identities.csv : identity, gender, age_group, skin_tone, region, ...
    images.csv     : path_or_filename, headwear, glasses, mask, lighting, device, ...
"""
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif"}


def normalize_identity(name: str) -> str:
    """'Aamir_Khan', 'aamir khan', 'AamirKhan' -> 'aamirkhan' so the same person never ends up
    under two labels when sources overlap (IMFDB and the Kaggle Bollywood sets share actors)."""
    return re.sub(r"[^a-z0-9]", "", name.lower())


def scan(root: str, merge_identities_across_sources: bool = True) -> pd.DataFrame:
    rows = []
    for p in Path(root).rglob("*"):
        if p.suffix.lower() in IMG_EXT and p.is_file():
            rel = p.relative_to(root).parts
            source = rel[0] if len(rel) >= 3 else "default"
            ident_raw = p.parent.name
            key = normalize_identity(ident_raw)
            # only merge real names across sources; codes like "p007", "s12", "0042" are dataset-local IDs
            is_code = re.fullmatch(r"[a-z]{0,3}_?\d+", key) is not None or len(key) < 4
            identity = key if (merge_identities_across_sources and not is_code) else f"{source}__{key}"
            rows.append({"path": str(p), "source": source, "identity_raw": ident_raw, "identity": identity})
    df = pd.DataFrame(rows)
    if df.empty:
        raise SystemExit(f"no images found under {root}")
    return df


# ---------------------------------------------------------------- de-duplication
def dhash(img_gray_small: np.ndarray) -> int:
    """64-bit difference hash of a 9x8 grayscale thumbnail."""
    diff = img_gray_small[:, 1:] > img_gray_small[:, :-1]
    return int(sum(1 << i for i, v in enumerate(diff.flatten()) if v))


def hamming(a: int, b: int) -> int:
    return bin(a ^ b).count("1")


def find_duplicates(df: pd.DataFrame, hashes: list[int], max_dist: int = 4):
    """Returns (within-identity duplicates to drop, cross-identity near-duplicates = probable label noise)."""
    hs = np.array(hashes, dtype=np.uint64)
    ids = df["identity"].to_numpy()
    drop, conflicts = set(), []
    # bucket by top 16 bits to keep it ~linear; exact near-dups almost always share them
    buckets: dict[int, list[int]] = {}
    for i, h in enumerate(hs):
        buckets.setdefault(int(h) >> 48, []).append(i)
    for idx in buckets.values():
        for a_pos, a in enumerate(idx):
            if a in drop:
                continue
            for b in idx[a_pos + 1 :]:
                if b in drop:
                    continue
                if hamming(int(hs[a]), int(hs[b])) <= max_dist:
                    if ids[a] == ids[b]:
                        drop.add(b)
                    else:
                        conflicts.append((df.iloc[a]["path"], df.iloc[b]["path"]))
    return drop, conflicts


# ---------------------------------------------------------------- splitting
def identity_disjoint_split(df: pd.DataFrame, val_frac=0.15, test_frac=0.2, seed=42,
                            min_images_eval=2, test_sources: list | None = None,
                            eval_source_train_frac: float = 0.0) -> pd.DataFrame:
    """Assign every IDENTITY (not image) to train/val/test so no person appears in two splits.

    test_sources: if given, only identities from these sources go to val/test (e.g. ["own_collection"]
    so the benchmark you report on is Pakistani, consented and never seen by the pretrained model).
    eval_source_train_frac: with test_sources, this fraction of their identities stays in train (the
    problem statement requires fine-tuning on your own collected data); the rest is split val/test.
    """
    rng = np.random.default_rng(seed)
    df = df.copy()
    counts = df.groupby("identity").size()
    eligible = counts[counts >= min_images_eval].index
    if test_sources:
        src_ok = df[df["source"].isin(test_sources)]["identity"].unique()
        eligible = [i for i in eligible if i in set(src_ok)]
    eligible = np.array(sorted(eligible))
    rng.shuffle(eligible)
    if test_sources and eval_source_train_frac > 0:
        eligible = eligible[int(round(len(eligible) * eval_source_train_frac)):]  # the rest stays in train
    n = len(eligible)
    if test_sources:  # every eligible identity comes from the eval sources -> split them val/test only
        n_test = int(round(n * test_frac / (val_frac + test_frac)))
        n_val = n - n_test
    else:
        n_test, n_val = int(round(n * test_frac)), int(round(n * val_frac))
    test_ids, val_ids = set(eligible[:n_test]), set(eligible[n_test : n_test + n_val])
    df["split"] = "train"
    df.loc[df["identity"].isin(val_ids), "split"] = "val"
    df.loc[df["identity"].isin(test_ids), "split"] = "test"
    assert_disjoint(df)
    return df


def assert_disjoint(df: pd.DataFrame) -> None:
    per_id = df.groupby("identity")["split"].nunique()
    bad = per_id[per_id > 1]
    if len(bad):
        raise AssertionError(f"{len(bad)} identities appear in more than one split: {list(bad.index[:5])}")


# ---------------------------------------------------------------- pairs
def make_pairs(df: pd.DataFrame, n_genuine_per_id: int = 20, impostor_ratio: float = 1.0, seed: int = 0,
               hard_negative_cols: tuple = ("gender",)) -> pd.DataFrame:
    """Balanced verification pairs. Half of the impostors are 'hard': same gender (and any other
    hard_negative_cols present), because the hidden benchmark will certainly contain look-alike pairs."""
    rng = np.random.default_rng(seed)
    groups = {k: v["path"].tolist() for k, v in df.groupby("identity")}
    ids = [k for k, v in groups.items() if len(v) >= 2]
    gen = []
    for i in ids:
        imgs = groups[i]
        all_pairs = [(a, b) for k, a in enumerate(imgs) for b in imgs[k + 1 :]]
        take = min(n_genuine_per_id, len(all_pairs))
        for j in rng.choice(len(all_pairs), take, replace=False):
            gen.append((*all_pairs[j], 1, i, i))
    n_imp = int(len(gen) * impostor_ratio)
    all_ids = list(groups)
    attrs = df.drop_duplicates("identity").set_index("identity")
    hard_cols = [c for c in hard_negative_cols if c in attrs.columns]
    imp, seen = [], set()
    tries = 0
    while len(imp) < n_imp and tries < n_imp * 20:
        tries += 1
        a_id = all_ids[int(rng.integers(len(all_ids)))]
        pool = []
        if hard_cols and rng.random() < 0.5:
            same = attrs.index[(attrs[hard_cols] == attrs.loc[a_id, hard_cols]).all(axis=1)]
            pool = [x for x in same if x != a_id]
        if pool:
            b_id = pool[int(rng.integers(len(pool)))]
        else:
            b_id = all_ids[int(rng.integers(len(all_ids)))]
            if b_id == a_id:
                continue
        a = groups[a_id][int(rng.integers(len(groups[a_id])))]
        b = groups[b_id][int(rng.integers(len(groups[b_id])))]
        key = (min(a, b), max(a, b))
        if key in seen:
            continue
        seen.add(key)
        imp.append((a, b, 0, a_id, b_id))
    pairs = pd.DataFrame(gen + imp, columns=["path_a", "path_b", "label", "id_a", "id_b"])
    return pairs.sample(frac=1, random_state=seed).reset_index(drop=True)


# ---------------------------------------------------------------- metadata for fairness breakdown
def attach_metadata(df: pd.DataFrame, identities_csv: str | None, images_csv: str | None) -> pd.DataFrame:
    if identities_csv and Path(identities_csv).exists():
        # matched on the folder name as written (normalised); add a `source` column to the CSV
        # if two sources reuse the same folder names (e.g. both have P001)
        meta = pd.read_csv(identities_csv)
        meta["_key"] = meta["identity"].astype(str).map(normalize_identity)
        meta = meta.drop(columns=["identity"])
        df["_key"] = df["identity_raw"].astype(str).map(normalize_identity)
        on = ["_key", "source"] if "source" in meta.columns else ["_key"]
        meta = meta.drop_duplicates(subset=on, keep="last")
        df = df.merge(meta, on=on, how="left").drop(columns=["_key"])
    if images_csv and Path(images_csv).exists():
        im = pd.read_csv(images_csv)
        key = im.columns[0]
        im["_fname"] = im[key].astype(str).map(lambda s: Path(s).name)
        df["_fname"] = df["path"].map(lambda s: Path(s).name)
        df = df.merge(im.drop(columns=[key]), on="_fname", how="left").drop(columns=["_fname"])
    return df

### `pakface/detect.py`
Robust face detection with retry cascade.

In [ ]:
%%writefile pakface/detect.py
"""Robust face detection with retry cascade.

Uses the SCRFD detector shipped in InsightFace model packs (a RetinaFace-family
detector with 5-point landmarks). Real benchmark images break detectors in
predictable ways, so we retry instead of giving up:

  1. plain detection
  2. pad the image (tightly-cropped faces have no context -> detector misses them)
  3. upscale tiny images (CCTV / thumbnails)
  4. brighten dark images
  5. last resort: treat the whole image as a face crop (flagged "no_detection")
"""
from __future__ import annotations

from dataclasses import dataclass

import cv2
import numpy as np

from .enhance import brighten_for_detection


@dataclass
class Detection:
    bbox: np.ndarray          # x1, y1, x2, y2 in original image coordinates
    kps: np.ndarray | None    # (5, 2) landmarks, None for the whole-image fallback
    score: float
    stage: str                # which retry stage found it

    def to_json(self) -> dict:
        return {
            "bbox": [float(v) for v in self.bbox[:4]],
            "kps": None if self.kps is None else np.asarray(self.kps).tolist(),
            "score": float(self.score),
            "stage": self.stage,
        }

    @classmethod
    def from_json(cls, d: dict) -> "Detection":
        return cls(
            np.array(d["bbox"], dtype=np.float32),
            None if d["kps"] is None else np.array(d["kps"], dtype=np.float32),
            d["score"],
            d["stage"],
        )


class FaceDetector:
    def __init__(self, pack: str = "buffalo_l", det_size: int = 640, det_thresh: float = 0.35,
                 ctx_id: int = 0, pad_retry: bool = True, upscale_retry: bool = True,
                 enhance_retry: bool = True):
        from insightface.app import FaceAnalysis

        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if ctx_id >= 0 else ["CPUExecutionProvider"]
        self.app = FaceAnalysis(name=pack, allowed_modules=["detection"], providers=providers)
        self.app.prepare(ctx_id=ctx_id, det_thresh=det_thresh, det_size=(det_size, det_size))
        self.det = self.app.det_model
        self.det_size = det_size
        self.pad_retry, self.upscale_retry, self.enhance_retry = pad_retry, upscale_retry, enhance_retry

    # ------------------------------------------------------------------
    def _raw(self, img: np.ndarray):
        bboxes, kpss = self.det.detect(img, max_num=0, metric="default")
        if bboxes is None or len(bboxes) == 0:
            return None, None
        return bboxes, kpss

    @staticmethod
    def _pick(bboxes: np.ndarray, kpss: np.ndarray, shape) -> int:
        """Primary face = large, confident and central."""
        h, w = shape[:2]
        areas = (bboxes[:, 2] - bboxes[:, 0]) * (bboxes[:, 3] - bboxes[:, 1])
        cx = (bboxes[:, 0] + bboxes[:, 2]) / 2 - w / 2
        cy = (bboxes[:, 1] + bboxes[:, 3]) / 2 - h / 2
        center_pen = 1.0 - 0.5 * np.sqrt((cx / w) ** 2 + (cy / h) ** 2)
        return int(np.argmax(areas / areas.max() * bboxes[:, 4] * center_pen))

    def detect(self, img: np.ndarray) -> Detection:
        h, w = img.shape[:2]

        attempts = [("plain", img, 0, 1.0)]
        if self.pad_retry:
            p = int(0.4 * max(h, w))
            padded = cv2.copyMakeBorder(img, p, p, p, p, cv2.BORDER_CONSTANT, value=(114, 114, 114))
            attempts.append(("padded", padded, p, 1.0))
        if self.upscale_retry and max(h, w) < 400:
            s = 400.0 / max(h, w)
            up = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
            p = int(0.4 * max(up.shape[:2]))
            up = cv2.copyMakeBorder(up, p, p, p, p, cv2.BORDER_CONSTANT, value=(114, 114, 114))
            attempts.append(("upscaled", up, p, s))
        if self.enhance_retry:
            gray_mean = float(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).mean())
            if gray_mean < 80:
                br = brighten_for_detection(img)
                p = int(0.4 * max(h, w))
                br = cv2.copyMakeBorder(br, p, p, p, p, cv2.BORDER_CONSTANT, value=(114, 114, 114))
                attempts.append(("brightened", br, p, 1.0))

        for stage, im, pad, scale in attempts:
            bboxes, kpss = self._raw(im)
            if bboxes is None:
                continue
            i = self._pick(bboxes, kpss, im.shape)
            bbox = bboxes[i, :4].astype(np.float32)
            kps = kpss[i].astype(np.float32) if kpss is not None else None
            # map back to original coordinates
            bbox = (bbox - pad) / scale
            if kps is not None:
                kps = (kps - pad) / scale
            return Detection(bbox, kps, float(bboxes[i, 4]), stage)

        return Detection(np.array([0, 0, w, h], dtype=np.float32), None, 0.0, "no_detection")

    def detect_all(self, img: np.ndarray, min_rel_size: float = 0.25, min_px: int = 12) -> list[Detection]:
        """Every face in a group / classroom photo, ordered left to right.

        Large photos are also scanned at a higher input resolution: at 640 px, a face 100 px wide in a
        4000 px photo shrinks to 16 px and is missed. Faces smaller than min_rel_size x the biggest face
        are dropped, because in a single-person photo they are bystanders or posters, not the subject.
        """
        h, w = img.shape[:2]
        sides = {self.det_size, min(1920, 32 * int(np.ceil(max(h, w) / 32)))}
        boxes, kps = [], []
        for side in sorted(s for s in sides if s >= self.det_size):
            b, k = self.det.detect(img, input_size=(side, side), max_num=0, metric="default")
            if b is not None and len(b):
                boxes.append(b)
                kps.append(k if k is not None else np.full((len(b), 5, 2), np.nan, dtype=np.float32))
        if not boxes:
            return []
        b, k = np.concatenate(boxes), np.concatenate(kps)
        keep = _nms(b[:, :4], b[:, 4], iou=0.4)
        b, k = b[keep], k[keep]
        side = np.minimum(b[:, 2] - b[:, 0], b[:, 3] - b[:, 1])
        ok = side >= max(min_rel_size * side.max(), min_px)
        b, k = b[ok], k[ok]
        return [Detection(b[i, :4].astype(np.float32), None if np.isnan(k[i]).any() else k[i].astype(np.float32),
                          float(b[i, 4]), "multi") for i in np.argsort(b[:, 0])]


def _nms(boxes: np.ndarray, scores: np.ndarray, iou: float) -> np.ndarray:
    """Greedy non-maximum suppression (merges detections of the same face from different scan sizes)."""
    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    order, keep = np.argsort(-scores), []
    while len(order):
        i, rest = order[0], order[1:]
        keep.append(i)
        iw = np.clip(np.minimum(boxes[i, 2], boxes[rest, 2]) - np.maximum(boxes[i, 0], boxes[rest, 0]), 0, None)
        ih = np.clip(np.minimum(boxes[i, 3], boxes[rest, 3]) - np.maximum(boxes[i, 1], boxes[rest, 1]), 0, None)
        inter = iw * ih
        order = rest[inter / (area[i] + area[rest] - inter + 1e-9) < iou]
    return np.array(keep, dtype=int)

### `pakface/align.py`
Face alignment: 5-point similarity transform onto the standard ArcFace 112x112 template.

In [ ]:
%%writefile pakface/align.py
"""Face alignment: 5-point similarity transform onto the standard ArcFace 112x112 template.

All modern recognisers (ArcFace, AdaFace, Glint360K models) were trained on crops
produced with exactly this template, so matching it is worth several points of
accuracy on its own (see the "no alignment" row in the ablation).
"""
from __future__ import annotations

import cv2
import numpy as np

# Canonical landmark positions in a 112x112 crop:
# left eye, right eye, nose tip, left mouth corner, right mouth corner
ARCFACE_TEMPLATE = np.array(
    [
        [38.2946, 51.6963],
        [73.5318, 51.5014],
        [56.0252, 71.7366],
        [41.5493, 92.3655],
        [70.7299, 92.2041],
    ],
    dtype=np.float32,
)


def estimate_transform(kps: np.ndarray, size: int = 112) -> np.ndarray:
    """Similarity transform (rotation + uniform scale + translation) from landmarks to template."""
    dst = ARCFACE_TEMPLATE * (size / 112.0)
    M, _ = cv2.estimateAffinePartial2D(np.asarray(kps, dtype=np.float32), dst, method=cv2.LMEDS)
    if M is None:  # degenerate landmarks: fall back to least squares
        M, _ = cv2.estimateAffinePartial2D(np.asarray(kps, dtype=np.float32), dst)
    return M


def align_face(img: np.ndarray, kps: np.ndarray, size: int = 112) -> np.ndarray:
    M = estimate_transform(kps, size)
    return cv2.warpAffine(img, M, (size, size), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def crop_no_align(img: np.ndarray, bbox: np.ndarray, size: int = 112, expand: float = 0.15) -> np.ndarray:
    """Plain square bbox crop without landmark alignment (used for the ablation and as a fallback)."""
    h, w = img.shape[:2]
    x1, y1, x2, y2 = [float(v) for v in bbox[:4]]
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * (1 + expand)
    x1, x2 = int(round(cx - side / 2)), int(round(cx + side / 2))
    y1, y2 = int(round(cy - side / 2)), int(round(cy + side / 2))
    pad = max(0, -x1, -y1, x2 - w, y2 - h)
    if pad:
        img = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_REPLICATE)
        x1, x2, y1, y2 = x1 + pad, x2 + pad, y1 + pad, y2 + pad
    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        crop = img
    return cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA if crop.shape[0] > size else cv2.INTER_CUBIC)


def whole_image_fallback(img: np.ndarray, size: int = 112) -> np.ndarray:
    """Used when no detector finds a face: assume the image is already a (tight) face crop."""
    h, w = img.shape[:2]
    side = min(h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    crop = img[y0 : y0 + side, x0 : x0 + side]
    return cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA if side > size else cv2.INTER_CUBIC)

### `pakface/quality.py`
Image-quality assessment for face crops.

In [ ]:
%%writefile pakface/quality.py
"""Image-quality assessment for face crops.

Quality is used in four places (never to *reject* a benchmark pair, because a
verification benchmark expects a decision for every pair):
  1. Cleaning training data (drop unusable crops before fine-tuning).
  2. Routing: deciding whether to run low-light / contrast enhancement.
  3. Template building: quality-weighted averaging of gallery images.
  4. Quality-binned thresholds: low-quality pairs get their own calibrated threshold.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass

import cv2
import numpy as np


@dataclass
class QualityReport:
    sharpness: float      # variance of Laplacian on the 112x112 crop (higher = sharper)
    brightness: float     # mean luminance 0..255
    contrast: float       # luminance std
    face_px: float        # min side of the detected face box in the ORIGINAL image (pixels)
    det_score: float      # detector confidence
    yaw_proxy: float      # 0 = frontal, ~1 = profile (from landmark geometry)
    score: float          # combined quality in [0, 1]
    flags: list

    def as_dict(self) -> dict:
        return asdict(self)


def _sigmoid01(x: float, mid: float, scale: float) -> float:
    return float(1.0 / (1.0 + np.exp(-(x - mid) / scale)))


def yaw_from_kps(kps: np.ndarray | None) -> float:
    if kps is None:
        return 0.5
    kps = np.asarray(kps, dtype=np.float32)
    le, re, nose = kps[0], kps[1], kps[2]
    eye_mid = (le + re) / 2
    inter_eye = np.linalg.norm(re - le) + 1e-6
    return float(min(1.0, abs(nose[0] - eye_mid[0]) / (0.5 * inter_eye)))


def assess(crop_bgr: np.ndarray, face_px: float, det_score: float, kps: np.ndarray | None) -> QualityReport:
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    sharp = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    bright = float(gray.mean())
    contrast = float(gray.std())
    yaw = yaw_from_kps(kps)

    flags = []
    if sharp < 60:
        flags.append("blurry")
    if bright < 60:
        flags.append("dark")
    if bright > 200:
        flags.append("overexposed")
    if contrast < 25:
        flags.append("low_contrast")
    if face_px < 40:
        flags.append("low_res")
    if yaw > 0.6:
        flags.append("profile")
    if det_score < 0.5:
        flags.append("weak_detection")

    # Each factor mapped to [0,1]; geometric mean so one terrible factor drags the score down.
    factors = [
        _sigmoid01(np.log1p(sharp), np.log1p(80), 0.5),
        1.0 - abs(bright - 127) / 127,
        _sigmoid01(contrast, 30, 8),
        _sigmoid01(face_px, 48, 12),
        float(np.clip(det_score, 0, 1)),
        1.0 - 0.6 * yaw,
    ]
    factors = np.clip(np.array(factors), 0.02, 1.0)
    score = float(np.exp(np.log(factors).mean()))
    return QualityReport(sharp, bright, contrast, float(face_px), float(det_score), yaw, score, flags)

### `pakface/enhance.py`
Adaptive, identity-safe enhancement.

In [ ]:
%%writefile pakface/enhance.py
"""Adaptive, identity-safe enhancement.

Deliberately conservative: only photometric fixes (gamma, CLAHE, mild denoise).
Generative face restoration (GFPGAN / CodeFormer) is NOT used by default because it
hallucinates facial detail and can make two different people look alike, which
raises the false-accept rate. It can be tested as an extra ablation row if time allows.
"""
from __future__ import annotations

import cv2
import numpy as np


def adaptive_gamma(img: np.ndarray, target: float = 0.5) -> np.ndarray:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean = float(np.clip(gray.mean() / 255.0, 0.02, 0.98))
    gamma = float(np.clip(np.log(target) / np.log(mean), 0.4, 2.5))
    lut = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
    return cv2.LUT(img, lut)


def clahe_lab(img: np.ndarray, clip: float = 2.0, grid: int = 4) -> np.ndarray:
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)


def needs_enhancement(flags: list) -> bool:
    return any(f in flags for f in ("dark", "overexposed", "low_contrast"))


def enhance_crop(crop: np.ndarray, flags: list) -> np.ndarray:
    out = crop
    if "dark" in flags or "overexposed" in flags:
        out = adaptive_gamma(out)
    if "dark" in flags or "low_contrast" in flags or "overexposed" in flags:
        out = clahe_lab(out)
    if "dark" in flags and "low_res" not in flags:
        # brightening amplifies sensor noise; mild NL-means keeps edges
        out = cv2.fastNlMeansDenoisingColored(out, None, 3, 3, 5, 15)
    return out


def brighten_for_detection(img: np.ndarray) -> np.ndarray:
    return clahe_lab(adaptive_gamma(img), clip=2.5, grid=8)

### `pakface/embedders.py`
Recognition backbones.

In [ ]:
%%writefile pakface/embedders.py
"""Recognition backbones.

* AdaFaceHF        - AdaFace IR-101 trained on WebFace12M (CVLFace release on Hugging Face).
                     AdaFace's quality-adaptive margin is designed for low-quality,
                     blurry, low-resolution faces: exactly the hidden-benchmark risk.
* InsightFaceONNX  - any InsightFace ArcFace ONNX model, e.g. glintr100.onnx (antelopev2 pack,
                     R100 on Glint360K) or w600k_r50.onnx (buffalo_l pack).

Both return L2-normalised 512-d embeddings plus the raw feature norm
(for AdaFace the norm is a learned image-quality signal).
"""
from __future__ import annotations

import glob
import os
import sys
from pathlib import Path

import numpy as np


def _l2n(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / n, n[:, 0]


class BaseEmbedder:
    name: str = "base"

    def embed_batch(self, crops: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
        raise NotImplementedError

    def embed(self, crops: list[np.ndarray], flip_tta: bool = True, batch_size: int = 64):
        """crops: list of aligned 112x112 BGR uint8 images."""
        feats, norms = [], []
        for i in range(0, len(crops), batch_size):
            chunk = crops[i : i + batch_size]
            f, n = self.embed_batch(chunk)
            if flip_tta:
                ff, nf = self.embed_batch([np.ascontiguousarray(c[:, ::-1]) for c in chunk])
                f, n = f * n[:, None] + ff * nf[:, None], (n + nf) / 2  # norm-weighted fusion
            feats.append(f)
            norms.append(n)
        feats = np.concatenate(feats).astype(np.float32)
        feats, _ = _l2n(feats)
        return feats, np.concatenate(norms).astype(np.float32)


# ---------------------------------------------------------------------------
class InsightFaceONNX(BaseEmbedder):
    def __init__(self, name: str, path: str, ctx_id: int = 0):
        import insightface

        path = os.path.expanduser(path)
        if not os.path.exists(path):
            # antelopev2 sometimes unzips into a nested folder
            hits = glob.glob(os.path.expanduser(f"~/.insightface/models/**/{Path(path).name}"), recursive=True)
            if not hits:
                raise FileNotFoundError(f"{path} not found - run scripts/00_download_models.py first")
            path = hits[0]
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if ctx_id >= 0 else ["CPUExecutionProvider"]
        self.model = insightface.model_zoo.get_model(path, providers=providers)
        self.model.prepare(ctx_id=ctx_id)
        self.name = name

    def embed_batch(self, crops):
        raw = np.asarray(self.model.get_feat(crops), dtype=np.float32)  # handles BGR->RGB + normalisation
        return _l2n(raw)


# ---------------------------------------------------------------------------
def download_hf_repo(repo_id: str, path: str, token: str | None = None) -> None:
    """Mirror of the official CVLFace download helper."""
    from huggingface_hub import hf_hub_download

    os.makedirs(path, exist_ok=True)
    files_txt = os.path.join(path, "files.txt")
    if not os.path.exists(files_txt):
        hf_hub_download(repo_id, "files.txt", token=token, local_dir=path)
    files = [f for f in Path(files_txt).read_text().split("\n") if f]
    for f in files + ["config.json", "wrapper.py", "model.safetensors"]:
        if not os.path.exists(os.path.join(path, f)):
            hf_hub_download(repo_id, f, token=token, local_dir=path)


def load_adaface_hf(path: str, repo_id: str = "", token: str | None = None):
    """Build the CVLFace model the way its wrapper.py does, without AutoModel.

    wrapper.py targets transformers 4.x (it never calls post_init(), which 5.x requires), so we replicate
    it: model.yaml -> get_model -> pretrained_model/model.pt, held under `.model` so state-dict keys (and
    fine-tuned checkpoints) match the original wrapper.
    """
    import torch
    import yaml
    from omegaconf import OmegaConf

    path = os.path.abspath(os.path.expanduser(path))
    if repo_id and not os.path.exists(os.path.join(path, "model.safetensors")):
        download_hf_repo(repo_id, path, token)
    cwd = os.getcwd()
    os.chdir(path)  # CVLFace code resolves yaml_path / pretrained_model/* relative to cwd
    # `from models import get_model` must hit the CVLFace package, not the project root's models/ dir
    # (weights), which Python would otherwise pick up as an empty namespace package.
    sys.path.insert(0, path)
    for name in [m for m in sys.modules if m == "models" or m.startswith("models.")]:
        del sys.modules[name]
    try:
        from models import get_model

        with open("pretrained_model/model.yaml") as f:
            conf = OmegaConf.create(yaml.safe_load(f))
        inner = get_model(conf)
        inner.load_state_dict_from_path("pretrained_model/model.pt")
    finally:
        os.chdir(cwd)
        sys.path.remove(path)

    class CVLFaceRecognitionModel(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model

        def forward(self, *args, **kwargs):
            return self.model(*args, **kwargs)

    return CVLFaceRecognitionModel(inner)


def adaface_forward(model, x):
    """CVLFace wrappers may return a tensor, a tuple (feat, norm) or a dict; normalise that."""
    out = model(x)
    if isinstance(out, dict):
        out = out.get("embedding", next(iter(out.values())))
    if isinstance(out, (tuple, list)):
        out = out[0]
    return out


class AdaFaceHF(BaseEmbedder):
    def __init__(self, name: str, path: str, repo_id: str = "", channel_order: str = "rgb",
                 ctx_id: int = 0, finetuned_ckpt: str = ""):
        import torch

        self.torch = torch
        self.device = torch.device(f"cuda:{ctx_id}" if ctx_id >= 0 and torch.cuda.is_available() else "cpu")
        self.model = load_adaface_hf(path, repo_id, os.environ.get("HF_TOKEN"))
        if finetuned_ckpt:
            state = torch.load(finetuned_ckpt, map_location="cpu")
            missing, unexpected = self.model.load_state_dict(state, strict=False)
            if missing or unexpected:
                print(f"[{name}] fine-tuned ckpt: {len(missing)} missing / {len(unexpected)} unexpected keys")
        self.model.eval().to(self.device)
        self.channel_order = channel_order
        self.name = name

    def preprocess(self, crops):
        arr = np.stack(crops).astype(np.float32)  # N,H,W,3 BGR
        if self.channel_order == "rgb":
            arr = arr[..., ::-1]
        arr = (arr / 255.0 - 0.5) / 0.5
        return self.torch.from_numpy(np.ascontiguousarray(arr.transpose(0, 3, 1, 2)))

    def embed_batch(self, crops):
        with self.torch.no_grad():
            x = self.preprocess(crops).to(self.device)
            f = adaface_forward(self.model, x).float().cpu().numpy()
        return _l2n(f)


# ---------------------------------------------------------------------------
def build_embedders(cfg) -> list[BaseEmbedder]:
    out = []
    for spec in cfg.models:
        if spec.kind == "adaface_hf":
            out.append(AdaFaceHF(spec.name, spec.path, spec.repo_id, spec.channel_order, cfg.ctx_id, spec.finetuned_ckpt))
        elif spec.kind == "insightface_onnx":
            out.append(InsightFaceONNX(spec.name, spec.path, cfg.ctx_id))
        else:
            raise ValueError(f"unknown model kind {spec.kind}")
    return out

### `pakface/pipeline.py`
End-to-end pipeline: image -> detection -> alignment -> quality -> enhancement -> embeddings.

In [ ]:
%%writefile pakface/pipeline.py
"""End-to-end pipeline: image -> detection -> alignment -> quality -> enhancement -> embeddings.

Embeddings are cached per pipeline variant so the ablation study and threshold
tuning never recompute the expensive parts.
"""
from __future__ import annotations

import hashlib
import json
import os
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import numpy as np
from tqdm import tqdm

from .align import align_face, crop_no_align, whole_image_fallback
from .config import PipelineConfig
from .detect import Detection
from .enhance import enhance_crop, needs_enhancement
from .quality import assess


FACE_SEP = "#face"  # "photo.jpg#face2" = the 3rd face (left to right) of a multi-person photo


def split_face_key(key: str) -> tuple[str, int | None]:
    path, sep, k = key.rpartition(FACE_SEP)
    return (path, int(k)) if sep and k.isdigit() else (key, None)


def load_image(path: str) -> np.ndarray:
    """EXIF-aware loader (phone photos are often stored rotated) that always returns BGR uint8."""
    from PIL import Image, ImageOps

    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        return cv2.cvtColor(np.asarray(im), cv2.COLOR_RGB2BGR)


@dataclass
class FeatureStore:
    paths: list
    emb: dict                      # model name -> (N, D) float32 normalised
    norm: dict                     # model name -> (N,) raw feature norm
    quality: np.ndarray            # (N,) combined quality score
    flags: list                    # per-image list of quality flags
    det_stage: list                # which detection stage succeeded
    index: dict = field(default_factory=dict)

    def __post_init__(self):
        self.index = {p: i for i, p in enumerate(self.paths)}

    def save(self, path: str | Path) -> None:
        arrays = {f"emb__{k}": v for k, v in self.emb.items()}
        arrays.update({f"norm__{k}": v for k, v in self.norm.items()})
        np.savez_compressed(
            path, quality=self.quality, paths=np.array(self.paths),
            flags=np.array([json.dumps(f) for f in self.flags]), det_stage=np.array(self.det_stage), **arrays,
        )

    @classmethod
    def load(cls, path: str | Path) -> "FeatureStore":
        z = np.load(path, allow_pickle=False)
        emb = {k[5:]: z[k] for k in z.files if k.startswith("emb__")}
        norm = {k[6:]: z[k] for k in z.files if k.startswith("norm__")}
        return cls(list(z["paths"]), emb, norm, z["quality"], [json.loads(f) for f in z["flags"]], list(z["det_stage"]))

    def subset_scores(self, a: list[str], b: list[str], weights: dict | None = None):
        """Cosine score per model + weighted fusion for pairs (a[i], b[i])."""
        ia = np.array([self.index[p] for p in a])
        ib = np.array([self.index[p] for p in b])
        per_model = {m: np.sum(e[ia] * e[ib], axis=1) for m, e in self.emb.items()}
        weights = weights or {m: 1.0 for m in per_model}
        wsum = sum(weights.get(m, 0.0) for m in per_model) or 1.0
        fused = sum(weights.get(m, 0.0) * s for m, s in per_model.items()) / wsum
        pair_q = np.minimum(self.quality[ia], self.quality[ib])
        return fused, per_model, pair_q


class FacePipeline:
    def __init__(self, cfg: PipelineConfig, load_models: bool = True, load_embedders: bool = True,
                 detector=None, embedders=None):
        """detector / embedders can be passed in to share loaded models between ablation variants."""
        self.cfg = cfg
        os.makedirs(cfg.cache_dir, exist_ok=True)
        self.det_cache_path = Path(cfg.cache_dir) / f"detections_{self._det_key()}.json"
        self.det_cache = json.loads(self.det_cache_path.read_text()) if self.det_cache_path.exists() else {}
        self.detector = detector
        self.embedders = embedders or []
        self.min_face_rel = 0.25  # set by expand_faces()
        if load_models:
            if self.detector is None:
                from .detect import FaceDetector

                self.detector = FaceDetector(cfg.detector_pack, cfg.det_size, cfg.det_thresh, cfg.ctx_id,
                                             cfg.pad_retry, cfg.upscale_retry, cfg.enhance_for_detection)
            if load_embedders and not self.embedders:
                from .embedders import build_embedders

                self.embedders = build_embedders(cfg)

    def save_det_cache(self) -> None:
        self.det_cache_path.write_text(json.dumps(self.det_cache))

    def _det_key(self) -> str:
        c = self.cfg
        return f"{c.detector_pack}_{c.det_size}_{c.det_thresh}_{int(c.pad_retry)}{int(c.upscale_retry)}{int(c.enhance_for_detection)}"

    # ------------------------------------------------------------------
    def detect(self, path: str, img: np.ndarray) -> Detection:
        key = os.path.abspath(path)
        if key in self.det_cache:
            return Detection.from_json(self.det_cache[key])
        det = self.detector.detect(img)
        self.det_cache[key] = det.to_json()
        return det

    def detect_faces(self, path: str, img: np.ndarray | None = None, min_rel_size: float = 0.25) -> list[Detection]:
        """All faces in the image (cached), for group photos."""
        key = f"multi{min_rel_size}::{os.path.abspath(path)}"
        if key not in self.det_cache:
            img = load_image(path) if img is None else img
            self.det_cache[key] = [d.to_json() for d in self.detector.detect_all(img, min_rel_size)]
        return [Detection.from_json(d) for d in self.det_cache[key]]

    def expand_faces(self, paths: list[str], min_rel_size: float = 0.25) -> tuple[dict, dict]:
        """Single-person images map to [path] and keep the robust single-face pipeline (retry cascade).
        Images with 2+ faces map to one key per face. Returns ({path: [keys]}, {key: bbox})."""
        self.min_face_rel = min_rel_size
        faces, boxes = {}, {}
        for p in tqdm(list(dict.fromkeys(paths)), desc="counting faces"):
            dets = self.detect_faces(p, min_rel_size=min_rel_size)
            if len(dets) >= 2:
                faces[p] = [f"{p}{FACE_SEP}{k}" for k in range(len(dets))]
                boxes.update({k: d.bbox for k, d in zip(faces[p], dets)})
            else:
                faces[p] = [p]
        self.save_det_cache()
        return faces, boxes

    def crop(self, img: np.ndarray, det: Detection) -> np.ndarray:
        if det.kps is not None and self.cfg.use_alignment:
            return align_face(img, det.kps)
        if det.stage != "no_detection":
            return crop_no_align(img, det.bbox)
        return whole_image_fallback(img)

    def crop_and_assess(self, img: np.ndarray, det: Detection):
        """Aligned crop of one detected face + its quality report and flags."""
        crop = self.crop(img, det)
        face_px = float(min(det.bbox[2] - det.bbox[0], det.bbox[3] - det.bbox[1]))
        q = assess(crop, face_px, det.score if det.stage != "no_detection" else 0.3, det.kps)
        flags = list(q.flags) + (["no_detection"] if det.stage == "no_detection" else [])
        return crop, q, flags

    def preprocess(self, path: str, img: np.ndarray | None = None):
        real, k = split_face_key(path)
        img = load_image(real) if img is None else img
        det = self.detect(path, img) if k is None else self.detect_faces(real, img, self.min_face_rel)[k]
        crop, q, flags = self.crop_and_assess(img, det)
        return crop, q.score, flags, det.stage

    def embed_crops(self, crops: list[np.ndarray], flags: list[list]) -> tuple[dict, dict]:
        """Per-model unit embeddings + raw norms for aligned crops. Poor-quality crops are also embedded
        after enhancement and the two embeddings are fused (or replaced, if enhancement_fusion is off)."""
        enh = self.cfg.use_enhancement
        enh_idx = [j for j, f in enumerate(flags) if enh and needs_enhancement(f)]
        enh_crops = [enhance_crop(crops[j], flags[j]) for j in enh_idx]
        if enh and not self.cfg.enhancement_fusion:
            crops = list(crops)
            for j, ec in zip(enh_idx, enh_crops):
                crops[j] = ec
        embs, norms = {}, {}
        for e in self.embedders:
            f, n = e.embed(crops, flip_tta=self.cfg.use_flip_tta)
            if enh and self.cfg.enhancement_fusion and enh_crops:
                fe, _ = e.embed(enh_crops, flip_tta=self.cfg.use_flip_tta)
                f[enh_idx] = f[enh_idx] + fe
                f[enh_idx] /= np.linalg.norm(f[enh_idx], axis=1, keepdims=True)
            embs[e.name], norms[e.name] = f, n
        return embs, norms

    # ------------------------------------------------------------------
    def store_path(self) -> Path:
        names = "+".join(f"{m.name}{'-ft' + hashlib.md5(m.finetuned_ckpt.encode()).hexdigest()[:6] if m.finetuned_ckpt else ''}"
                         for m in self.cfg.models)
        return Path(self.cfg.cache_dir) / f"features_{self.cfg.variant_key()}_{names}.npz"

    def build_store(self, paths: list[str], batch: int = 128, degrade_fn=None, tag: str = "") -> FeatureStore:
        """Embed every path once (cached). degrade_fn(img)->img is used for robustness stress tests."""
        sp = self.store_path()
        if tag:
            sp = sp.with_name(sp.stem + f"_{tag}.npz")
        paths = list(dict.fromkeys(paths))
        cached = FeatureStore.load(sp) if sp.exists() else None
        todo = [p for p in paths if cached is None or p not in cached.index]

        new = None
        if todo:
            if not self.embedders:
                raise RuntimeError("pipeline created with load_models=False but uncached images were requested")
            embs = {e.name: [] for e in self.embedders}
            norms = {e.name: [] for e in self.embedders}
            quals, flags_all, stages = [], [], []
            for i in tqdm(range(0, len(todo), batch), desc=f"embedding [{self.cfg.variant_key()}]"):
                chunk = todo[i : i + batch]
                crops, chunk_flags = [], []
                for p in chunk:
                    img = load_image(split_face_key(p)[0])
                    if degrade_fn is not None:
                        img = degrade_fn(img)
                    crop, q, flags, stage = self.preprocess(p, img) if degrade_fn is None else self._preprocess_nocache(img)
                    crops.append(crop)
                    chunk_flags.append(flags)
                    quals.append(q)
                    flags_all.append(flags)
                    stages.append(stage)
                e_chunk, n_chunk = self.embed_crops(crops, chunk_flags)
                for name in embs:
                    embs[name].append(e_chunk[name])
                    norms[name].append(n_chunk[name])
            new = FeatureStore(
                todo, {k: np.concatenate(v) for k, v in embs.items()}, {k: np.concatenate(v) for k, v in norms.items()},
                np.array(quals, dtype=np.float32), flags_all, stages,
            )
            if degrade_fn is None:
                self.det_cache_path.write_text(json.dumps(self.det_cache))

        store = _merge(cached, new)
        store.save(sp)
        return store

    def _preprocess_nocache(self, img):
        det = self.detector.detect(img)
        crop, q, flags = self.crop_and_assess(img, det)
        return crop, q.score, flags, det.stage


def _merge(a: FeatureStore | None, b: FeatureStore | None) -> FeatureStore:
    if a is None:
        return b
    if b is None:
        return a
    return FeatureStore(
        a.paths + b.paths,
        {k: np.concatenate([a.emb[k], b.emb[k]]) for k in a.emb},
        {k: np.concatenate([a.norm[k], b.norm[k]]) for k in a.norm},
        np.concatenate([a.quality, b.quality]),
        a.flags + b.flags,
        a.det_stage + b.det_stage,
    )

### `pakface/matching.py`
Matching: similarity metric, threshold calibration, quality-binned thresholds, identification.

In [ ]:
%%writefile pakface/matching.py
"""Matching: similarity metric, threshold calibration, quality-binned thresholds, identification.

Why cosine similarity: every backbone here is trained with an angular-margin loss
(ArcFace / AdaFace) on L2-normalised features, so identity is encoded in the angle
between embeddings. Cosine is the metric the training objective optimises;
Euclidean distance on normalised vectors is a monotone function of it
(||a-b||^2 = 2 - 2cos), so the two give identical rankings and ROC curves.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np

from .metrics import eer_threshold


def best_accuracy_threshold(scores: np.ndarray, labels: np.ndarray) -> tuple[float, float]:
    """Threshold maximising accuracy (exhaustive over sorted midpoints, O(n log n))."""
    order = np.argsort(scores)
    s, y = scores[order], labels[order].astype(int)
    n_pos = y.sum()
    # predict positive if score > t; sweeping t past s[i] moves sample i to the negative side
    fn = np.cumsum(y)                 # positives at or below index i
    tn = np.cumsum(1 - y)             # negatives at or below index i
    correct = tn + (n_pos - fn)
    i = int(np.argmax(correct))
    t = (s[i] + s[i + 1]) / 2 if i + 1 < len(s) else s[i] + 1e-6
    best = correct[i] / len(s)
    # also consider "everything positive"
    if n_pos / len(s) > best:
        return float(s[0] - 1e-6), float(n_pos / len(s))
    return float(t), float(best)


def best_f1_threshold(scores: np.ndarray, labels: np.ndarray) -> tuple[float, float]:
    """Threshold maximising F1 = 2TP / (2TP + FP + FN), the organisers' ranking metric."""
    order = np.argsort(-scores)
    s, y = scores[order], labels[order].astype(int)
    tp = np.cumsum(y)                 # predicting the top i+1 scores as "same person"
    fp = np.cumsum(1 - y)
    f1 = 2 * tp / (tp + fp + y.sum())
    i = int(np.argmax(f1))
    t = (s[i] + s[i + 1]) / 2 if i + 1 < len(s) else s[i] - 1e-6
    return float(t), float(f1[i])


THRESHOLD_METHODS = ("max_f1", "max_accuracy", "eer")


@dataclass
class Calibration:
    method: str = "max_accuracy"        # or "max_f1" / "eer"
    global_threshold: float = 0.3
    use_bins: bool = True
    bin_edges: list = field(default_factory=list)       # quality edges, len = n_bins + 1
    bin_thresholds: list = field(default_factory=list)
    weights: dict = field(default_factory=dict)          # per-model fusion weights

    # ------------------------------------------------------------------
    def fit(self, scores, labels, pair_q=None, n_bins: int = 3, min_per_bin: int = 200,
            shrink: float = 0.5) -> "Calibration":
        scores, labels = np.asarray(scores), np.asarray(labels)
        fit_fn = {"max_accuracy": lambda s, y: best_accuracy_threshold(s, y)[0],
                  "max_f1": lambda s, y: best_f1_threshold(s, y)[0],
                  "eer": eer_threshold}[self.method]
        self.global_threshold = float(fit_fn(scores, labels))
        self.bin_edges, self.bin_thresholds = [], []
        if self.use_bins and pair_q is not None:
            pair_q = np.asarray(pair_q)
            edges = np.quantile(pair_q, np.linspace(0, 1, n_bins + 1))
            edges[0], edges[-1] = -np.inf, np.inf
            for lo, hi in zip(edges[:-1], edges[1:]):
                m = (pair_q > lo) & (pair_q <= hi)
                if m.sum() >= min_per_bin and 0 < labels[m].sum() < m.sum():
                    t = fit_fn(scores[m], labels[m])
                    # shrink towards the global threshold so small bins cannot overfit
                    t = shrink * t + (1 - shrink) * self.global_threshold
                else:
                    t = self.global_threshold
                self.bin_thresholds.append(float(t))
            self.bin_edges = [float(e) for e in edges]
        return self

    def thresholds_for(self, pair_q=None, n: int | None = None) -> np.ndarray:
        if not self.use_bins or not self.bin_edges or pair_q is None:
            return np.full(n if n is not None else len(pair_q), self.global_threshold)
        idx = np.clip(np.searchsorted(self.bin_edges, np.asarray(pair_q), side="left") - 1, 0, len(self.bin_thresholds) - 1)
        return np.array(self.bin_thresholds)[idx]

    def decide(self, scores, pair_q=None) -> np.ndarray:
        scores = np.asarray(scores)
        return (scores > self.thresholds_for(pair_q, len(scores))).astype(int)

    def save(self, path) -> None:
        Path(path).write_text(json.dumps(asdict(self), indent=2))

    @classmethod
    def load(cls, path) -> "Calibration":
        return cls(**json.loads(Path(path).read_text()))


# ---------------------------------------------------------------------------
def tune_fusion_weights(per_model: dict, labels: np.ndarray, grid=np.linspace(0, 1, 11)) -> dict:
    """Grid-search a 2-model fusion weight on validation pairs by EER (lower is better)."""
    from .metrics import compute_eer

    names = list(per_model)
    if len(names) == 1:
        return {names[0]: 1.0}
    if len(names) > 2:
        return {n: 1.0 for n in names}
    best, best_w = 1e9, 0.5
    for w in grid:
        s = w * per_model[names[0]] + (1 - w) * per_model[names[1]]
        e = compute_eer(s, labels)
        if e < best:
            best, best_w = e, w
    return {names[0]: float(best_w), names[1]: float(1 - best_w)}


# ---------------------------------------------------------------------------
def fused_embeddings(store, paths: list, weights: dict | None = None) -> np.ndarray:
    """Concatenate per-model embeddings scaled by sqrt(normalised weight): the dot product of two fused
    vectors equals the weighted mean of per-model cosines, i.e. the same score pair verification uses."""
    names = list(store.emb)
    w = weights or {m: 1.0 for m in names}
    tot = sum(w.get(m, 0) for m in names) or 1.0
    idx = [store.index[p] for p in paths]
    return np.concatenate([store.emb[m][idx] * np.sqrt(w.get(m, 0) / tot) for m in names], axis=1)


def build_templates(emb: np.ndarray, identities: list, quality: np.ndarray | None = None) -> tuple[list, np.ndarray]:
    """One template per identity: quality-weighted mean of its embeddings (renormalised)."""
    ids = sorted(set(identities))
    identities = np.asarray(identities)
    q = np.ones(len(emb)) if quality is None else np.clip(quality, 0.05, 1.0)
    T = []
    for i in ids:
        m = identities == i
        v = (emb[m] * q[m, None]).sum(0)
        T.append(v / (np.linalg.norm(v) + 1e-12))
    return ids, np.stack(T)


def identify(probe_emb: np.ndarray, gallery_ids: list, templates: np.ndarray, threshold: float | None = None):
    """Closed-set (threshold=None) or open-set identification. Returns (pred_ids, best_scores)."""
    sims = probe_emb @ templates.T
    best = sims.argmax(1)
    scores = sims[np.arange(len(sims)), best]
    preds = [gallery_ids[b] for b in best]
    if threshold is not None:
        preds = [p if s > threshold else "unknown" for p, s in zip(preds, scores)]
    return preds, scores

### `pakface/metrics.py`
Verification metrics required by the problem statement + plots.

In [ ]:
%%writefile pakface/metrics.py
"""Verification metrics required by the problem statement + plots.

Accuracy, precision, recall, F1, FAR, FRR, EER, ROC/AUC, TAR@FAR, DET.
Convention: label 1 = same person (genuine), 0 = different (impostor);
predict "same" when score > threshold.
"""
from __future__ import annotations

import numpy as np


def roc(scores, labels):
    from sklearn.metrics import roc_curve

    fpr, tpr, thr = roc_curve(np.asarray(labels).astype(int), np.asarray(scores), drop_intermediate=False)
    return fpr, tpr, thr


def compute_eer(scores, labels) -> float:
    fpr, tpr, _ = roc(scores, labels)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    # linear interpolation between the two closest points for a stable estimate
    if i + 1 < len(fpr):
        d0, d1 = fnr[i] - fpr[i], fnr[i + 1] - fpr[i + 1]
        if d0 != d1 and np.sign(d0) != np.sign(d1):
            a = d0 / (d0 - d1)
            return float(fpr[i] + a * (fpr[i + 1] - fpr[i]))
    return float((fpr[i] + fnr[i]) / 2)


def eer_threshold(scores, labels) -> float:
    fpr, tpr, thr = roc(scores, labels)
    i = int(np.nanargmin(np.abs((1 - tpr) - fpr)))
    t = thr[i]
    return float(t if np.isfinite(t) else np.max(scores))


def tar_at_far(scores, labels, far: float) -> float:
    fpr, tpr, _ = roc(scores, labels)
    ok = fpr <= far
    return float(tpr[ok].max()) if ok.any() else 0.0


def confusion_metrics(scores, labels, thresholds) -> dict:
    scores, labels = np.asarray(scores), np.asarray(labels).astype(int)
    thresholds = np.broadcast_to(np.asarray(thresholds, dtype=float), scores.shape)
    pred = (scores > thresholds).astype(int)
    tp = int(((pred == 1) & (labels == 1)).sum())
    tn = int(((pred == 0) & (labels == 0)).sum())
    fp = int(((pred == 1) & (labels == 0)).sum())
    fn = int(((pred == 0) & (labels == 1)).sum())
    n = len(labels)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "n_pairs": n, "n_genuine": int(labels.sum()), "n_impostor": int(n - labels.sum()),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "accuracy": (tp + tn) / n if n else 0.0,
        "precision": precision, "recall": recall, "f1": f1,
        "far": fp / (fp + tn) if fp + tn else 0.0,      # false accept rate
        "frr": fn / (fn + tp) if fn + tp else 0.0,      # false reject rate
    }


def full_report(scores, labels, thresholds) -> dict:
    from sklearn.metrics import roc_auc_score

    scores, labels = np.asarray(scores), np.asarray(labels).astype(int)
    rep = confusion_metrics(scores, labels, thresholds)
    if 0 < labels.sum() < len(labels):
        rep.update({
            "auc": float(roc_auc_score(labels, scores)),
            "eer": compute_eer(scores, labels),
            "tar@far=1e-2": tar_at_far(scores, labels, 1e-2),
            "tar@far=1e-3": tar_at_far(scores, labels, 1e-3),
            "tar@far=1e-4": tar_at_far(scores, labels, 1e-4),
        })
    rep["genuine_mean"] = float(scores[labels == 1].mean()) if labels.sum() else float("nan")
    rep["impostor_mean"] = float(scores[labels == 0].mean()) if (labels == 0).sum() else float("nan")
    return rep


# ---------------------------------------------------------------------------
def plot_all(scores, labels, threshold, out_prefix: str, title: str = "") -> list[str]:
    """ROC (log-FAR), DET, and genuine/impostor score histogram. Returns saved paths."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from scipy.stats import norm

    scores, labels = np.asarray(scores), np.asarray(labels).astype(int)
    fpr, tpr, _ = roc(scores, labels)
    eer = compute_eer(scores, labels)
    paths = []

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.plot(np.clip(fpr, 1e-5, 1), tpr, lw=2, color="#2563eb", label=f"ROC (EER={eer*100:.2f}%)")
    ax.set_xscale("log")
    ax.set_xlim(1e-4, 1)
    shown = tpr[fpr >= 1e-4]  # don't clip weaker curves (e.g. the naive ablation baseline) off the plot
    ax.set_ylim(min(0.5, np.floor(shown.min() * 10) / 10) if len(shown) else 0.5, 1.001)
    ax.set_xlabel("False Accept Rate (log)")
    ax.set_ylabel("True Accept Rate")
    ax.grid(alpha=0.3, which="both")
    ax.legend(loc="lower right")
    ax.set_title(title or "ROC")
    fig.tight_layout()
    paths.append(f"{out_prefix}_roc.png")
    fig.savefig(paths[-1], dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    fnr = 1 - tpr
    eps = 1e-5
    ax.plot(norm.ppf(np.clip(fpr, eps, 1 - eps)), norm.ppf(np.clip(fnr, eps, 1 - eps)), lw=2, color="#dc2626")
    ticks = [0.001, 0.01, 0.05, 0.2, 0.5]
    ax.set_xticks(norm.ppf(ticks))
    ax.set_xticklabels([f"{t*100:g}%" for t in ticks])
    ax.set_yticks(norm.ppf(ticks))
    ax.set_yticklabels([f"{t*100:g}%" for t in ticks])
    ax.set_xlabel("FAR")
    ax.set_ylabel("FRR")
    ax.grid(alpha=0.3)
    ax.set_title((title + " - " if title else "") + "DET")
    fig.tight_layout()
    paths.append(f"{out_prefix}_det.png")
    fig.savefig(paths[-1], dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    bins = np.linspace(min(scores.min(), -0.2), max(scores.max(), 1.0), 80)
    ax.hist(scores[labels == 0], bins=bins, alpha=0.6, color="#dc2626", label="impostor (different people)", density=True)
    ax.hist(scores[labels == 1], bins=bins, alpha=0.6, color="#16a34a", label="genuine (same person)", density=True)
    thr = float(np.median(np.atleast_1d(threshold)))
    ax.axvline(thr, color="k", ls="--", lw=1.5, label=f"threshold = {thr:.3f}")
    ax.set_xlabel("cosine similarity")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)
    ax.set_title((title + " - " if title else "") + "score distributions")
    fig.tight_layout()
    paths.append(f"{out_prefix}_hist.png")
    fig.savefig(paths[-1], dpi=160)
    plt.close(fig)
    return paths

### `pakface/augment.py`
Degradation and occlusion augmentation (pure OpenCV/NumPy, no extra deps).

In [ ]:
%%writefile pakface/augment.py
"""Degradation and occlusion augmentation (pure OpenCV/NumPy, no extra deps).

Two uses:
  * training-time augmentation on aligned 112x112 crops (fine-tuning), so the model
    sees CCTV blur, low light, masks, hijab/dupatta, caps, sunglasses;
  * test-time stress tests on full images (robustness tables for the report).

Occlusions are placed using the fixed ArcFace template landmark positions, which is
why they are only applied in crop space.
"""
from __future__ import annotations

import cv2
import numpy as np

# ---------------------------------------------------------------- photometric / sensor


def gaussian_blur(img, rng, lo=0.6, hi=3.0):
    s = rng.uniform(lo, hi)
    return cv2.GaussianBlur(img, (0, 0), s)


def motion_blur(img, rng, lo=5, hi=15):
    k = int(rng.integers(lo, hi + 1)) | 1
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), rng.uniform(0, 180), 1.0)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= kernel.sum() + 1e-8
    return cv2.filter2D(img, -1, kernel)


def downscale(img, rng, lo=14, hi=40):
    """Simulate a low-resolution face: shrink so the long side is lo..hi px, then upsample back."""
    h, w = img.shape[:2]
    target = rng.uniform(lo, hi)
    s = target / max(h, w)
    if s >= 1:
        return img
    small = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
    interp = [cv2.INTER_LINEAR, cv2.INTER_CUBIC, cv2.INTER_NEAREST][int(rng.integers(0, 3))]
    return cv2.resize(small, (w, h), interpolation=interp)


def jpeg(img, rng, lo=8, hi=45):
    q = int(rng.integers(lo, hi + 1))
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, q])
    return cv2.imdecode(buf, cv2.IMREAD_COLOR) if ok else img


def noise(img, rng, lo=3, hi=18):
    n = rng.normal(0, rng.uniform(lo, hi), img.shape)
    return np.clip(img.astype(np.float32) + n, 0, 255).astype(np.uint8)


def low_light(img, rng):
    g = rng.uniform(1.6, 3.2)
    out = ((img.astype(np.float32) / 255.0) ** g) * 255 * rng.uniform(0.5, 0.9)
    out = np.clip(out + rng.normal(0, rng.uniform(2, 8), img.shape), 0, 255)
    return out.astype(np.uint8)


def overexpose(img, rng):
    g = rng.uniform(0.35, 0.7)
    return np.clip(((img.astype(np.float32) / 255.0) ** g) * 255 * rng.uniform(1.0, 1.2), 0, 255).astype(np.uint8)


def color_cast(img, rng):
    gains = rng.uniform(0.75, 1.25, 3)
    return np.clip(img.astype(np.float32) * gains, 0, 255).astype(np.uint8)


def grayscale(img, rng=None):
    return cv2.cvtColor(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), cv2.COLOR_GRAY2BGR)


def cctv(img, rng):
    """Composite CCTV look: optional IR grayscale, low-res, blur, compression and sensor noise."""
    out = img
    if rng.random() < 0.5:
        out = grayscale(out)
    out = downscale(out, rng, 18, 36) if min(out.shape[:2]) <= 160 else downscale(out, rng, 60, 120)
    if rng.random() < 0.5:
        out = gaussian_blur(out, rng, 0.5, 1.5)
    out = noise(out, rng, 3, 12)
    return jpeg(out, rng, 10, 35)


# ---------------------------------------------------------------- occlusions (crop space, 112x112 template)


def _sc(pts, size):
    return (np.array(pts, dtype=np.float32) * size / 112.0).astype(np.int32)


def _fabric_color(rng):
    palette = [(235, 235, 235), (30, 30, 30), (200, 160, 110), (60, 40, 140), (40, 90, 160),
               (120, 60, 30), (90, 30, 120), (20, 110, 60), (180, 180, 60), (150, 150, 200)]
    c = np.array(palette[int(rng.integers(0, len(palette)))], dtype=np.float32)
    return tuple(int(v) for v in np.clip(c + rng.normal(0, 15, 3), 0, 255))


def _textured_fill(img, mask, color, rng):
    layer = np.full_like(img, color)
    if rng.random() < 0.5:  # simple print / fold shading
        h, w = img.shape[:2]
        yy, xx = np.mgrid[0:h, 0:w]
        stripes = (np.sin((xx * rng.uniform(0.2, 0.6) + yy * rng.uniform(0, 0.4))) * rng.uniform(5, 25))
        layer = np.clip(layer.astype(np.float32) + stripes[..., None], 0, 255).astype(np.uint8)
    out = img.copy()
    out[mask > 0] = layer[mask > 0]
    return out


def face_mask(img, rng):
    s = img.shape[0]
    top = rng.uniform(62, 74)
    poly = _sc([[14, top], [56, top - 4], [98, top], [102, 96], [82, 113], [30, 113], [10, 96]], s)
    m = np.zeros(img.shape[:2], np.uint8)
    cv2.fillPoly(m, [poly], 255)
    col = [(200, 170, 120), (235, 235, 235), (30, 30, 30), (150, 150, 200)][int(rng.integers(0, 4))]
    return _textured_fill(img, m, col, rng)


def sunglasses(img, rng):
    s = img.shape[0]
    out = img.copy()
    col = tuple(int(v) for v in rng.integers(5, 45, 3))
    for cx, cy in [(38.3, 51.7), (73.5, 51.5)]:
        c = tuple(int(v) for v in _sc([[cx, cy]], s)[0])
        ax = tuple(int(v) for v in _sc([[rng.uniform(11, 15), rng.uniform(8, 11)]], s)[0])
        cv2.ellipse(out, c, ax, 0, 0, 360, col, -1)
    cv2.line(out, tuple(_sc([[50, 50]], s)[0]), tuple(_sc([[62, 50]], s)[0]), col, max(1, s // 56))
    return out


def head_cover(img, rng):
    """Hijab / dupatta / turban style: fabric over the forehead and down the sides of the face."""
    s = img.shape[0]
    top = rng.uniform(22, 36)       # how low the fabric comes over the forehead
    side = rng.uniform(6, 18)       # how much of the cheeks/temples it hides
    left = [[0, 0], [112, 0], [112, 112], [112 - side, 112], [112 - side * 0.6, 60], [96, top], [56, top - 6],
            [16, top], [side * 0.6, 60], [side, 112], [0, 112]]
    m = np.zeros(img.shape[:2], np.uint8)
    cv2.fillPoly(m, [_sc(left, s)], 255)
    return _textured_fill(img, m, _fabric_color(rng), rng)


def cap(img, rng):
    s = img.shape[0]
    out = img.copy()
    brim = rng.uniform(18, 32)
    m = np.zeros(img.shape[:2], np.uint8)
    cv2.fillPoly(m, [_sc([[0, 0], [112, 0], [112, brim], [56, brim + 4], [0, brim]], s)], 255)
    out = _textured_fill(out, m, _fabric_color(rng), rng)
    # visor shadow over the eyes
    y0, y1 = int(brim * s / 112), int(min(112, brim + rng.uniform(15, 30)) * s / 112)
    out[y0:y1] = (out[y0:y1].astype(np.float32) * rng.uniform(0.45, 0.75)).astype(np.uint8)
    return out


def random_erase(img, rng):
    s = img.shape[0]
    out = img.copy()
    w, h = int(rng.uniform(0.15, 0.4) * s), int(rng.uniform(0.15, 0.4) * s)
    x, y = int(rng.integers(0, s - w)), int(rng.integers(0, s - h))
    out[y : y + h, x : x + w] = rng.integers(0, 255, 3)
    return out


PHOTOMETRIC = [gaussian_blur, motion_blur, downscale, jpeg, noise, low_light, overexpose, color_cast, grayscale, cctv]
OCCLUSIONS = [face_mask, sunglasses, head_cover, cap, random_erase]


def train_augment(crop: np.ndarray, rng: np.random.Generator, p_degrade: float = 0.6, p_occlude: float = 0.3,
                  p_flip: float = 0.5) -> np.ndarray:
    out = crop
    if rng.random() < p_flip:
        out = np.ascontiguousarray(out[:, ::-1])
    if rng.random() < p_occlude:
        out = OCCLUSIONS[int(rng.integers(0, len(OCCLUSIONS)))](out, rng)
    if rng.random() < p_degrade:
        for _ in range(int(rng.integers(1, 3))):
            out = PHOTOMETRIC[int(rng.integers(0, len(PHOTOMETRIC)))](out, rng)
    return out


# ---------------------------------------------------------------- stress tests on full images
def _stress(fn, **kw):
    def f(img, seed=0):
        return fn(img, np.random.default_rng(seed), **kw)
    return f


STRESS_TESTS = {
    "blur_sigma2": lambda img: cv2.GaussianBlur(img, (0, 0), 2.0 * max(img.shape[:2]) / 250),
    "motion_blur": _stress(motion_blur, lo=11, hi=11),
    "lowres_long_side_48": lambda img: downscale(img, np.random.default_rng(0), 48, 48),
    "lowres_long_side_32": lambda img: downscale(img, np.random.default_rng(0), 32, 32),
    "low_light": _stress(low_light),
    "jpeg_q10": lambda img: jpeg(img, np.random.default_rng(0), 10, 10),
    "cctv": _stress(cctv),
    "grayscale": lambda img: grayscale(img),
}

### `pakface/heads.py`
Margin-based classification heads for fine-tuning (PyTorch).

In [ ]:
%%writefile pakface/heads.py
"""Margin-based classification heads for fine-tuning (PyTorch).

AdaFaceHead follows the AdaFace paper (Kim et al., CVPR 2022): the margin adapts to
image quality, using the feature norm as a quality proxy. High-quality hard samples
get pushed harder; low-quality samples are not over-emphasised (they may be unrecognisable).
"""
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class AdaFaceHead(nn.Module):
    def __init__(self, emb_dim: int, n_classes: int, m: float = 0.4, h: float = 0.333, s: float = 64.0,
                 t_alpha: float = 0.01, init_centers: torch.Tensor | None = None):
        super().__init__()
        self.kernel = nn.Parameter(torch.empty(emb_dim, n_classes))
        if init_centers is not None:  # warm start: class centres from the pretrained model
            self.kernel.data.copy_(F.normalize(init_centers, dim=1).t())
        else:
            nn.init.normal_(self.kernel, std=0.01)
        self.m, self.h, self.s, self.t_alpha, self.eps = m, h, s, t_alpha, 1e-3
        self.register_buffer("batch_mean", torch.ones(1) * 20)
        self.register_buffer("batch_std", torch.ones(1) * 100)

    def forward(self, emb: torch.Tensor, norms: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        emb = F.normalize(emb, dim=1)
        cos = (emb @ F.normalize(self.kernel, dim=0)).clamp(-1 + self.eps, 1 - self.eps)
        safe = norms.clamp(0.001, 100).detach().view(-1, 1)
        with torch.no_grad():
            mean, std = safe.mean(), safe.std() if safe.numel() > 1 else torch.zeros_like(safe.mean())
            self.batch_mean = mean * self.t_alpha + (1 - self.t_alpha) * self.batch_mean
            self.batch_std = std * self.t_alpha + (1 - self.t_alpha) * self.batch_std
        scaler = ((safe - self.batch_mean) / (self.batch_std + self.eps) * self.h).clamp(-1, 1)

        onehot = torch.zeros_like(cos).scatter_(1, labels.view(-1, 1), 1.0)
        # angular margin  g_angle = -m * scaler
        theta = cos.acos()
        theta_m = (theta + onehot * (-self.m * scaler)).clamp(self.eps, math.pi - self.eps)
        cos_m = theta_m.cos()
        # additive margin g_add = m * scaler + m
        cos_m = cos_m - onehot * (self.m + self.m * scaler)
        return cos_m * self.s


class ArcFaceHead(nn.Module):
    def __init__(self, emb_dim: int, n_classes: int, m: float = 0.5, s: float = 64.0,
                 init_centers: torch.Tensor | None = None):
        super().__init__()
        self.kernel = nn.Parameter(torch.empty(emb_dim, n_classes))
        if init_centers is not None:
            self.kernel.data.copy_(F.normalize(init_centers, dim=1).t())
        else:
            nn.init.normal_(self.kernel, std=0.01)
        self.m, self.s = m, s

    def forward(self, emb, norms, labels):
        cos = (F.normalize(emb, dim=1) @ F.normalize(self.kernel, dim=0)).clamp(-1 + 1e-7, 1 - 1e-7)
        onehot = torch.zeros_like(cos).scatter_(1, labels.view(-1, 1), 1.0)
        theta = cos.acos()
        logits = torch.cos((theta + onehot * self.m).clamp(max=math.pi))
        return logits * self.s

### `pakface/antispoof.py`
Optional presentation-attack (anti-spoofing) check.

In [ ]:
%%writefile pakface/antispoof.py
"""Optional presentation-attack (anti-spoofing) check.

Off by default and NEVER applied to benchmark scoring: a benchmark photo of a
person is not a spoof attempt, and rejecting it would only cost accuracy. This
module is for the live demo / deployment story in the presentation.

Expected model: a MiniFASNet-style ONNX export (e.g. from the
minivision-ai Silent-Face-Anti-Spoofing project) that takes an 80x80 crop around
the face (box scaled by ~2.7) and outputs logits for [fake_print, real, fake_replay]
or similar. Verify the class order and input scaling on ~10 live and ~10
printed/screen photos before trusting it; adjust `real_index` / `scale_255`.
"""
from __future__ import annotations

import cv2
import numpy as np


class AntiSpoof:
    def __init__(self, onnx_path: str, input_size: int = 80, box_scale: float = 2.7,
                 real_index: int = 1, scale_255: bool = False):
        import onnxruntime as ort

        self.sess = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
        self.inp = self.sess.get_inputs()[0].name
        self.size, self.box_scale, self.real_index, self.scale_255 = input_size, box_scale, real_index, scale_255

    def _crop(self, img, bbox):
        h, w = img.shape[:2]
        x1, y1, x2, y2 = bbox[:4]
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        bw, bh = (x2 - x1) * self.box_scale, (y2 - y1) * self.box_scale
        x1, x2 = int(max(0, cx - bw / 2)), int(min(w, cx + bw / 2))
        y1, y2 = int(max(0, cy - bh / 2)), int(min(h, cy + bh / 2))
        return cv2.resize(img[y1:y2, x1:x2], (self.size, self.size))

    def real_probability(self, img_bgr: np.ndarray, bbox) -> float:
        x = self._crop(img_bgr, bbox).astype(np.float32)
        if self.scale_255:
            x /= 255.0
        x = x.transpose(2, 0, 1)[None]
        logits = self.sess.run(None, {self.inp: x})[0][0]
        p = np.exp(logits - logits.max())
        p /= p.sum()
        return float(p[self.real_index])

### `pakface/evaluation.py`
Shared evaluation logic used by 04_evaluate.py and 05_ablation.py.

In [ ]:
%%writefile pakface/evaluation.py
"""Shared evaluation logic used by 04_evaluate.py and 05_ablation.py."""
from __future__ import annotations

import numpy as np
import pandas as pd

from .matching import Calibration, fused_embeddings, tune_fusion_weights
from .metrics import compute_eer, confusion_metrics, full_report

DEMOGRAPHIC_ATTRS = ["gender", "age_group", "skin_tone", "region", "ethnicity"]
CONDITION_ATTRS = ["photo_type", "pose", "headwear", "glasses", "mask", "facial_hair", "lighting", "device"]
NEGATIVE_VALUES = {"", "none", "no", "0", "false", "nan", "normal", "frontal", "phone"}


def calibrate_and_score(store, pairs_val: pd.DataFrame, pairs_test: pd.DataFrame, method="max_accuracy",
                        use_bins=True, weights: dict | None = None, tune_weights=True):
    _, pm_val, q_val = store.subset_scores(pairs_val["path_a"].tolist(), pairs_val["path_b"].tolist())
    y_val = pairs_val["label"].to_numpy()
    if weights is None:
        weights = tune_fusion_weights(pm_val, y_val) if tune_weights else {m: 1.0 for m in pm_val}
    s_val, _, _ = store.subset_scores(pairs_val["path_a"].tolist(), pairs_val["path_b"].tolist(), weights)
    calib = Calibration(method=method, use_bins=use_bins, weights=weights).fit(s_val, y_val, q_val)

    s_test, pm_test, q_test = store.subset_scores(pairs_test["path_a"].tolist(), pairs_test["path_b"].tolist(), weights)
    thr = calib.thresholds_for(q_test, len(s_test))
    y_test = pairs_test["label"].to_numpy()
    rep = full_report(s_test, y_test, thr)
    rep["threshold_global"] = calib.global_threshold
    rep["val_eer"] = compute_eer(s_val, y_val)
    return calib, rep, s_test, q_test, thr, pm_test


def _attr_lookup(meta: pd.DataFrame, col: str) -> dict:
    return dict(zip(meta["path"], meta[col].astype(str).str.strip().str.lower()))


def group_breakdown(pairs: pd.DataFrame, scores, thresholds, meta: pd.DataFrame | None, store=None) -> pd.DataFrame:
    """Per-group metrics. Demographic groups: both images in the group. Conditions: either image has it."""
    y = pairs["label"].to_numpy()
    rows = []

    def add(group, value, mask):
        if mask.sum() < 20:
            return
        r = confusion_metrics(scores[mask], y[mask], thresholds[mask])
        r_eer = compute_eer(scores[mask], y[mask]) if 0 < y[mask].sum() < mask.sum() else float("nan")
        rows.append({"group": group, "value": value, "n_pairs": int(mask.sum()), "accuracy": r["accuracy"],
                     "far": r["far"], "frr": r["frr"], "f1": r["f1"], "eer": r_eer})

    if meta is not None:
        for col in DEMOGRAPHIC_ATTRS:
            if col in meta.columns:
                lk = _attr_lookup(meta, col)
                va = pairs["path_a"].map(lk).fillna("nan").to_numpy()
                vb = pairs["path_b"].map(lk).fillna("nan").to_numpy()
                for v in sorted(set(va) - {"nan"}):
                    add(col, v, (va == v) & (vb == v))
        for col in CONDITION_ATTRS:
            if col in meta.columns:
                lk = _attr_lookup(meta, col)
                va = pairs["path_a"].map(lk).fillna("nan").to_numpy()
                vb = pairs["path_b"].map(lk).fillna("nan").to_numpy()
                for v in sorted((set(va) | set(vb)) - NEGATIVE_VALUES):
                    add(col, v, (va == v) | (vb == v))

    if store is not None:  # automatic quality-flag groups, always available
        fl = {p: set(store.flags[i]) for p, i in store.index.items()}
        for flag in ["dark", "blurry", "low_res", "profile", "low_contrast", "no_detection"]:
            m = np.array([flag in fl.get(a, set()) or flag in fl.get(b, set())
                          for a, b in zip(pairs["path_a"], pairs["path_b"])])
            add("auto_quality_flag", flag, m)
    return pd.DataFrame(rows)


def identification_report(store, paths: list, identities: list, weights: dict | None = None,
                          threshold: float | None = None, unknown_frac: float = 0.3, seed: int = 0) -> dict:
    """1:N identification ("pick one face out of a whole school") on held-out identities.

    Each person is enrolled with ONE image (their highest-quality one); every other image is a probe.
    Closed set: rank-1 / rank-5 accuracy. Open set (if threshold given): unknown_frac of the people are
    removed from the gallery; DIR = known probes identified correctly AND above threshold,
    FPIR = unknown probes wrongly accepted as someone in the gallery.
    """
    df = pd.DataFrame({"path": paths, "identity": [str(i) for i in identities]}).drop_duplicates("path")
    df["q"] = store.quality[[store.index[p] for p in df["path"]]]
    df = df[df.groupby("identity")["path"].transform("size") >= 2]
    if df["identity"].nunique() < 2:
        return {}
    gal_idx = df.groupby("identity")["q"].idxmax()
    gal, probes = df.loc[gal_idx], df.drop(gal_idx)
    gids, truth = gal["identity"].to_numpy(), probes["identity"].to_numpy()
    sims = fused_embeddings(store, probes["path"].tolist(), weights) @ fused_embeddings(store, gal["path"].tolist(), weights).T
    ranked = gids[np.argsort(-sims, axis=1)]
    rep = {"n_gallery_ids": len(gal), "n_probes": len(probes),
           "rank1": float((ranked[:, 0] == truth).mean()),
           "rank5": float((ranked[:, :5] == truth[:, None]).any(1).mean())}
    if threshold is not None and len(gal) >= 3:
        rng = np.random.default_rng(seed)
        unknown = rng.choice(gids, max(1, int(round(len(gids) * unknown_frac))), replace=False)
        keep, known = ~np.isin(gids, unknown), ~np.isin(truth, unknown)
        s = sims[:, keep]
        best = s.argmax(1)
        accepted = s[np.arange(len(s)), best] > threshold
        rep.update({"open_set_threshold": float(threshold), "n_unknown_ids": len(unknown),
                    "dir": float(((gids[keep][best] == truth) & accepted)[known].mean()),
                    "fpir": float(accepted[~known].mean()) if (~known).any() else float("nan")})
    return rep


def identification_markdown(rep: dict) -> str:
    lines = ["### Identification (1:N) - one enrolment image per person", "", "| metric | value |", "|---|---|",
             f"| gallery identities | {rep['n_gallery_ids']} |", f"| probes | {rep['n_probes']} |",
             f"| rank-1 accuracy | {rep['rank1']*100:.3f}% |", f"| rank-5 accuracy | {rep['rank5']*100:.3f}% |"]
    if "dir" in rep:
        lines += [f"| open-set: unknown identities | {rep['n_unknown_ids']} (threshold {rep['open_set_threshold']:.4f}) |",
                  f"| DIR (known, correctly identified) | {rep['dir']*100:.3f}% |",
                  f"| FPIR (unknown, wrongly accepted) | {rep['fpir']*100:.3f}% |"]
    return "\n".join(lines)


def report_markdown(rep: dict, title: str) -> str:
    keys = ["accuracy", "precision", "recall", "f1", "far", "frr", "eer", "auc",
            "tar@far=1e-2", "tar@far=1e-3", "tar@far=1e-4", "threshold_global", "val_eer"]
    lines = [f"### {title}", "", "| metric | value |", "|---|---|"]
    for k in keys:
        if k in rep:
            v = rep[k]
            pct = k not in ("auc", "threshold_global")
            lines.append(f"| {k} | {v*100:.3f}% |" if pct else f"| {k} | {v:.4f} |")
    lines.append(f"| pairs | {rep['n_pairs']} ({rep['n_genuine']} genuine / {rep['n_impostor']} impostor) |")
    lines.append(f"| confusion | TP {rep['tp']}  TN {rep['tn']}  FP {rep['fp']}  FN {rep['fn']} |")
    return "\n".join(lines)

## `scripts/` — runnable stages

### `scripts/00_download_models.py`
Download every pretrained model the pipeline needs (run once per Kaggle/Colab session).

In [ ]:
%%writefile scripts/00_download_models.py
"""Download every pretrained model the pipeline needs (run once per Kaggle/Colab session).

  * buffalo_l   : SCRFD-10G detector (+ w600k_r50 ArcFace, used as the naive baseline)
  * antelopev2  : glintr100.onnx (ArcFace R100 trained on Glint360K)
  * AdaFace IR-101 WebFace12M from Hugging Face (CVLFace release)

    python scripts/00_download_models.py
"""
import glob
import os
import shutil
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from pakface.config import default_config  # noqa: E402


def fix_nested(pack: str) -> None:
    """antelopev2.zip unpacks to models/antelopev2/antelopev2/*.onnx on some versions."""
    root = os.path.expanduser(f"~/.insightface/models/{pack}")
    for f in glob.glob(os.path.join(root, "*", "*.onnx")):
        dst = os.path.join(root, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.move(f, dst)


def main():
    from insightface.app import FaceAnalysis

    for pack in ["buffalo_l", "antelopev2"]:
        print(f"== insightface pack {pack}")
        try:
            FaceAnalysis(name=pack, providers=["CPUExecutionProvider"])
        except Exception as e:  # nested-folder assertion on antelopev2 is expected the first time
            print(f"   first load raised {type(e).__name__}: {e} -> fixing folder layout")
        fix_nested(pack)
        print("   files:", sorted(os.listdir(os.path.expanduser(f"~/.insightface/models/{pack}"))))

    cfg = default_config()
    from pakface.embedders import download_hf_repo

    for spec in cfg.models:
        if spec.kind == "adaface_hf":
            print(f"== Hugging Face {spec.repo_id} -> {spec.path}")
            download_hf_repo(spec.repo_id, spec.path, os.environ.get("HF_TOKEN"))
    print("done. Next: python scripts/00_sanity_check.py --data data/raw")


if __name__ == "__main__":
    main()

### `scripts/00_sanity_check.py`
Five-minute sanity check before anything else.

In [ ]:
%%writefile scripts/00_sanity_check.py
"""Five-minute sanity check before anything else.

1. Confirms the GPU is used by ONNX Runtime and PyTorch.
2. Settles AdaFace's input channel order (RGB vs BGR) empirically: the right one gives a
   clearly larger genuine/impostor gap. Writes the winner into configs/pipeline.json.
3. Prints a quick EER for each backbone on a small sample of your data.

    python scripts/00_sanity_check.py --data data/raw --ids 40
"""
import argparse
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import numpy as np  # noqa: E402

from pakface.config import PipelineConfig, default_config  # noqa: E402
from pakface.data import make_pairs, scan  # noqa: E402
from pakface.metrics import compute_eer  # noqa: E402


def dprime(s, y):
    g, i = s[y == 1], s[y == 0]
    return (g.mean() - i.mean()) / np.sqrt(0.5 * (g.var() + i.var()) + 1e-9)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/raw")
    ap.add_argument("--ids", type=int, default=40)
    ap.add_argument("--config", default="configs/pipeline.json")
    ap.add_argument("--ctx", type=int, default=0)
    args = ap.parse_args()

    import onnxruntime as ort
    import torch

    print("onnxruntime providers:", ort.get_available_providers())
    print("torch cuda:", torch.cuda.is_available())

    df = scan(args.data)
    keep = df.groupby("identity").size()
    keep = keep[keep >= 3].index[: args.ids]
    df = df[df["identity"].isin(keep)]
    pairs = make_pairs(df, n_genuine_per_id=6, impostor_ratio=1.0, hard_negative_cols=())
    print(f"sample: {len(keep)} identities, {len(pairs)} pairs")

    cfg = PipelineConfig.from_json(args.config) if os.path.exists(args.config) else default_config()
    cfg.ctx_id = args.ctx
    cfg.cache_dir = "cache/sanity"

    from pakface.pipeline import FacePipeline

    results = {}
    for order in ["rgb", "bgr"]:
        for m in cfg.models:
            if m.kind == "adaface_hf":
                m.channel_order = order
        cfg.cache_dir = f"cache/sanity_{order}"
        pipe = FacePipeline(cfg)
        store = pipe.build_store(pairs["path_a"].tolist() + pairs["path_b"].tolist())
        _, per_model, _ = store.subset_scores(pairs["path_a"].tolist(), pairs["path_b"].tolist())
        y = pairs["label"].to_numpy()
        for name, s in per_model.items():
            results[(order, name)] = (dprime(s, y), compute_eer(s, y))
            print(f"[{order}] {name:16s} d'={results[(order, name)][0]:.2f}  EER={results[(order, name)][1]*100:.2f}%")

    ada = [m for m in cfg.models if m.kind == "adaface_hf"]
    if ada:
        best = max(["rgb", "bgr"], key=lambda o: results[(o, ada[0].name)][0])
        for m in ada:
            m.channel_order = best
        print(f"\n==> AdaFace channel order: {best.upper()} (written to {args.config})")
    cfg.cache_dir = "cache"
    os.makedirs(os.path.dirname(args.config), exist_ok=True)
    cfg.to_json(args.config)


if __name__ == "__main__":
    main()

### `scripts/import_public.py`
Import a downloaded public dataset into the unified layout  data/raw/<source>/<identity>/<image>.

In [ ]:
%%writefile scripts/import_public.py
"""Import a downloaded public dataset into the unified layout  data/raw/<source>/<identity>/<image>.

  python scripts/import_public.py --format generic --src /kaggle/input/bollywood-celeb-faces --name bollywood
  python scripts/import_public.py --format imfdb   --src /kaggle/input/imfdb/IMFDB_final  --name imfdb
  python scripts/import_public.py --format halfface --src downloads/halfFace/Dataset-1 --name halfface

generic  : <src>/<identity>/**/<images>      (identity = first folder level under src)
imfdb    : <src>/<Actor>/<Movie>/images/*.jpg (identity = actor; same rule as generic)
halfface : <src>/P_XXX/pXXX_<masked><idx><m|f>.jpg -> also writes gender + mask metadata

Images are symlinked (fast, no disk use); pass --copy if the source will disappear.
Metadata rows are appended to data/identities.csv and data/images.csv.
"""
import argparse
import os
import re
import shutil
import sys
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import pandas as pd  # noqa: E402

from pakface.data import IMG_EXT, normalize_identity  # noqa: E402

HALF_RE = re.compile(r"p(\d+)_([01])(\d)([mf])", re.I)


def place(src: Path, dst: Path, copy: bool):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    if copy:
        shutil.copy2(src, dst)
    else:
        try:
            os.symlink(src.resolve(), dst)
        except OSError:
            shutil.copy2(src, dst)


def append_csv(path: Path, rows: list, key: str):
    if not rows:
        return
    new = pd.DataFrame(rows)
    if path.exists():
        old = pd.read_csv(path)
        new = pd.concat([old, new]).drop_duplicates(subset=[key], keep="last")
    new.to_csv(path, index=False)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--format", choices=["generic", "imfdb", "halfface"], default="generic")
    ap.add_argument("--src", required=True)
    ap.add_argument("--name", required=True, help="source name, e.g. imfdb / halfface / bollywood")
    ap.add_argument("--raw", default="data/raw")
    ap.add_argument("--meta-dir", default="data")
    ap.add_argument("--max-per-id", type=int, default=60, help="cap images per identity (keeps sources balanced)")
    ap.add_argument("--copy", action="store_true")
    args = ap.parse_args()

    src = Path(args.src)
    out = Path(args.raw) / args.name
    id_rows, img_rows, n = [], [], 0
    for ident_dir in sorted(d for d in src.iterdir() if d.is_dir()):
        imgs = sorted(p for p in ident_dir.rglob("*") if p.suffix.lower() in IMG_EXT)[: args.max_per_id]
        if len(imgs) < 2:
            continue
        ident = f"{args.name}_{normalize_identity(ident_dir.name)}" if args.format == "halfface" else ident_dir.name
        for p in imgs:
            dst_name = f"{p.parent.name}_{p.name}" if p.parent != ident_dir else p.name
            place(p, out / ident / dst_name, args.copy)
            n += 1
            if args.format == "halfface":
                m = HALF_RE.search(p.stem)
                if m:
                    img_rows.append({"filename": dst_name, "mask": "mask" if m.group(2) == "1" else "none"})
                    id_rows.append({"identity": ident, "gender": "male" if m.group(4).lower() == "m" else "female"})
    append_csv(Path(args.meta_dir) / "identities.csv", id_rows, "identity")
    append_csv(Path(args.meta_dir) / "images.csv", img_rows, "filename")
    print(f"imported {n} images from {src} into {out} ({len(list(out.iterdir())) if out.exists() else 0} identities)")


if __name__ == "__main__":
    main()

### `scripts/import_google_form.py`
Import the Google Form photo collection into data/raw/own_collection/<code>/ + metadata CSVs.

In [ ]:
%%writefile scripts/import_google_form.py
"""Import the Google Form photo collection into data/raw/own_collection/<code>/ + metadata CSVs.

Inputs (both exported from Google, nothing to rearrange by hand):
  --csv    the responses spreadsheet as CSV (Form -> Responses -> spreadsheet -> File -> Download -> CSV)
  --files  the form's "File responses" folder: on Colab the mounted Drive folder, on Windows the folder
           extracted from Drive's zip download. Google keeps one sub-folder per upload question and renames
           every file "<original name> - <uploader's Google name>", which is how photos are matched to rows.

What it does:
  * keeps only rows whose consent box is ticked (18+ and agreed); everyone else is skipped entirely
  * gives every person a stable code P0001, P0002, ... (same email -> same code, also across re-runs, so
    newly arrived responses never reshuffle existing people between train/val/test)
  * converts every photo to an EXIF-rotated JPEG (also HEIC from iPhones, if pillow-heif is installed),
    which strips GPS/location metadata, and caps the long side at --max-side
  * writes the upload question of each photo as conditions for the per-group breakdown:
    photo_type, pose, glasses, headwear, mask, lighting, device
  * names/emails go ONLY to data/private/form_code_map.csv (keep it with one team member, delete after
    the competition); identities.csv / images.csv contain codes only

    python scripts/import_google_form.py --csv responses.csv --files "Face Collection (File responses)"
If some uploader names cannot be matched to a row, the script lists them; add them to a CSV with columns
uploader,email (or uploader,full_name) and pass it as --name-map.
"""
import argparse
import difflib
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from PIL import Image, ImageOps

# (header prefix, photo_type, conditions) - matched on the start of each upload question's text
CATEGORIES = [
    ("front face", "front", {}),
    ("side pose", "side", {"pose": "profile"}),
    ("glasses or sunglasses", "glasses", {"glasses": "glasses"}),
    ("head covering", "head_covering", {"headwear": "head_covering"}),
    ("face mask", "mask", {"mask": "mask"}),
    ("difficult lighting", "difficult_light", {"lighting": "difficult"}),
    ("low quality photos", "low_quality", {"device": "low_quality"}),
    ("different look", "different_look", {}),
]
DEFAULTS = {"pose": "frontal", "glasses": "none", "headwear": "none", "mask": "none", "lighting": "normal",
            "device": "phone"}
FIELDS = {"timestamp": "timestamp", "email address": "email", "full name": "full_name", "age group": "age_group",
          "gender": "gender", "i am 18": "consent"}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif", ".heic", ".heif"}


def norm(s) -> str:
    """Lowercase letters/digits only: survives ':' -> '_' on Windows zips, double spaces, punctuation."""
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def words(s) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()


def map_columns(columns) -> tuple[dict, dict]:
    """-> ({column: field}, {column: (photo_type, conditions)})"""
    fields, uploads = {}, {}
    for c in columns:
        w = words(c)
        for prefix, field in FIELDS.items():
            if w.startswith(words(prefix)):
                fields[c] = field
        for prefix, ptype, cond in CATEGORIES:
            if w.startswith(prefix):
                uploads[c] = (ptype, cond)
    missing = {"consent"} - set(fields.values())
    if missing or not uploads:
        raise SystemExit(f"could not find the consent column and upload questions in the CSV header: {list(columns)}")
    return fields, uploads


def person_key(row) -> str:
    return str(row.get("email") or "").strip().lower() or "name:" + norm(row.get("full_name", ""))


def index_files(files_dir: Path, upload_cols: list) -> list[dict]:
    """Every uploaded file with its question (from the sub-folder name) and uploader name (from the filename)."""
    keys = {c: norm(c) for c in upload_cols}
    out = []
    for p in files_dir.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in IMG_EXT:
            continue
        folder = norm(re.sub(r"\(file responses\)", "", p.parent.name, flags=re.I))
        col = max(keys, key=lambda c: len(_common_prefix(keys[c], folder)))
        # 12 matching characters, or the whole name for short questions like "Face mask" (folders may be truncated)
        if len(_common_prefix(keys[col], folder)) < min(12, len(keys[col])):
            col = None
        stem, sep, uploader = p.stem.rpartition(" - ")
        out.append({"path": p, "column": col, "uploader": uploader.strip() if sep else ""})
    return out


def _common_prefix(a: str, b: str) -> str:
    n = 0
    while n < min(len(a), len(b)) and a[n] == b[n]:
        n += 1
    return a[:n]


def to_jpeg(src: Path, dst: Path, max_side: int) -> bool:
    try:
        with Image.open(src) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            if max(im.size) > max_side:
                im.thumbnail((max_side, max_side), Image.LANCZOS)
            dst.parent.mkdir(parents=True, exist_ok=True)
            im.save(dst, "JPEG", quality=95)  # no exif= -> location / device metadata is dropped
        return True
    except Exception as e:
        hint = " (pip install pillow-heif for iPhone HEIC photos)" if src.suffix.lower() in {".heic", ".heif"} else ""
        print(f"  ! could not read {src.name}: {type(e).__name__}{hint}")
        return False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True)
    ap.add_argument("--files", required=True, help='the "File responses" folder (or any folder above it)')
    ap.add_argument("--raw", default="data/raw")
    ap.add_argument("--meta-dir", default="data")
    ap.add_argument("--source", default="own_collection")
    ap.add_argument("--code-map", default="data/private/form_code_map.csv")
    ap.add_argument("--name-map", default="", help="CSV uploader,email (or uploader,full_name) for unmatched uploaders")
    ap.add_argument("--max-side", type=int, default=2048)
    ap.add_argument("--min-photos", type=int, default=2)
    args = ap.parse_args()

    try:
        import pillow_heif

        pillow_heif.register_heif_opener()
    except ImportError:
        pass

    raw = pd.read_csv(args.csv, dtype=str).fillna("")
    fields, uploads = map_columns(raw.columns)
    df = raw.rename(columns=fields)
    if "timestamp" in df:
        df = df.assign(_t=pd.to_datetime(df["timestamp"], errors="coerce", format="mixed")).sort_values("_t", kind="stable")
    consented = df[df["consent"].str.strip() != ""]
    print(f"responses: {len(df)}, with consent: {len(consented)} (others skipped entirely)")

    # stable codes: reuse the existing map, append new people in submission order
    cmap_path = Path(args.code_map)
    cmap = pd.read_csv(cmap_path, dtype=str) if cmap_path.exists() else pd.DataFrame(columns=["code", "key", "full_name", "email"])
    code_of = dict(zip(cmap["key"], cmap["code"]))
    next_n = 1 + max([int(c[1:]) for c in cmap["code"]] or [0])
    people = {}
    for _, r in consented.iterrows():
        k = person_key(r)
        if k not in code_of:
            code_of[k] = f"P{next_n:04d}"
            next_n += 1
        people.setdefault(k, {"code": code_of[k], "full_name": r.get("full_name", ""), "email": r.get("email", ""),
                              "gender": r.get("gender", "").strip().lower(), "age_group": r.get("age_group", "").strip(),
                              "expected": Counter()})
        for col in uploads:
            people[k]["expected"][col] += str(r[col]).count("http")

    # uploader name (Google account) -> person
    by_name = defaultdict(set)
    for k, p in people.items():
        by_name[norm(p["full_name"])].add(k)
    manual = {}
    if args.name_map:
        nm = pd.read_csv(args.name_map, dtype=str).fillna("")
        for _, r in nm.iterrows():
            target = (r.get("email", "").strip().lower() if "email" in nm else "") or "name:" + norm(r.get("full_name", ""))
            manual[norm(r["uploader"])] = target

    def match(uploader: str):
        u = norm(uploader)
        if u in manual:
            return manual[u] if manual[u] in people else None
        if len(by_name.get(u, ())) == 1:
            return next(iter(by_name[u]))
        close = difflib.get_close_matches(u, [n for n, ks in by_name.items() if len(ks) == 1], n=1, cutoff=0.85)
        return next(iter(by_name[close[0]])) if close else None

    files = index_files(Path(args.files), list(uploads))
    print(f"uploaded files found: {len(files)}")
    # uploads by people who did not consent are expected to stay unmatched - they are never copied
    no_consent = {norm(n) for n in df[df["consent"].str.strip() == ""].get("full_name", [])}
    out_root = Path(args.raw) / args.source
    staged, counters = defaultdict(list), defaultdict(Counter)
    unmatched, no_question = Counter(), 0
    for f in sorted(files, key=lambda f: str(f["path"])):
        if f["column"] is None:
            no_question += 1
            continue
        k = match(f["uploader"])
        if k is None:
            if norm(f["uploader"]) not in no_consent:
                unmatched[f["uploader"] or "(no name in filename)"] += 1
            continue
        ptype, cond = uploads[f["column"]]
        counters[k][ptype] += 1
        staged[k].append((f, f"{people[k]['code']}_{ptype}_{counters[k][ptype]}.jpg", ptype, cond))

    id_rows, img_rows = [], []
    for k, p in people.items():
        if len(staged[k]) < args.min_photos:
            print(f"  ! {p['code']}: only {len(staged[k])} photo(s) found (min {args.min_photos}) - not imported")
            continue
        got = Counter()
        for f, name, ptype, cond in staged[k]:
            if to_jpeg(f["path"], out_root / p["code"] / name, args.max_side):
                img_rows.append({"filename": name, "photo_type": ptype, **DEFAULTS, **cond})
                got[f["column"]] += 1
        short = [f"{uploads[c][0]} {got[c]}/{n}" for c, n in p["expected"].items() if got[c] < n]
        if short:
            print(f"  ! {p['code']}: fewer photos than the form lists for " + ", ".join(short))
        id_rows.append({"identity": p["code"], "gender": p["gender"], "age_group": p["age_group"]})

    # withdrawals: people imported earlier but no longer consented / present lose their photos and metadata
    kept = {r["identity"] for r in id_rows}
    is_ours = re.compile(r"^P\d{4}$")
    for d in sorted(out_root.iterdir()) if out_root.exists() else []:
        if d.is_dir() and is_ours.match(d.name) and d.name not in kept:
            for f in d.iterdir():
                f.unlink()
            d.rmdir()
            print(f"  - removed {d.name}: no longer among the consented responses")
    meta = Path(args.meta_dir)
    meta.mkdir(parents=True, exist_ok=True)
    for path, rows, key in [(meta / "identities.csv", id_rows, "identity"), (meta / "images.csv", img_rows, "filename")]:
        old = pd.read_csv(path, dtype=str) if path.exists() else pd.DataFrame(columns=[key])
        old = old[~old[key].astype(str).str.match(r"^P\d{4}(_|$)")]  # our previous rows are fully replaced
        pd.concat([old, pd.DataFrame(rows)]).to_csv(path, index=False)
    cmap_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([{"code": p["code"], "key": k, "full_name": p["full_name"], "email": p["email"]}
                  for k, p in people.items()] +
                 [r for r in cmap.to_dict("records") if r["key"] not in people]).sort_values("code").to_csv(cmap_path, index=False)

    print(f"\nimported {len(img_rows)} photos of {len(kept)} people into {out_root}")
    print("photos per type:", dict(Counter(r["photo_type"] for r in img_rows)))
    if no_question:
        print(f"  ! {no_question} files were not inside a recognisable question folder - point --files at the "
              "\"File responses\" folder")
    if unmatched:
        print(f"  ! files from {len(unmatched)} uploader name(s) could not be matched to a consented response: "
              + ", ".join(f"{u} ({n})" for u, n in sorted(unmatched.items())))
        print("    -> add them to a CSV with columns uploader,email and re-run with --name-map")
    print(f"code <-> name map: {cmap_path}  (private: never share or commit; delete after the competition)")


if __name__ == "__main__":
    sys.exit(main())

### `scripts/01_prepare_data.py`
Prepare the merged dataset: scan sources, de-duplicate, detect + align + quality-score every image.

In [ ]:
%%writefile scripts/01_prepare_data.py
"""Prepare the merged dataset: scan sources, de-duplicate, detect + align + quality-score every image.

Outputs
  data/manifest.csv          one row per usable image (raw path, aligned crop path, identity, source, quality...)
  data/aligned/<id>/*.png    112x112 aligned crops (used by fine-tuning)
  data/report_prepare.txt    counts, detection-stage breakdown, quality flags
  data/label_conflicts.csv   near-identical images filed under DIFFERENT identities (fix these by hand!)

    python scripts/01_prepare_data.py --raw data/raw --identities data/identities.csv --images data/images.csv
"""
import argparse
import hashlib
import os
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import cv2  # noqa: E402
import pandas as pd  # noqa: E402
from tqdm import tqdm  # noqa: E402

from pakface.config import PipelineConfig, default_config  # noqa: E402
from pakface.data import attach_metadata, dhash, find_duplicates, scan  # noqa: E402
from pakface.pipeline import FacePipeline, load_image  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", default="data/raw")
    ap.add_argument("--out", default="data")
    ap.add_argument("--identities", default="data/identities.csv")
    ap.add_argument("--images", default="data/images.csv")
    ap.add_argument("--config", default="configs/pipeline.json")
    ap.add_argument("--min-train-quality", type=float, default=0.15,
                    help="crops below this are marked unusable for TRAINING (still allowed in val/test)")
    ap.add_argument("--dedupe-dist", type=int, default=4,
                    help="max dHash Hamming distance treated as duplicate (0 disables); burst shots are dropped")
    ap.add_argument("--ctx", type=int, default=0)
    args = ap.parse_args()

    df = scan(args.raw)
    df = attach_metadata(df, args.identities, args.images)
    print(f"scanned {len(df)} images, {df['identity'].nunique()} identities, sources: {dict(Counter(df['source']))}")

    # ---- de-duplication
    hashes, ok = [], []
    for p in tqdm(df["path"], desc="hashing"):
        try:
            g = cv2.cvtColor(load_image(p), cv2.COLOR_BGR2GRAY)
            hashes.append(dhash(cv2.resize(g, (9, 8), interpolation=cv2.INTER_AREA)))
            ok.append(True)
        except Exception as e:  # unreadable / corrupt file
            print(f"  unreadable {p}: {e}")
            hashes.append(0)
            ok.append(False)
    df = df[ok].reset_index(drop=True)
    hashes = [h for h, k in zip(hashes, ok) if k]
    drop, conflicts = find_duplicates(df, hashes, args.dedupe_dist) if args.dedupe_dist > 0 else (set(), [])
    pd.DataFrame(conflicts, columns=["path_a", "path_b"]).to_csv(Path(args.out) / "label_conflicts.csv", index=False)
    df = df.drop(index=list(drop)).reset_index(drop=True)
    print(f"removed {len(drop)} duplicate images; {len(conflicts)} cross-identity near-duplicates -> label_conflicts.csv")

    # ---- detect / align / quality
    cfg = PipelineConfig.from_json(args.config) if os.path.exists(args.config) else default_config()
    cfg.ctx_id = args.ctx
    pipe = FacePipeline(cfg, load_embedders=False)
    aligned_root = Path(args.out) / "aligned"
    rows = []
    for r in tqdm(df.itertuples(), total=len(df), desc="detect+align"):
        crop, q, flags, stage = pipe.preprocess(r.path)
        dst = aligned_root / r.identity / (Path(r.path).stem + f"_{hashlib.md5(r.path.encode()).hexdigest()[:8]}.png")
        dst.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(dst), crop)
        rows.append({"aligned_path": str(dst), "quality": q, "flags": "|".join(flags), "det_stage": stage})
    pipe.save_det_cache()
    df = pd.concat([df, pd.DataFrame(rows)], axis=1)
    df["train_usable"] = (df["quality"] >= args.min_train_quality) & (df["det_stage"] != "no_detection")
    df.to_csv(Path(args.out) / "manifest.csv", index=False)

    lines = [
        f"images: {len(df)}   identities: {df['identity'].nunique()}",
        f"per source: {dict(Counter(df['source']))}",
        f"detection stages: {dict(Counter(df['det_stage']))}",
        f"quality flags: {dict(Counter(f for fl in df['flags'].fillna('') for f in fl.split('|') if f))}",
        f"train-usable: {int(df['train_usable'].sum())} / {len(df)}",
        f"images per identity: min {df.groupby('identity').size().min()}, median {df.groupby('identity').size().median()}",
    ]
    Path(args.out, "report_prepare.txt").write_text("\n".join(lines))
    print("\n".join(lines))


if __name__ == "__main__":
    main()

### `scripts/02_split.py`
Identity-disjoint train / val / test split + verification pairs.

In [ ]:
%%writefile scripts/02_split.py
"""Identity-disjoint train / val / test split + verification pairs.

By default val and test identities come ONLY from your own consented Pakistani
collection (--eval-sources own_collection), because:
  * that is the population the hidden benchmark is drawn from;
  * public celebrity sets may already be inside the pretrained models' training data
    (WebFace / Glint360K are scraped from the web), which would inflate your numbers.

--own-train-frac (default 0.4) of those own-collection identities stay in TRAIN, because the problem
statement requires fine-tuning on your own collected data; the rest are split val/test.

    python scripts/02_split.py --eval-sources own_collection
"""
import argparse
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import pandas as pd  # noqa: E402

from pakface.data import assert_disjoint, identity_disjoint_split, make_pairs  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", default="data/manifest.csv")
    ap.add_argument("--out", default="data")
    ap.add_argument("--val-frac", type=float, default=None,
                    help="default 0.4 of eval-source identities, or 0.15 of all identities without eval sources")
    ap.add_argument("--test-frac", type=float, default=None,
                    help="default 0.6 of eval-source identities, or 0.15 of all identities without eval sources")
    ap.add_argument("--eval-sources", nargs="*", default=["own_collection"],
                    help="sources whose identities may be used for val/test (empty = any source)")
    ap.add_argument("--own-train-frac", type=float, default=0.4,
                    help="fraction of eval-source identities kept for TRAINING (fine-tuning must use your own "
                         "collected data); the rest is split val/test by --val-frac/--test-frac")
    ap.add_argument("--genuine-per-id", type=int, default=30)
    ap.add_argument("--impostor-ratio", type=float, default=1.0)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    df = pd.read_csv(args.manifest)
    eval_sources = [s for s in args.eval_sources if s in set(df["source"])] or None
    if args.eval_sources and not eval_sources:
        print(f"WARNING: none of {args.eval_sources} found; val/test will be drawn from all sources "
              "(fine for a practice run, but celebrity test numbers are inflated - don't report them)")
    # with eval sources the fractions split THOSE identities (the rest of train comes from other sources);
    # without them they are fractions of ALL identities, so most must stay in train
    dv, dt = (0.4, 0.6) if eval_sources else (0.15, 0.15)
    args.val_frac = dv if args.val_frac is None else args.val_frac
    args.test_frac = dt if args.test_frac is None else args.test_frac
    df = identity_disjoint_split(df, args.val_frac, args.test_frac, args.seed, test_sources=eval_sources,
                                 eval_source_train_frac=args.own_train_frac)
    assert_disjoint(df)
    df.to_csv(Path(args.out) / "splits.csv", index=False)

    hard = ("gender",) if "gender" in df.columns else ()
    for split in ["val", "test"]:
        sub = df[df["split"] == split]
        pairs = make_pairs(sub, args.genuine_per_id, args.impostor_ratio, seed=args.seed + (split == "test"),
                           hard_negative_cols=hard)
        pairs.to_csv(Path(args.out) / f"pairs_{split}.csv", index=False)
        print(f"{split:5s}: {sub['identity'].nunique():4d} identities, {len(sub):6d} images, "
              f"{int(pairs['label'].sum())} genuine + {int((pairs['label'] == 0).sum())} impostor pairs")
    tr = df[(df["split"] == "train") & df["train_usable"]]
    print(f"train: {tr['identity'].nunique():4d} identities, {len(tr):6d} usable images")
    if eval_sources:
        own = tr[tr["source"].isin(eval_sources)]
        print(f"       of which {own['identity'].nunique()} identities / {len(own)} images from {eval_sources}")
    # leakage check across splits (identity names)
    s = df.groupby("split")["identity"].apply(set)
    for a in s.index:
        for b in s.index:
            if a < b:
                assert not (s[a] & s[b]), f"identity overlap between {a} and {b}"
    print("OK: no identity appears in more than one split")


if __name__ == "__main__":
    main()

### `scripts/03_finetune.py`
Careful fine-tuning of AdaFace IR-101 on the Pakistani training identities.

In [ ]:
%%writefile scripts/03_finetune.py
"""Careful fine-tuning of AdaFace IR-101 on the Pakistani training identities.

The danger with fine-tuning a model trained on 12M images using a few thousand
images is catastrophic forgetting: it gets better on the training identities and
WORSE on new people (which is all the hidden benchmark contains). Guards used here:

  1. Partial unfreezing      - only the last --unfreeze-frac of the backbone trains.
  2. Frozen BatchNorm        - small batches would corrupt running statistics.
  3. Feature distillation    - loss term keeping embeddings close to the frozen original
                               (Learning-without-Forgetting style), weight --distill.
  4. Warm-started head       - class centres initialised from pretrained embeddings,
                               so early gradients are sensible instead of random.
  5. Tiny backbone LR + cosine decay, few epochs.
  6. Model selection on VALIDATION IDENTITIES (unseen people): the best epoch is always
     saved (fine-tuning is a required deliverable), and finetune_history.json records
     whether it beat the pretrained model. Whether to SHIP it is decided by
     04_evaluate / 05_ablation on the test split; "pretrained was better" is a legitimate
     result to report.
  7. Heavy degradation + occlusion augmentation (CCTV blur, low light, masks, hijab /
     dupatta, caps, sunglasses) so the adaptation is towards robustness, not memorisation.

    python scripts/03_finetune.py --epochs 6 --batch 64
"""
import argparse
import copy
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import cv2  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
import torch.nn.functional as F  # noqa: E402
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler  # noqa: E402
from tqdm import tqdm  # noqa: E402

from pakface.augment import train_augment  # noqa: E402
from pakface.config import PipelineConfig, default_config  # noqa: E402
from pakface.embedders import adaface_forward, load_adaface_hf  # noqa: E402
from pakface.heads import AdaFaceHead, ArcFaceHead  # noqa: E402
from pakface.metrics import compute_eer  # noqa: E402


def to_tensor(crop_bgr: np.ndarray, channel_order: str) -> torch.Tensor:
    x = crop_bgr.astype(np.float32)
    if channel_order == "rgb":
        x = x[..., ::-1]
    x = (x / 255.0 - 0.5) / 0.5
    return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))


class CropDataset(Dataset):
    def __init__(self, paths, labels, channel_order, augment=True, seed=0):
        self.paths, self.labels, self.co, self.augment = list(paths), list(labels), channel_order, augment
        self.seed = seed

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = cv2.imread(self.paths[i])
        if img.shape[:2] != (112, 112):
            img = cv2.resize(img, (112, 112))
        if self.augment:
            rng = np.random.default_rng([self.seed, i, int(torch.randint(1 << 30, (1,)).item())])
            img = train_augment(img, rng)
        return to_tensor(img, self.co), self.labels[i]


def freeze_bn(model):
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            m.eval()
            for p in m.parameters():
                p.requires_grad_(False)


def set_trainable(model, frac: float) -> int:
    params = list(model.named_parameters())
    for _, p in params:
        p.requires_grad_(False)
    k = max(1, int(len(params) * frac))
    n = 0
    for _, p in params[-k:]:
        p.requires_grad_(True)
        n += p.numel()
    return n


@torch.no_grad()
def embed_paths(model, paths, co, device, bs=128, flip=True):
    model.eval()
    out = []
    for i in range(0, len(paths), bs):
        imgs = [cv2.resize(cv2.imread(p), (112, 112)) for p in paths[i : i + bs]]
        x = torch.stack([to_tensor(im, co) for im in imgs]).to(device)
        f = adaface_forward(model, x).float()
        if flip:
            f = f + adaface_forward(model, torch.flip(x, dims=[3])).float()
        out.append(F.normalize(f, dim=1).cpu())
    return torch.cat(out)


def val_eer(model, pairs, raw2aligned, co, device):
    pa = [raw2aligned[p] for p in pairs["path_a"]]
    pb = [raw2aligned[p] for p in pairs["path_b"]]
    uniq = list(dict.fromkeys(pa + pb))
    E = embed_paths(model, uniq, co, device)
    idx = {p: i for i, p in enumerate(uniq)}
    s = (E[[idx[p] for p in pa]] * E[[idx[p] for p in pb]]).sum(1).numpy()
    return compute_eer(s, pairs["label"].to_numpy())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--splits", default="data/splits.csv")
    ap.add_argument("--pairs-val", default="data/pairs_val.csv")
    ap.add_argument("--config", default="configs/pipeline.json")
    ap.add_argument("--model-name", default="adaface_ir101")
    ap.add_argument("--out", default="checkpoints")
    ap.add_argument("--epochs", type=int, default=6)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--lr", type=float, default=2e-5, help="backbone LR")
    ap.add_argument("--head-lr", type=float, default=1e-3)
    ap.add_argument("--unfreeze-frac", type=float, default=0.25)
    ap.add_argument("--distill", type=float, default=1.0, help="weight of the stay-close-to-pretrained loss")
    ap.add_argument("--head", choices=["adaface", "arcface"], default="adaface")
    ap.add_argument("--min-images", type=int, default=3, help="identities with fewer train images are skipped")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    cfg = PipelineConfig.from_json(args.config) if os.path.exists(args.config) else default_config()
    spec = next(m for m in cfg.models if m.name == args.model_name)
    co = spec.channel_order
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    df = pd.read_csv(args.splits)
    raw2aligned = dict(zip(df["path"], df["aligned_path"]))
    tr = df[(df["split"] == "train") & (df["train_usable"])]
    counts = tr.groupby("identity").size()
    tr = tr[tr["identity"].isin(counts[counts >= args.min_images].index)]
    classes = sorted(tr["identity"].unique())
    cls_idx = {c: i for i, c in enumerate(classes)}
    labels = tr["identity"].map(cls_idx).to_numpy()
    print(f"train: {len(tr)} images, {len(classes)} identities, device={device}")
    pairs_val = pd.read_csv(args.pairs_val)

    student = load_adaface_hf(spec.path, spec.repo_id, os.environ.get("HF_TOKEN")).to(device)
    teacher = copy.deepcopy(student).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    base_eer = val_eer(student, pairs_val, raw2aligned, co, device)
    print(f"pretrained val EER: {base_eer*100:.3f}%")

    # warm-start class centres from pretrained embeddings
    E = embed_paths(teacher, tr["aligned_path"].tolist(), co, device, flip=False)
    centers = torch.zeros(len(classes), E.shape[1])
    centers.index_add_(0, torch.from_numpy(labels.copy()), E)
    head_cls = AdaFaceHead if args.head == "adaface" else ArcFaceHead
    head = head_cls(E.shape[1], len(classes), init_centers=centers).to(device)

    n_train = set_trainable(student, args.unfreeze_frac)
    print(f"trainable backbone params: {n_train/1e6:.2f}M (last {args.unfreeze_frac*100:.0f}% of tensors)")
    opt = torch.optim.AdamW([
        {"params": [p for p in student.parameters() if p.requires_grad], "lr": args.lr, "weight_decay": 5e-4},
        {"params": head.parameters(), "lr": args.head_lr, "weight_decay": 0.0},
    ])
    # class-balanced sampling: big public identities must not drown the Pakistani ones
    w = 1.0 / np.sqrt(counts.reindex(tr["identity"]).to_numpy())
    sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(tr), replacement=True)
    loader = DataLoader(CropDataset(tr["aligned_path"], labels, co, augment=True, seed=args.seed),
                        batch_size=args.batch, sampler=sampler, num_workers=args.workers, drop_last=True,
                        pin_memory=device.type == "cuda")
    steps = args.epochs * len(loader)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[args.lr, args.head_lr], total_steps=steps,
                                                pct_start=0.1, anneal_strategy="cos")
    try:
        scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")
    except (AttributeError, TypeError, RuntimeError):
        scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    os.makedirs(args.out, exist_ok=True)
    best_path = Path(args.out) / f"{args.model_name}_finetuned.pt"
    best_eer, best_epoch, history = float("inf"), 0, [{"epoch": 0, "val_eer": base_eer}]
    for ep in range(1, args.epochs + 1):
        student.train()
        freeze_bn(student)
        pbar = tqdm(loader, desc=f"epoch {ep}/{args.epochs}")
        for x, y in pbar:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                raw = adaface_forward(student, x)
                with torch.no_grad():
                    t = F.normalize(adaface_forward(teacher, x).float(), dim=1)
            raw = raw.float()
            norms = raw.norm(dim=1)
            logits = head(raw, norms, y)
            ce = F.cross_entropy(logits, y)
            distill = (1 - (F.normalize(raw, dim=1) * t).sum(1)).mean()
            loss = ce + args.distill * distill
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            pbar.set_postfix(ce=f"{ce.item():.3f}", distill=f"{distill.item():.4f}")
        eer = val_eer(student, pairs_val, raw2aligned, co, device)
        history.append({"epoch": ep, "val_eer": eer})
        print(f"epoch {ep}: val EER {eer*100:.3f}%  (pretrained {base_eer*100:.3f}%, "
              f"best fine-tuned {min(best_eer, eer)*100:.3f}%)")
        if eer < best_eer:
            best_eer, best_epoch = eer, ep
            torch.save(student.state_dict(), best_path)
            print(f"  saved {best_path}")

    beat = best_eer < base_eer
    Path(args.out, "finetune_history.json").write_text(json.dumps(
        {"pretrained_val_eer": base_eer, "best_val_eer": best_eer, "best_epoch": best_epoch,
         "beat_pretrained": beat, "epochs": history}, indent=2))
    print(f"\nBest fine-tuned val EER {best_eer*100:.3f}% (epoch {best_epoch}) vs pretrained {base_eer*100:.3f}%"
          f" -> {'better' if beat else 'NOT better'} on unseen validation identities.\n"
          f"Compare on the TEST split: python scripts/05_ablation.py --finetuned {best_path}")
    if beat:
        print(f"To ship it: set models[{args.model_name}].finetuned_ckpt = \"{best_path}\" in configs/pipeline.json,"
              " then re-run 04_evaluate.py.")
    else:
        print("Keep shipping the pretrained model, and report this honestly in the ablation (it is a real finding).")


if __name__ == "__main__":
    main()

### `scripts/04_evaluate.py`
Full evaluation of the pipeline on the held-out TEST identities.

In [ ]:
%%writefile scripts/04_evaluate.py
"""Full evaluation of the pipeline on the held-out TEST identities.

* thresholds + fusion weights are fitted on VALIDATION pairs only (never on test)
* reports accuracy, precision, recall, F1, FAR, FRR, EER, AUC, TAR@FAR
* ROC, DET and score-distribution plots
* per-group breakdown (gender, age group, skin tone, region, headwear, glasses, mask, lighting, quality flags)
* 1:N identification on the test identities: rank-1 / rank-5, open-set DIR / FPIR
* writes results/calibration.json, which predict.py uses on the hidden benchmark

    python scripts/04_evaluate.py --name full_pipeline
"""
import argparse
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import pandas as pd  # noqa: E402

from pakface.config import PipelineConfig, default_config  # noqa: E402
from pakface.evaluation import (  # noqa: E402
    calibrate_and_score, group_breakdown, identification_markdown, identification_report, report_markdown,
)
from pakface.matching import Calibration  # noqa: E402
from pakface.metrics import full_report, plot_all  # noqa: E402
from pakface.pipeline import FacePipeline  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/pipeline.json")
    ap.add_argument("--pairs-val", default="data/pairs_val.csv")
    ap.add_argument("--pairs-test", default="data/pairs_test.csv")
    ap.add_argument("--splits", default="data/splits.csv", help="for per-group metadata")
    ap.add_argument("--method", choices=["max_f1", "max_accuracy", "eer"], default="max_f1",
                    help="threshold objective; max_f1 because F1 is the organisers' ranking metric")
    ap.add_argument("--no-bins", action="store_true", help="single global threshold")
    ap.add_argument("--name", default="full_pipeline")
    ap.add_argument("--out", default="results")
    ap.add_argument("--ctx", type=int, default=0)
    args = ap.parse_args()

    cfg = PipelineConfig.from_json(args.config) if os.path.exists(args.config) else default_config()
    cfg.ctx_id = args.ctx
    out = Path(args.out) / args.name
    out.mkdir(parents=True, exist_ok=True)

    pv, pt = pd.read_csv(args.pairs_val), pd.read_csv(args.pairs_test)
    pipe = FacePipeline(cfg)
    store = pipe.build_store(pd.concat([pv["path_a"], pv["path_b"], pt["path_a"], pt["path_b"]]).tolist())

    use_bins = cfg.quality_binned_threshold and not args.no_bins
    calib, rep, s, q, thr, per_model = calibrate_and_score(store, pv, pt, args.method, use_bins)
    calib.save(Path(args.out) / "calibration.json")
    calib.save(out / "calibration.json")

    y = pt["label"].to_numpy()
    md = [report_markdown(rep, f"{args.name} - test identities (threshold fitted on val)")]
    _, per_model_val, _ = store.subset_scores(pv["path_a"].tolist(), pv["path_b"].tolist())
    for m, sm in per_model.items():
        c = Calibration(method=args.method, use_bins=False).fit(per_model_val[m], pv["label"].to_numpy())
        md.append(report_markdown(full_report(sm, y, c.global_threshold), f"single model: {m}"))
    plots = plot_all(s, y, thr, str(out / "test"), title=args.name)

    meta = pd.read_csv(args.splits) if os.path.exists(args.splits) else None
    groups = group_breakdown(pt, s, thr, meta, store)
    groups.to_csv(out / "per_group.csv", index=False)
    if len(groups):
        md.append("### Per-group breakdown (test)\n\n" + groups.to_markdown(index=False, floatfmt=".4f"))

    ident = identification_report(store, pd.concat([pt["path_a"], pt["path_b"]]).tolist(),
                                  pd.concat([pt["id_a"], pt["id_b"]]).tolist(), calib.weights, calib.global_threshold)
    if ident:
        rep["identification"] = ident
        md.append(identification_markdown(ident))
    pd.DataFrame({"path_a": pt["path_a"], "path_b": pt["path_b"], "label": y, "score": s, "pair_quality": q,
                  "threshold": thr, "pred": (s > thr).astype(int)}).to_csv(out / "test_scores.csv", index=False)
    # hardest errors for the "challenges" slide
    errs = pd.read_csv(out / "test_scores.csv")
    errs = errs[errs["pred"] != errs["label"]].assign(margin=lambda d: (d["score"] - d["threshold"]).abs())
    errs.sort_values("margin", ascending=False).head(40).to_csv(out / "worst_errors.csv", index=False)

    (out / "metrics.json").write_text(json.dumps(rep, indent=2))
    (out / "summary.md").write_text("\n\n".join(md))
    print("\n\n".join(md))
    print(f"\nplots: {plots}\ncalibration -> {Path(args.out) / 'calibration.json'}")


if __name__ == "__main__":
    main()

### `scripts/05_ablation.py`
Ablation study + robustness stress tests (the "how much does each stage contribute" table).

In [ ]:
%%writefile scripts/05_ablation.py
"""Ablation study + robustness stress tests (the "how much does each stage contribute" table).

Every row is evaluated identically: thresholds/weights fitted on VAL identities, metrics on TEST identities.

Rows
  naive baseline        buffalo_l ArcFace R50 only, no retries, no TTA, no enhancement, single threshold
  FULL PIPELINE         everything on
  - flip TTA            full minus test-time flip augmentation
  - enhancement         full minus adaptive low-light/contrast enhancement
  - alignment           full minus 5-point alignment (plain bbox crop)
  - detection retries   full minus pad/upscale/brighten retry cascade
  - quality thresholds  full with one global threshold instead of quality-binned thresholds
  only AdaFace          single backbone
  only Glint R100       single backbone
  + fine-tuned AdaFace  (with --finetuned path) full pipeline using the fine-tuned checkpoint

Stress tests: image B of every test pair is degraded (blur, low-res, low light, JPEG, CCTV, grayscale)
for the naive baseline and the full pipeline.

    python scripts/05_ablation.py [--finetuned checkpoints/adaface_ir101_finetuned.pt] [--skip-stress]
"""
import argparse
import copy
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from pakface.augment import STRESS_TESTS  # noqa: E402
from pakface.config import ModelSpec, PipelineConfig, default_config  # noqa: E402
from pakface.evaluation import calibrate_and_score  # noqa: E402
from pakface.metrics import compute_eer, confusion_metrics  # noqa: E402
from pakface.pipeline import FacePipeline  # noqa: E402

COLS = ["accuracy", "f1", "precision", "recall", "far", "frr", "eer", "auc", "tar@far=1e-3"]


class ModelCache:
    def __init__(self):
        self.detectors, self.embedders = {}, {}

    def pipeline(self, cfg):
        from pakface.detect import FaceDetector
        from pakface.embedders import build_embedders

        dkey = (cfg.detector_pack, cfg.det_size, cfg.det_thresh, cfg.pad_retry, cfg.upscale_retry, cfg.enhance_for_detection)
        if dkey not in self.detectors:
            self.detectors[dkey] = FaceDetector(cfg.detector_pack, cfg.det_size, cfg.det_thresh, cfg.ctx_id,
                                                cfg.pad_retry, cfg.upscale_retry, cfg.enhance_for_detection)
        embs = []
        for spec in cfg.models:
            k = (spec.name, spec.kind, spec.path, spec.channel_order, spec.finetuned_ckpt)
            if k not in self.embedders:
                one = copy.deepcopy(cfg)
                one.models = [spec]
                self.embedders[k] = build_embedders(one)[0]
            embs.append(self.embedders[k])
        return FacePipeline(cfg, detector=self.detectors[dkey], embedders=embs)


def naive_cfg(base: PipelineConfig) -> PipelineConfig:
    c = copy.deepcopy(base)
    c.models = [ModelSpec(name="arcface_r50_buffalo", kind="insightface_onnx",
                          path="~/.insightface/models/buffalo_l/w600k_r50.onnx")]
    c.pad_retry = c.upscale_retry = c.enhance_for_detection = False
    c.det_thresh = 0.5
    c.use_flip_tta = c.use_enhancement = False
    c.quality_binned_threshold = False
    return c


def off(**kw):
    def f(c):
        for k, v in kw.items():
            setattr(c, k, v)
    return f


def two_store_scores(store_a, store_b, pa, pb, weights):
    ia = np.array([store_a.index[p] for p in pa])
    ib = np.array([store_b.index[p] for p in pb])
    per = {m: np.sum(store_a.emb[m][ia] * store_b.emb[m][ib], axis=1) for m in store_a.emb}
    wsum = sum(weights.get(m, 0) for m in per) or 1.0
    fused = sum(weights.get(m, 0) * s for m, s in per.items()) / wsum
    return fused, np.minimum(store_a.quality[ia], store_b.quality[ib])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/pipeline.json")
    ap.add_argument("--pairs-val", default="data/pairs_val.csv")
    ap.add_argument("--pairs-test", default="data/pairs_test.csv")
    ap.add_argument("--finetuned", default="", help="checkpoint from 03_finetune.py")
    ap.add_argument("--method", choices=["max_f1", "max_accuracy", "eer"], default="max_f1",
                    help="threshold objective; max_f1 because F1 is the organisers' ranking metric")
    ap.add_argument("--skip-stress", action="store_true")
    ap.add_argument("--out", default="results/ablation")
    ap.add_argument("--ctx", type=int, default=0)
    args = ap.parse_args()

    base = PipelineConfig.from_json(args.config) if os.path.exists(args.config) else default_config()
    base.ctx_id = args.ctx
    for m in base.models:  # the ablation's FULL row is the pretrained pipeline; fine-tuned gets its own row
        m.finetuned_ckpt = ""
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    pv, pt = pd.read_csv(args.pairs_val), pd.read_csv(args.pairs_test)
    paths = pd.concat([pv["path_a"], pv["path_b"], pt["path_a"], pt["path_b"]]).tolist()
    cache = ModelCache()

    names = [m.name for m in base.models]
    variants = [
        ("naive baseline (ArcFace R50, no extras)", None, {}),
        ("FULL PIPELINE", off(), {}),
        ("- flip TTA", off(use_flip_tta=False), {}),
        ("- enhancement", off(use_enhancement=False), {}),
        ("- alignment (bbox crop)", off(use_alignment=False), {}),
        ("- detection retry cascade", off(pad_retry=False, upscale_retry=False, enhance_for_detection=False), {}),
        ("- quality-binned thresholds", off(quality_binned_threshold=False), {}),
    ]
    variants += [(f"only {n}", off(), {"weights": {k: float(k == n) for k in names}}) for n in names]
    if args.finetuned:
        def ft(c):
            for m in c.models:
                if m.kind == "adaface_hf":
                    m.finetuned_ckpt = args.finetuned
        variants.append(("+ fine-tuned AdaFace", ft, {}))

    rows, calibs, stores = [], {}, {}
    for name, mutate, extra in variants:
        cfg = naive_cfg(base) if mutate is None else copy.deepcopy(base)
        if mutate is not None:
            mutate(cfg)
        print(f"\n=== {name}")
        store = cache.pipeline(cfg).build_store(paths)
        calib, rep, *_ = calibrate_and_score(store, pv, pt, args.method, cfg.quality_binned_threshold,
                                             weights=extra.get("weights"))
        rows.append({"variant": name, **{k: rep.get(k, np.nan) for k in COLS}, "weights": json.dumps(calib.weights)})
        calibs[name], stores[name] = calib, (cfg, store)
        print({k: round(rep.get(k, np.nan), 5) for k in COLS})

    tab = pd.DataFrame(rows)
    full_acc = tab.loc[tab["variant"] == "FULL PIPELINE", "accuracy"].iloc[0]
    full_eer = tab.loc[tab["variant"] == "FULL PIPELINE", "eer"].iloc[0]
    tab["d_accuracy_vs_full"] = tab["accuracy"] - full_acc
    tab["d_eer_vs_full"] = tab["eer"] - full_eer
    tab.to_csv(out / "ablation.csv", index=False)
    pct = tab.copy()
    for c in COLS:
        pct[c] = pct[c].map(lambda v: f"{v:.4f}") if c == "auc" else (pct[c] * 100).map(lambda v: f"{v:.2f}")
    for c in ["d_accuracy_vs_full", "d_eer_vs_full"]:
        pct[c] = (pct[c] * 100).map(lambda v: f"{v:+.2f}")
    md = ["## Ablation (test identities, %; thresholds fitted on val)", "",
          pct.drop(columns=["weights"]).to_markdown(index=False)]

    if not args.skip_stress:
        srows = []
        pa, pb, y = pt["path_a"].tolist(), pt["path_b"].tolist(), pt["label"].to_numpy()
        for vname in ["naive baseline (ArcFace R50, no extras)", "FULL PIPELINE"]:
            cfg, clean = stores[vname]
            calib = calibs[vname]
            pipe = cache.pipeline(cfg)
            for sname, fn in STRESS_TESTS.items():
                print(f"--- stress {sname} on {vname}")
                deg = pipe.build_store(pb, degrade_fn=fn, tag=f"stress_{sname}")
                s, q = two_store_scores(clean, deg, pa, pb, calib.weights)
                r = confusion_metrics(s, y, calib.thresholds_for(q, len(s)))
                srows.append({"pipeline": "naive" if "naive" in vname else "full", "degradation": sname,
                              "accuracy": r["accuracy"], "far": r["far"], "frr": r["frr"], "eer": compute_eer(s, y)})
        st = pd.DataFrame(srows)
        st.to_csv(out / "stress_tests.csv", index=False)
        piv = st.pivot(index="degradation", columns="pipeline", values=["accuracy", "eer"]) * 100
        piv.columns = [f"{a}_{b} (%)" for a, b in piv.columns]
        md += ["", "## Robustness stress tests (image B degraded; %)", "", piv.round(2).to_markdown()]

    (out / "ablation.md").write_text("\n".join(md))
    print("\n".join(md))


if __name__ == "__main__":
    main()

### `scripts/predict.py`
Run the frozen pipeline on the HIDDEN BENCHMARK (or any new data).

In [ ]:
%%writefile scripts/predict.py
"""Run the frozen pipeline on the HIDDEN BENCHMARK (or any new data).

Four modes, so you are ready whatever format the TechFest team uses:

1) Verification pairs:
     python scripts/predict.py pairs --csv benchmark/pairs.csv --root benchmark/images --out submission.csv
   The CSV needs two image columns (auto-detected: img1/img2, image1/image2, path_a/path_b, left/right, a/b,
   or simply the first two columns). Output adds `score` (cosine) and `prediction` (1 = same person).
   If the CSV also has a ground-truth column (label/is_same/same/match), metrics are printed.

2) Identification (gallery of known people + probe images):
     python scripts/predict.py identify --gallery benchmark/gallery --probes benchmark/probes --out ids.csv [--open-set]
   gallery/<person>/*.jpg ; probes/*.jpg (or probes/<person>/*.jpg to get accuracy).

3) Search: find one person in a set of photos ("pick a single face out of an entire school"):
     python scripts/predict.py search --query person.jpg --photos school_photos --out matches.csv --draw found/
   --query may be one image or a folder of images of the same person.

4) Single pair:
     python scripts/predict.py one --img1 a.jpg --img2 b.jpg

Group photos (--multi-face auto, the default): an image with 2+ faces is split into one entry per face;
single-person images keep the normal single-face pipeline, so their results are unchanged.
  pairs    : score = best-matching face combination ("is the person in A also in B?")
  identify : one output row per face; faces in group photos below the threshold become "unknown"
  search   : every face of every photo is compared with the query person
Gallery images are always treated as single-person enrolment photos.

Uses results/calibration.json (threshold + fusion weights fitted on your validation identities).
"""
import argparse
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from pakface.config import PipelineConfig, default_config  # noqa: E402
from pakface.data import IMG_EXT  # noqa: E402
from pakface.matching import Calibration, build_templates, fused_embeddings, identify  # noqa: E402
from pakface.metrics import full_report  # noqa: E402
from pakface.pipeline import FacePipeline, load_image, split_face_key  # noqa: E402

PAIR_COLS = [("img1", "img2"), ("image1", "image2"), ("path_a", "path_b"), ("left", "right"), ("a", "b"),
             ("image_1", "image_2"), ("img_1", "img_2"), ("probe", "reference"), ("query", "reference")]
LABEL_COLS = ["label", "is_same", "same", "match", "issame", "ground_truth", "gt"]


def load(args):
    cfg = PipelineConfig.from_json(args.config) if os.path.exists(args.config) else default_config()
    cfg.ctx_id = args.ctx
    cfg.cache_dir = args.cache
    calib = Calibration.load(args.calibration) if os.path.exists(args.calibration) else None
    if calib is None:
        print(f"WARNING: {args.calibration} not found -> using an uncalibrated default threshold of 0.30")
        calib = Calibration(global_threshold=0.30, use_bins=False)
    return FacePipeline(cfg), calib


def resolve(p, root):
    p = str(p).strip()
    return p if (os.path.isabs(p) or not root) else os.path.join(root, p)


def list_images(d):
    return sorted(str(p) for p in Path(d).rglob("*") if p.suffix.lower() in IMG_EXT)


def faces_for(pipe, paths, args, force=False):
    """{image: [face keys]} - one key per face for group photos, [image] otherwise."""
    if args.multi_face == "off" and not force:
        return {p: [p] for p in dict.fromkeys(paths)}, {}
    return pipe.expand_faces(paths, args.min_face_rel)


def face_index(key):
    k = split_face_key(key)[1]
    return "" if k is None else k


def bbox_str(pipe, key, boxes):
    """x1 y1 x2 y2 of the face, from the group-photo detections or the single-face detection cache."""
    if key in boxes:
        b = boxes[key]
    else:
        d = pipe.det_cache.get(os.path.abspath(key))
        b = d["bbox"] if d else None
    return "" if b is None else " ".join(str(int(round(v))) for v in b)


def best_face_pairs(store, faces, pa, pb, weights):
    """Score every face in A against every face in B and keep the best combination per pair.
    For two single-person images this is exactly the ordinary pair score."""
    rows, ka, kb = [], [], []
    for r, (a, b) in enumerate(zip(pa, pb)):
        for x in faces[a]:
            for y in faces[b]:
                rows.append(r)
                ka.append(x)
                kb.append(y)
    s, _, q = store.subset_scores(ka, kb, weights)
    best = pd.Series(s).groupby(np.array(rows)).idxmax().to_numpy()
    return s[best], q[best], [ka[i] for i in best], [kb[i] for i in best]


def cmd_pairs(args):
    df = pd.read_csv(args.csv)
    lower = {c.lower(): c for c in df.columns}
    ca, cb = next(((lower[a], lower[b]) for a, b in PAIR_COLS if a in lower and b in lower), (df.columns[0], df.columns[1]))
    pa = [resolve(p, args.root) for p in df[ca]]
    pb = [resolve(p, args.root) for p in df[cb]]
    missing = [p for p in set(pa + pb) if not os.path.exists(p)]
    if missing:
        raise SystemExit(f"{len(missing)} images not found, e.g. {missing[:3]} - check --root")
    pipe, calib = load(args)
    faces, _ = faces_for(pipe, pa + pb, args)
    store = pipe.build_store([k for ks in faces.values() for k in ks])
    s, q, fa, fb = best_face_pairs(store, faces, pa, pb, calib.weights or None)
    thr = calib.thresholds_for(q, len(s))
    df["score"] = np.round(s, 6)
    df["prediction"] = (s > thr).astype(int)
    n_group = sum(len(faces[a]) > 1 or len(faces[b]) > 1 for a, b in zip(pa, pb))
    if n_group:
        df["faces_a"], df["faces_b"] = [len(faces[a]) for a in pa], [len(faces[b]) for b in pb]
        df["matched_face_a"], df["matched_face_b"] = [face_index(k) for k in fa], [face_index(k) for k in fb]
        print(f"{n_group} pairs involve group photos (scored by their best-matching face)")
    df.to_csv(args.out, index=False)
    print(f"wrote {args.out}: {len(df)} pairs, {int(df['prediction'].sum())} predicted same-person")
    lab = next((lower[c] for c in LABEL_COLS if c in lower), None)
    if lab:
        y = df[lab].astype(str).str.strip().str.lower().map(
            {"1": 1, "1.0": 1, "true": 1, "same": 1, "yes": 1, "0": 0, "0.0": 0, "false": 0, "different": 0, "no": 0}
        ).to_numpy()
        rep = full_report(s, y, thr)
        print({k: round(v, 4) for k, v in rep.items() if isinstance(v, float)})


def cmd_identify(args):
    pipe, calib = load(args)
    gal = list_images(args.gallery)
    gal_ids = [Path(p).parent.name for p in gal]
    probes = list_images(args.probes)
    faces, boxes = faces_for(pipe, probes, args)
    keys = [k for p in probes for k in faces[p]]
    store = pipe.build_store(gal + keys)
    w = calib.weights or None

    ids, T = build_templates(fused_embeddings(store, gal, w), gal_ids, store.quality[[store.index[p] for p in gal]])
    preds, scores = identify(fused_embeddings(store, keys, w), ids, T)
    # open-set rejection reuses the calibrated verification threshold (probe-vs-template cosine). Faces in group
    # photos always use it: bystanders who are not enrolled must come out as "unknown", not as their look-alike.
    group = np.array([split_face_key(k)[1] is not None for k in keys], dtype=bool)
    reject = (scores <= calib.global_threshold) & (group | args.open_set)
    preds = ["unknown" if r else p for p, r in zip(preds, reject)]
    images = [split_face_key(k)[0] for k in keys]
    out = pd.DataFrame({"probe": images, "predicted_identity": preds, "score": np.round(scores, 6)})
    if group.any():
        out.insert(1, "face", [face_index(k) for k in keys])
        out.insert(2, "bbox", [bbox_str(pipe, k, boxes) for k in keys])
    probe_parent = [Path(p).parent.name for p in images]
    if set(probe_parent) & set(ids):
        out["true_identity"] = probe_parent
        # per image: correct if the true person is among the faces identified in it
        hit = (out["true_identity"] == out["predicted_identity"]).groupby(out["probe"]).any()
        print(f"rank-1 identification accuracy: {hit.mean()*100:.2f}% of {len(hit)} probe images")
    out.to_csv(args.out, index=False)
    print(f"wrote {args.out}" + (f" ({int(group.sum())} faces from group photos)" if group.any() else ""))


def cmd_search(args):
    pipe, calib = load(args)
    query = [args.query] if os.path.isfile(args.query) else list_images(args.query)
    photos = [p for p in list_images(args.photos) if p not in set(query)]
    faces, boxes = faces_for(pipe, photos, args, force=True)
    keys = [k for p in photos for k in faces[p]]
    store = pipe.build_store(query + keys)
    w = calib.weights or None
    _, T = build_templates(fused_embeddings(store, query, w), ["query"] * len(query),
                           store.quality[[store.index[p] for p in query]])
    s = fused_embeddings(store, keys, w) @ T[0]
    thr = calib.global_threshold
    out = pd.DataFrame({"photo": [split_face_key(k)[0] for k in keys], "face": [face_index(k) for k in keys],
                        "bbox": [bbox_str(pipe, k, boxes) for k in keys], "score": np.round(s, 6),
                        "match": (s > thr).astype(int)}).sort_values("score", ascending=False)
    out.to_csv(args.out, index=False)
    found = out[out["match"] == 1]["photo"].nunique()
    print(f"person found in {found} of {len(photos)} photos ({len(keys)} faces searched, threshold {thr:.4f})")
    print(out.head(10).to_string(index=False))
    if args.draw:
        draw_matches(out, args.draw)
        print(f"annotated photos -> {args.draw}")
    print(f"wrote {args.out}")


def draw_matches(out: pd.DataFrame, out_dir: str):
    """Green box + score on the best-matching face of every photo where the person was found; grey on the rest."""
    import cv2

    os.makedirs(out_dir, exist_ok=True)
    for photo, d in out.groupby("photo"):
        if not d["match"].any():
            continue
        img = load_image(photo)
        lw, best = max(2, img.shape[1] // 400), d["score"].idxmax()
        for i, r in d.iterrows():
            if not r["bbox"]:
                continue
            x1, y1, x2, y2 = map(int, r["bbox"].split())
            color = (0, 200, 0) if i == best else (160, 160, 160)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, lw)
            if i == best:
                cv2.putText(img, f"{r['score']:.2f}", (x1, max(15, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX,
                            max(0.5, img.shape[1] / 1500), color, lw)
        cv2.imwrite(str(Path(out_dir) / f"{Path(photo).stem}_match.jpg"), img)


def cmd_one(args):
    pipe, calib = load(args)
    faces, _ = faces_for(pipe, [args.img1, args.img2], args)
    store = pipe.build_store([k for ks in faces.values() for k in ks])
    s, q, fa, fb = best_face_pairs(store, faces, [args.img1], [args.img2], calib.weights or None)
    t = calib.thresholds_for(q, 1)[0]
    print(f"score={s[0]:.4f} threshold={t:.4f} -> {'SAME person' if s[0] > t else 'DIFFERENT people'}")
    _, per, _ = store.subset_scores(fa, fb, calib.weights or None)
    print({m: round(float(v[0]), 4) for m, v in per.items()}, "pair quality", round(float(q[0]), 3))
    if len(faces[args.img1]) > 1 or len(faces[args.img2]) > 1:
        print(f"faces: {len(faces[args.img1])} vs {len(faces[args.img2])}; best match = face "
              f"{face_index(fa[0]) or 0} vs face {face_index(fb[0]) or 0} (left to right)")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/pipeline.json")
    ap.add_argument("--calibration", default="results/calibration.json")
    ap.add_argument("--cache", default="cache/predict")
    ap.add_argument("--ctx", type=int, default=0)
    ap.add_argument("--multi-face", choices=["auto", "off"], default="auto",
                    help="auto: images with 2+ faces are split into one entry per face (group photos)")
    ap.add_argument("--min-face-rel", type=float, default=0.25,
                    help="ignore faces smaller than this fraction of the image's biggest face (bystanders, posters)")
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("pairs")
    p.add_argument("--csv", required=True)
    p.add_argument("--root", default="")
    p.add_argument("--out", default="submission.csv")
    i = sub.add_parser("identify")
    i.add_argument("--gallery", required=True)
    i.add_argument("--probes", required=True)
    i.add_argument("--open-set", action="store_true")
    i.add_argument("--out", default="identification.csv")
    s = sub.add_parser("search")
    s.add_argument("--query", required=True, help="image of the person, or a folder of their images")
    s.add_argument("--photos", required=True, help="folder of photos to search (group, classroom, CCTV frames)")
    s.add_argument("--out", default="matches.csv")
    s.add_argument("--draw", default="", help="folder for annotated copies of the photos where the person was found")
    o = sub.add_parser("one")
    o.add_argument("--img1", required=True)
    o.add_argument("--img2", required=True)
    args = ap.parse_args()
    {"pairs": cmd_pairs, "identify": cmd_identify, "search": cmd_search, "one": cmd_one}[args.cmd](args)


if __name__ == "__main__":
    main()

## `webapp/` — Nemo's Deep Dive demo app (Gradio)

### `webapp/__init__.py`
Gradio demo app (Nemo's Deep Dive theme) on top of the PakFace pipeline.

In [ ]:
%%writefile webapp/__init__.py
"""Gradio demo app (Nemo's Deep Dive theme) on top of the PakFace pipeline."""

### `webapp/engine.py`
Face analysis for the demo app: the same pipeline as predict.py, run on in-memory images.

In [ ]:
%%writefile webapp/engine.py
"""Face analysis for the demo app: the same pipeline as predict.py, run on in-memory images.

Webcam captures and uploads never touch the feature / detection caches on disk. Each image goes through
detection (group photos are split into faces, single-person photos keep the retry cascade) -> alignment ->
quality -> enhancement fusion -> AdaFace + ArcFace embeddings, and pairs are scored with the same weighted
cosine fusion and calibrated threshold that the benchmark predictions use.
"""
from __future__ import annotations

import os
import tempfile
import threading
from dataclasses import dataclass

import cv2
import numpy as np

from pakface.config import PipelineConfig, default_config
from pakface.matching import Calibration
from pakface.pipeline import FacePipeline

MODEL_NAMES = {"adaface_ir101": "AdaFace IR-101", "glint_r100": "ArcFace R100 (Glint360K)"}
DEFAULT_THRESHOLD = 0.30  # same fallback as scripts/predict.py when results/calibration.json is missing


class NoFaceError(Exception):
    pass


@dataclass
class Face:
    bbox: np.ndarray       # x1, y1, x2, y2 in the input image
    crop: np.ndarray       # aligned 112x112 BGR, what the backbones actually see
    quality: float
    flags: list
    emb: dict              # model name -> unit embedding


@dataclass
class Match:
    score: float
    threshold: float
    per_model: dict
    faces_a: list
    faces_b: list
    ia: int                # index of the best-matching face in faces_a
    ib: int

    @property
    def same(self) -> bool:
        return self.score > self.threshold


@dataclass
class Sweep:
    query: Face
    crowd: list
    scores: np.ndarray
    thresholds: np.ndarray

    @property
    def matches(self) -> np.ndarray:
        return self.scores > self.thresholds


class FaceEngine:
    def __init__(self, config: str = "configs/pipeline.json", calibration: str = "results/calibration.json",
                 ctx: int = 0):
        cfg = PipelineConfig.from_json(config) if os.path.exists(config) else default_config()
        cfg.ctx_id = ctx
        cfg.cache_dir = os.path.join(tempfile.gettempdir(), "pakface_app")  # never written: everything is in memory
        self.calibrated = os.path.exists(calibration)
        self.calib = (Calibration.load(calibration) if self.calibrated
                      else Calibration(global_threshold=DEFAULT_THRESHOLD, use_bins=False))
        self.pipe = FacePipeline(cfg)
        self.model_names = [e.name for e in self.pipe.embedders]
        self.weights = self.calib.weights or {m: 1.0 for m in self.model_names}
        self.lock = threading.Lock()  # one request at a time on the shared models

    def faces(self, rgb: np.ndarray, multi: bool = True, min_rel: float = 0.25) -> list[Face]:
        """All faces in a group photo (left to right), or the single primary face otherwise."""
        img = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        with self.lock:
            dets = self.pipe.detector.detect_all(img, min_rel) if multi else []
            if len(dets) < 2:
                dets = [self.pipe.detector.detect(img)]
                if dets[0].stage == "no_detection":
                    return []
            parts = [self.pipe.crop_and_assess(img, d) for d in dets]
            embs, _ = self.pipe.embed_crops([c for c, _, _ in parts], [f for _, _, f in parts])
        return [Face(d.bbox, c, q.score, f, {m: e[i] for m, e in embs.items()})
                for i, (d, (c, q, f)) in enumerate(zip(dets, parts))]

    def score(self, a: Face, b: Face) -> tuple[float, dict]:
        per = {m: float(a.emb[m] @ b.emb[m]) for m in self.model_names}
        tot = sum(self.weights.get(m, 0.0) for m in per) or 1.0
        return sum(self.weights.get(m, 0.0) * s for m, s in per.items()) / tot, per

    def threshold(self, a: Face, b: Face) -> float:
        return float(self.calib.thresholds_for(np.array([min(a.quality, b.quality)]), 1)[0])

    def compare(self, rgb_a: np.ndarray, rgb_b: np.ndarray) -> Match:
        """Same person or not; with group photos, the best-matching face combination decides."""
        fa, fb = self.faces(rgb_a), self.faces(rgb_b)
        for faces, name in ((fa, "Face 1"), (fb, "Face 2")):
            if not faces:
                raise NoFaceError(name)
        best = max(((i, j) for i in range(len(fa)) for j in range(len(fb))),
                   key=lambda ij: self.score(fa[ij[0]], fb[ij[1]])[0])
        a, b = fa[best[0]], fb[best[1]]
        s, per = self.score(a, b)
        return Match(s, self.threshold(a, b), per, fa, fb, *best)

    def sweep(self, query_rgb: np.ndarray, crowd_rgb: np.ndarray) -> Sweep:
        """Find the query person among every face of a group photo."""
        q = self.faces(query_rgb, multi=False)
        if not q:
            raise NoFaceError("the face to find")
        crowd = self.faces(crowd_rgb)
        if not crowd:
            raise NoFaceError("the group photo")
        scores = np.array([self.score(q[0], f)[0] for f in crowd])
        thr = np.array([self.threshold(q[0], f) for f in crowd])
        return Sweep(q[0], crowd, scores, thr)

### `webapp/ui.py`
Gradio UI in the Nemo's Deep Dive theme: Face Match, Sea of Faces and the session's Dive Log.

In [ ]:
%%writefile webapp/ui.py
"""Gradio UI in the Nemo's Deep Dive theme: Face Match, Sea of Faces and the session's Dive Log.

Face Match:   Face 1 and Face 2, each captured live or uploaded -> same person or not, with the evidence
              (aligned crops, similarity vs threshold, per-backbone scores, quality flags, group-photo boxes).
Sea of Faces: one person's face + a group photo -> that person is boxed in the crowd.
Dive Log:     every result of this browser session, kept in memory only.
"""
from __future__ import annotations

import base64
import time
from pathlib import Path

import cv2
import gradio as gr
import numpy as np

from .engine import MODEL_NAMES, FaceEngine, Match, NoFaceError, Sweep

HERE = Path(__file__).parent
CSS = (HERE / "style.css").read_text(encoding="utf-8")
HEAD = ('<link rel="preconnect" href="https://fonts.googleapis.com">'
        '<link href="https://fonts.googleapis.com/css2?family=Fredoka:wght@500;600;700&display=swap" rel="stylesheet">')

MATCH_BUTTON = "🐠 Every Fish Has a Face?"
SWEEP_BUTTON = "📡 Sonar Sweep the Crowd"
FLAG_TEXT = {"blurry": "blurry", "dark": "too dark", "overexposed": "overexposed", "low_contrast": "low contrast",
             "low_res": "small / low-res face", "profile": "turned sideways", "weak_detection": "faint detection"}
GREEN, CORAL, GREY = (61, 220, 151), (255, 93, 115), (170, 190, 205)
METER_LO, METER_HI = -0.2, 1.0
LOG_MAX = 40


def theme() -> gr.themes.Base:
    """Same deep-sea palette whether the browser is in light or dark mode."""
    vals = dict(
        body_background_fill="#041a30", body_text_color="#e8f7ff", body_text_color_subdued="#93b9cf",
        background_fill_primary="rgba(6, 34, 58, 0.66)", background_fill_secondary="rgba(4, 26, 48, 0.55)",
        block_background_fill="rgba(6, 34, 58, 0.66)", block_border_color="rgba(94, 228, 255, 0.18)",
        block_label_background_fill="rgba(4, 26, 48, 0.85)", block_label_text_color="#bfe9f7",
        block_title_text_color="#e8f7ff", border_color_primary="rgba(94, 228, 255, 0.18)",
        border_color_accent="#5ee4ff", color_accent_soft="rgba(94, 228, 255, 0.15)",
        input_background_fill="rgba(2, 16, 30, 0.6)", input_border_color="rgba(94, 228, 255, 0.2)",
        button_primary_background_fill="#ff7a3d", button_primary_background_fill_hover="#ff9f5a",
        button_primary_text_color="#ffffff", button_primary_border_color="#ff7a3d",
        button_secondary_background_fill="rgba(94, 228, 255, 0.08)",
        button_secondary_background_fill_hover="rgba(94, 228, 255, 0.16)",
        button_secondary_text_color="#e8f7ff", button_secondary_border_color="rgba(94, 228, 255, 0.3)",
        panel_background_fill="rgba(6, 34, 58, 0.5)", link_text_color="#5ee4ff", loader_color="#5ee4ff",
        table_even_background_fill="rgba(255,255,255,0.03)", table_odd_background_fill="rgba(255,255,255,0.06)",
        table_border_color="rgba(94, 228, 255, 0.18)",
    )
    t = gr.themes.Base(primary_hue="orange", secondary_hue="cyan", neutral_hue="slate", radius_size="lg",
                       font=[gr.themes.GoogleFont("Plus Jakarta Sans"), "ui-sans-serif", "system-ui", "sans-serif"])
    return t.set(**vals, **{k + "_dark": v for k, v in vals.items()}, color_accent="#5ee4ff",
                 block_radius="20px", button_large_radius="999px")


# ---------------------------------------------------------------------------- drawing helpers
def b64(rgb: np.ndarray, max_side: int | None = None) -> str:
    if max_side and max(rgb.shape[:2]) > max_side:
        s = max_side / max(rgb.shape[:2])
        rgb = cv2.resize(rgb, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    ok, buf = cv2.imencode(".jpg", cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 88])
    return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode()


def crop_rgb(face) -> np.ndarray:
    return cv2.cvtColor(face.crop, cv2.COLOR_BGR2RGB)


def draw_faces(rgb: np.ndarray, faces: list, colors: dict, labels: dict) -> np.ndarray:
    """Numbered box on every face; faces in `colors` are highlighted, the rest are grey."""
    out = rgb.copy()
    lw = max(2, round(max(out.shape[:2]) / 320))
    fs = max(0.6, max(out.shape[:2]) / 750)
    for i, f in enumerate(faces):
        x1, y1, x2, y2 = (int(round(v)) for v in f.bbox[:4])
        c = colors.get(i, GREY)
        cv2.rectangle(out, (x1, y1), (x2, y2), c, lw * (2 if i in colors else 1))
        text = labels.get(i, f"#{i + 1}")
        (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, fs, max(1, lw - 1))
        ty = max(th + base + 4, y1)
        cv2.rectangle(out, (x1, ty - th - base - 6), (x1 + tw + 10, ty), c, -1)
        cv2.putText(out, text, (x1 + 5, ty - base - 2), cv2.FONT_HERSHEY_SIMPLEX, fs, (4, 26, 48), max(1, lw - 1),
                    cv2.LINE_AA)
    return out


def meter_pos(x: float) -> float:
    return 100 * min(max((x - METER_LO) / (METER_HI - METER_LO), 0.0), 1.0)


def meter(score: float, thr: float, good: bool) -> str:
    return (f'<div class="dd-meter"><div class="dd-meter-track">'
            f'<div class="dd-meter-fill {"dd-good" if good else "dd-bad"}" style="width:{meter_pos(score):.1f}%"></div>'
            f'<div class="dd-meter-thr" style="left:{meter_pos(thr):.1f}%"><span>threshold {thr:.2f}</span></div></div>'
            f'<div class="dd-meter-scale"><span>different people</span><span>same person</span></div></div>')


def quality_html(name: str, face) -> str:
    chips = "".join(f'<span class="dd-chip">{FLAG_TEXT.get(f, f.replace("_", " "))}</span>' for f in face.flags) \
        or '<span class="dd-chip dd-ok">clear face</span>'
    return f'<div class="dd-row"><span>{name}</span><span>{chips}</span><span class="dd-num">{face.quality:.0%}</span></div>'


# ---------------------------------------------------------------------------- cards
def card_idle(text: str) -> str:
    return (f'<div class="dd-card dd-idle"><div class="dd-sonar"><span></span><span></span><span></span>'
            f'<i class="dd-sweep"></i><b>🐠</b></div><p class="dd-muted">{text}</p></div>')


def card_scanning(title: str, text: str) -> str:
    return (f'<div class="dd-card dd-scan"><div class="dd-sonar"><span></span><span></span><span></span>'
            f'<i class="dd-sweep"></i><b>🐟</b></div><p class="dd-title dd-dots">{title}</p>'
            f'<p class="dd-muted">{text}</p></div>')


def card_warn(title: str, text: str) -> str:
    return f'<div class="dd-card dd-warn"><div class="dd-verdict">🫧 {title}</div><p class="dd-quip">{text}</p></div>'


def calib_note(engine: FaceEngine) -> str:
    if engine.calibrated:
        return '<p class="dd-note">Threshold calibrated on our validation divers (results/calibration.json).</p>'
    return ('<p class="dd-note">Using the default threshold of 0.30: copy <code>results/calibration.json</code> from '
            'the Colab run into <code>results/</code> to use the calibrated one.</p>')


def card_match(engine: FaceEngine, m: Match, rgb_a: np.ndarray, rgb_b: np.ndarray) -> str:
    a, b = m.faces_a[m.ia], m.faces_b[m.ib]
    margin = m.score - m.threshold
    cls = "dd-same" if m.same else "dd-diff"
    verdict = "✅ SAME PERSON" if m.same else "❌ DIFFERENT PEOPLE"
    quip = ("Same fish! This face was never forgotten." if m.same
            else "Two different fish in the sea.")
    call = ('<span class="dd-pill">clear call</span>' if abs(margin) >= 0.10
            else '<span class="dd-pill dd-close">close call: near the threshold</span>')

    per = "".join(
        f'<div class="dd-row"><span>{MODEL_NAMES.get(k, k)}</span>'
        f'<div class="dd-bar"><div style="width:{meter_pos(v):.1f}%"></div></div><span class="dd-num">{v:.3f}</span></div>'
        for k, v in m.per_model.items())

    groups = ""
    for name, rgb, faces, idx in (("Face 1", rgb_a, m.faces_a, m.ia), ("Face 2", rgb_b, m.faces_b, m.ib)):
        if len(faces) > 1:
            img = draw_faces(rgb, faces, {idx: GREEN if m.same else CORAL}, {})
            groups += (f'<figure><img src="{b64(img, 640)}"><figcaption>{name}: group photo, {len(faces)} faces, '
                       f'#{idx + 1} matched best</figcaption></figure>')
    if groups:
        groups = f'<h4>Group photos</h4><div class="dd-group">{groups}</div>'

    return f"""<div class="dd-card {cls}">
  <div class="dd-verdict">{verdict}</div>
  <p class="dd-quip">{quip}</p>{call}
  <div class="dd-pair">
    <figure><img src="{b64(crop_rgb(a))}"><figcaption>Face 1 (aligned)</figcaption></figure>
    <div class="dd-link">{"=" if m.same else "≠"}</div>
    <figure><img src="{b64(crop_rgb(b))}"><figcaption>Face 2 (aligned)</figcaption></figure>
  </div>
  {meter(m.score, m.threshold, m.same)}
  <div class="dd-stats">
    <div class="dd-stat"><b>{m.score:.3f}</b><span>similarity</span></div>
    <div class="dd-stat"><b>{m.threshold:.3f}</b><span>threshold</span></div>
    <div class="dd-stat"><b>{margin:+.3f}</b><span>margin</span></div>
  </div>
  <div class="dd-details">
    <h4>Each backbone's cosine similarity</h4>{per}
    <h4>Image quality</h4>{quality_html("Face 1", a)}{quality_html("Face 2", b)}
    {groups}
  </div>
  {calib_note(engine)}
</div>"""


def card_sweep(engine: FaceEngine, sw: Sweep) -> str:
    hits = np.flatnonzero(sw.matches)
    best = int(np.argmax(sw.scores))
    n = len(sw.crowd)
    if len(hits):
        cls, verdict = "dd-same", "✅ FOUND IN THE CROWD"
        quip = (f"Spotted! Face #{best + 1} is the one you're looking for." if len(hits) == 1
                else f"{len(hits)} faces pass the threshold; face #{best + 1} is the closest match.")
    else:
        cls, verdict = "dd-diff", "❌ NOT IN THIS SEA"
        quip = f"The closest face, #{best + 1}, scored {sw.scores[best]:.3f}: below the threshold."
    order = np.argsort(-sw.scores)[:12]
    rows = "".join(
        f'<div class="dd-item"><img src="{b64(crop_rgb(sw.crowd[i]))}"><b>#{i + 1}</b>'
        f'<div class="dd-bar"><div style="width:{meter_pos(sw.scores[i]):.1f}%"></div></div>'
        f'<span class="dd-num">{sw.scores[i]:.3f}</span>'
        f'{"<span class=dd-tag-yes>match</span>" if sw.matches[i] else "<span class=dd-tag-no>other fish</span>"}</div>'
        for i in order)
    more = f'<p class="dd-muted">…and {n - len(order)} more faces.</p>' if n > len(order) else ""
    return f"""<div class="dd-card {cls}">
  <div class="dd-verdict">{verdict}</div>
  <p class="dd-quip">{quip}</p>
  <p class="dd-muted">{n} face{"s" if n != 1 else ""} swept</p>
  <div class="dd-pair"><figure><img src="{b64(crop_rgb(sw.query))}"><figcaption>The face to find</figcaption></figure></div>
  {meter(float(sw.scores[best]), float(sw.thresholds[best]), bool(len(hits)))}
  <div class="dd-list">{rows}</div>{more}
  {calib_note(engine)}
</div>"""


# ---------------------------------------------------------------------------- dive log
def log_thumb(left: np.ndarray, right: np.ndarray, ok: bool) -> np.ndarray:
    """Two faces side by side on a green (match) or coral (no match) strip."""
    size, pad = 112, 6
    tiles = [cv2.resize(x, (size, size), interpolation=cv2.INTER_AREA) for x in (left, right)]
    canvas = np.zeros((size + 2 * pad, 2 * size + 3 * pad, 3), np.uint8)
    canvas[:] = GREEN if ok else CORAL
    canvas[pad:pad + size, pad:pad + size] = tiles[0]
    canvas[pad:pad + size, 2 * pad + size:2 * pad + 2 * size] = tiles[1]
    return canvas


def log_view(log: list):
    same = sum(e["kind"] == "match" and e["ok"] for e in log)
    diff = sum(e["kind"] == "match" and not e["ok"] for e in log)
    found = sum(e["kind"] == "sweep" and e["ok"] for e in log)
    sweeps = sum(e["kind"] == "sweep" for e in log)
    stats = (f'<div class="dd-logstats"><div><b>{len(log)}</b><span>dives</span></div>'
             f'<div><b>{same}</b><span>same person</span></div><div><b>{diff}</b><span>different</span></div>'
             f'<div><b>{found}/{sweeps}</b><span>found in crowd</span></div></div>')
    if not log:
        stats += '<p class="dd-intro">No dives yet. Results from Face Match and Sea of Faces appear here.</p>'
    return stats, [(e["img"], e["caption"]) for e in log]


def log_add(log: list, img: np.ndarray, caption: str, kind: str, ok: bool) -> list:
    return ([{"img": img, "caption": f"{time.strftime('%H:%M:%S')} · {caption}", "kind": kind, "ok": ok}] + log)[:LOG_MAX]


# ---------------------------------------------------------------------------- app
BUBBLES = '<div class="dd-ocean">' + "".join(
    f'<i style="left:{x}%;width:{s}px;height:{s}px;animation-duration:{d}s;animation-delay:-{dl}s"></i>'
    for x, s, d, dl in [(4, 10, 16, 2), (11, 18, 22, 9), (19, 8, 13, 5), (27, 14, 19, 12), (36, 6, 11, 3),
                        (44, 22, 26, 15), (53, 9, 15, 7), (61, 16, 21, 1), (69, 7, 12, 10), (77, 20, 24, 4),
                        (85, 11, 17, 13), (92, 15, 20, 6), (97, 8, 14, 11)]) + "</div>"

HERO = """<div class="dd-hero">
  <div class="dd-kicker">AI Odyssey · Finding the Unknown · Tech Fest '26</div>
  <h1>NEMO'S <span>DEEP</span> DIVE</h1>
  <div class="dd-sub">Every Fish Has a Face</div>
  <p class="dd-tag">Go deeper. Never forget a face.</p>
  <svg class="dd-wave" viewBox="0 0 1200 26" preserveAspectRatio="none" aria-hidden="true">
    <path d="M0 13 C150 0 250 26 400 13 S650 0 800 13 S1050 26 1200 13" fill="none" stroke="#5ee4ff" stroke-width="2"/>
  </svg>
</div>"""

FOOTER = """<div class="dd-footer">
  <b>PakFace</b> · SCRFD detection → 5-point alignment → AdaFace IR-101 + ArcFace R100 ensemble · cosine similarity<br>
  Photos are processed in memory for this session only; the server's upload cache is cleared every 10 minutes.<br>
  See you beneath the surface. 🫧
</div>"""


def build(engine: FaceEngine) -> gr.Blocks:
    def image_box(label: str, first: str = "webcam") -> gr.Image:
        sources = ["webcam", "upload"] if first == "webcam" else ["upload", "webcam"]
        return gr.Image(label=label, sources=sources, type="numpy", image_mode="RGB", height=380,
                        webcam_options=gr.WebcamOptions(mirror=True), elem_classes="dd-cam")

    def run_match(rgb_a, rgb_b, log):
        if rgb_a is None or rgb_b is None:
            missing = " and ".join(n for n, x in (("Face 1", rgb_a), ("Face 2", rgb_b)) if x is None)
            return card_warn("The tank is empty", f"{missing}: capture a live picture or upload a photo first."), log, *log_view(log)
        try:
            m = engine.compare(rgb_a, rgb_b)
        except NoFaceError as e:
            return card_warn(f"No face surfaced in {e}",
                             "Face the camera, add some light or move a little closer, then try again."), log, *log_view(log)
        thumb = log_thumb(crop_rgb(m.faces_a[m.ia]), crop_rgb(m.faces_b[m.ib]), m.same)
        log = log_add(log, thumb, f"{'✅ same' if m.same else '❌ different'} · {m.score:.3f}", "match", m.same)
        return card_match(engine, m, rgb_a, rgb_b), log, *log_view(log)

    def run_sweep(query, crowd, log):
        if query is None or crowd is None:
            missing = " and ".join(n for n, x in (("the face to find", query), ("the group photo", crowd)) if x is None)
            return None, card_warn("The sea is empty", f"Add {missing} first."), log, *log_view(log)
        try:
            sw = engine.sweep(query, crowd)
        except NoFaceError as e:
            return None, card_warn(f"No face surfaced in {e}",
                                   "Try a clearer, brighter photo where faces are not too small."), log, *log_view(log)
        colors = {int(i): GREEN for i in np.flatnonzero(sw.matches)}
        labels = {i: f"#{i + 1} {s:.2f}" for i, s in enumerate(sw.scores)}
        annotated = draw_faces(crowd, sw.crowd, colors, labels)
        found = bool(sw.matches.any())
        best = int(np.argmax(sw.scores))
        thumb = log_thumb(crop_rgb(sw.query), crop_rgb(sw.crowd[best]), found)
        caption = f"{'✅ found' if found else '❌ not found'} in crowd of {len(sw.crowd)} · {sw.scores[best]:.3f}"
        log = log_add(log, thumb, caption, "sweep", found)
        return annotated, card_sweep(engine, sw), log, *log_view(log)

    with gr.Blocks(title="Nemo's Deep Dive · Face Match", delete_cache=(600, 600)) as demo:
        gr.HTML(BUBBLES + HERO, padding=False)
        log = gr.State([])
        with gr.Tabs():
            with gr.Tab("🐠 Face Match"):
                gr.HTML('<p class="dd-intro">Capture a <b>live picture</b> or <b>upload a photo</b> for each side, '
                        'then ask the deep: is it the same face?</p>')
                with gr.Row(equal_height=True, elem_classes="dd-split"):
                    img_a = image_box("Face 1")
                    img_b = image_box("Face 2")
                with gr.Row(elem_classes=["dd-actions", "dd-split"]):
                    dive = gr.Button(MATCH_BUTTON, variant="primary", size="lg", elem_id="dive-btn", scale=4)
                    reset = gr.Button("↺ New dive", variant="secondary", size="lg", scale=1)
                match_out = gr.HTML(card_idle("Two faces in, one answer out. The result surfaces here."))

            with gr.Tab("🌊 Sea of Faces"):
                gr.HTML('<p class="dd-intro"><b>A sea of faces. One to find.</b> Give one person\'s face and a '
                        'group or classroom photo: the sonar boxes that person in the crowd.</p>')
                with gr.Row(equal_height=True, elem_classes="dd-split"):
                    query = image_box("The face to find")
                    crowd = image_box("The sea of faces (group photo)", first="upload")
                with gr.Row(elem_classes=["dd-actions", "dd-split"]):
                    sweep = gr.Button(SWEEP_BUTTON, variant="primary", size="lg", elem_id="sweep-btn", scale=4)
                    reset_sea = gr.Button("↺ New sweep", variant="secondary", size="lg", scale=1)
                sweep_img = gr.Image(label="Sonar result: green = match, grey = other faces", interactive=False,
                                     visible=True, height=460, elem_classes="dd-cam")
                sweep_out = gr.HTML(card_idle("Every fish has a face. The sweep result surfaces here."))

            with gr.Tab("📜 Dive Log"):
                stats0, _ = log_view([])
                log_stats = gr.HTML(stats0)
                gallery = gr.Gallery(show_label=False, columns=4, height="auto", object_fit="contain",
                                     allow_preview=True)
                clear_log = gr.Button("Clear the log", variant="secondary", size="sm")
        gr.HTML(FOOTER)

        dive.click(lambda: card_scanning("Sending sonar pings", "Detecting, aligning and comparing both faces"),
                   outputs=match_out, show_progress="hidden", queue=False
                   ).then(run_match, [img_a, img_b, log], [match_out, log, log_stats, gallery], show_progress="hidden")
        reset.click(lambda: (None, None, card_idle("Two faces in, one answer out. The result surfaces here.")),
                    outputs=[img_a, img_b, match_out], queue=False)
        sweep.click(lambda: card_scanning("Sweeping the sea of faces", "Every face in the photo is being compared"),
                    outputs=sweep_out, show_progress="hidden", queue=False
                    ).then(run_sweep, [query, crowd, log], [sweep_img, sweep_out, log, log_stats, gallery],
                           show_progress="hidden")
        reset_sea.click(lambda: (None, None, None, card_idle("Every fish has a face. The sweep result surfaces here.")),
                        outputs=[query, crowd, sweep_img, sweep_out], queue=False)
        clear_log.click(lambda: ([], *log_view([])), outputs=[log, log_stats, gallery], queue=False)
    return demo

### `webapp/style.css`

In [ ]:
%%writefile webapp/style.css
/* Nemo's Deep Dive: surface teal fading into the deep, clownfish-orange accents, sonar cyan highlights. */
:root {
  --dd-surface: #0e6a86;
  --dd-mid: #08385a;
  --dd-deep: #041a30;
  --dd-abyss: #010a15;
  --dd-ink: #e8f7ff;
  --dd-muted: #93b9cf;
  --dd-sonar: #5ee4ff;
  --dd-coral: #ff7a3d;
  --dd-coral-2: #ff9f5a;
  --dd-same: #3ddc97;
  --dd-diff: #ff5d73;
  --dd-panel: rgba(6, 34, 58, 0.66);
  --dd-line: rgba(94, 228, 255, 0.18);
  --dd-display: "Fredoka", "Plus Jakarta Sans", ui-rounded, system-ui, sans-serif;
}

body, gradio-app {
  background: linear-gradient(180deg, var(--dd-surface) 0%, var(--dd-mid) 28%, var(--dd-deep) 62%, var(--dd-abyss) 100%) fixed !important;
  color: var(--dd-ink);
}
.gradio-container { background: transparent !important; max-width: 1180px !important; margin: 0 auto !important; }
.gradio-container > .main, .gradio-container .wrap { position: relative; z-index: 1; }
footer { opacity: 0.55; }

/* ---------- rising bubbles + light rays ---------- */
.dd-ocean { position: fixed; inset: 0; pointer-events: none; z-index: 0; overflow: hidden; }
.dd-ocean::before {
  content: ""; position: absolute; inset: -20% -10% auto -10%; height: 70%;
  background: repeating-linear-gradient(100deg, rgba(255,255,255,0.05) 0 40px, transparent 40px 140px);
  filter: blur(18px); transform: skewX(-12deg); animation: dd-rays 14s ease-in-out infinite alternate;
}
.dd-ocean i {
  position: absolute; bottom: -40px; border-radius: 50%;
  border: 1.5px solid rgba(200, 245, 255, 0.7);
  background: radial-gradient(circle at 32% 30%, rgba(255,255,255,0.85) 0 12%, rgba(160,235,255,0.18) 40%, rgba(94,228,255,0.05) 70%);
  box-shadow: 0 0 10px rgba(94, 228, 255, 0.25);
  animation: dd-rise linear infinite;
}
@keyframes dd-rise {
  0% { transform: translate(0, 0); opacity: 0; }
  10% { opacity: 0.9; }
  50% { transform: translate(14px, -55vh); }
  100% { transform: translate(-10px, -112vh); opacity: 0; }
}
@keyframes dd-rays { from { transform: skewX(-12deg) translateX(-3%); } to { transform: skewX(-8deg) translateX(3%); } }

/* ---------- hero ---------- */
.dd-hero { text-align: center; padding: 34px 12px 10px; }
.dd-kicker { letter-spacing: 0.32em; font-size: 0.72rem; font-weight: 700; color: var(--dd-sonar); text-transform: uppercase; }
.dd-hero h1 {
  font-family: var(--dd-display); font-weight: 700; font-size: clamp(2.3rem, 6vw, 4.2rem); line-height: 1.02;
  margin: 10px 0 4px; color: #fff; letter-spacing: 0.01em;
  text-shadow: 0 4px 28px rgba(94, 228, 255, 0.35);
}
.dd-hero h1 span { color: var(--dd-coral); }
.dd-sub { font-family: var(--dd-display); font-size: clamp(1.1rem, 2.6vw, 1.5rem); font-weight: 500; color: var(--dd-ink); }
.dd-tag { color: var(--dd-muted); margin: 6px 0 0; font-size: 0.98rem; }
.dd-wave { display: block; width: 100%; height: 26px; margin-top: 18px; opacity: 0.55; }

.dd-intro { text-align: center; color: var(--dd-muted); margin: 4px auto 6px; max-width: 720px; font-size: 0.96rem; }
.dd-intro b { color: var(--dd-ink); }

/* ---------- tabs ---------- */
.tabs button[role="tab"] { font-family: var(--dd-display); font-size: 1.05rem !important; font-weight: 600; color: var(--dd-muted) !important; }
.tabs button[role="tab"][aria-selected="true"] { color: #fff !important; border-color: var(--dd-coral) !important; }

/* ---------- camera / upload panels ---------- */
.dd-cam { border: 1px solid var(--dd-line) !important; backdrop-filter: blur(6px); box-shadow: 0 10px 40px rgba(0, 0, 0, 0.35) !important; }
.dd-cam button[aria-label="Capture from camera"],
.dd-cam button[aria-label="Upload file"],
.dd-cam button[aria-label="Upload a file"] {
  width: auto !important; min-width: max-content; padding: 6px 14px !important; gap: 6px; border-radius: 999px !important;
  white-space: nowrap; height: auto !important;
  border: 1px solid var(--dd-line) !important; color: var(--dd-ink) !important; display: inline-flex; align-items: center;
}
.dd-cam button[aria-label="Capture from camera"]::after { content: "Live capture"; font-size: 0.82rem; font-weight: 600; }
.dd-cam button[aria-label="Upload file"]::after,
.dd-cam button[aria-label="Upload a file"]::after { content: "Upload photo"; font-size: 0.82rem; font-weight: 600; }
.dd-cam button[aria-label="Capture from camera"] svg,
.dd-cam button[aria-label="Upload file"] svg,
.dd-cam button[aria-label="Upload a file"] svg { width: 16px !important; height: 16px !important; flex: none; }
.dd-cam button[aria-label="Capture from camera"]:hover,
.dd-cam button[aria-label="Upload file"]:hover { border-color: var(--dd-sonar) !important; }

/* ---------- the big themed button ---------- */
#dive-btn, #sweep-btn {
  font-family: var(--dd-display) !important; font-size: 1.35rem !important; font-weight: 600 !important; letter-spacing: 0.01em;
  padding: 16px 22px !important; border: none !important; border-radius: 999px !important; color: #fff !important;
  background: linear-gradient(135deg, var(--dd-coral) 0%, var(--dd-coral-2) 50%, var(--dd-coral) 100%) !important;
  background-size: 200% 100% !important;
  box-shadow: 0 0 0 0 rgba(255, 122, 61, 0.55), 0 12px 32px rgba(255, 122, 61, 0.35) !important;
  animation: dd-pulse 2.6s ease-out infinite; transition: transform 0.15s ease, background-position 0.4s ease;
}
#dive-btn:hover, #sweep-btn:hover { transform: translateY(-2px); background-position: 100% 0 !important; }
@keyframes dd-pulse {
  0% { box-shadow: 0 0 0 0 rgba(255, 122, 61, 0.5), 0 12px 32px rgba(255, 122, 61, 0.35); }
  70% { box-shadow: 0 0 0 16px rgba(255, 122, 61, 0), 0 12px 32px rgba(255, 122, 61, 0.35); }
  100% { box-shadow: 0 0 0 0 rgba(255, 122, 61, 0), 0 12px 32px rgba(255, 122, 61, 0.35); }
}
.dd-actions { align-items: stretch; margin-top: 4px; }

/* ---------- result cards ---------- */
.dd-card {
  background: var(--dd-panel); border: 1px solid var(--dd-line); border-radius: 22px; padding: 26px 24px;
  backdrop-filter: blur(8px); box-shadow: 0 18px 50px rgba(0, 0, 0, 0.35); color: var(--dd-ink); text-align: center;
  animation: dd-surface 0.45s ease-out;
}
@keyframes dd-surface { from { opacity: 0; transform: translateY(14px); } to { opacity: 1; transform: none; } }
.dd-card p { margin: 6px 0; }
.dd-card.dd-same { border-color: rgba(61, 220, 151, 0.55); box-shadow: 0 0 40px rgba(61, 220, 151, 0.16), 0 18px 50px rgba(0,0,0,0.35); }
.dd-card.dd-diff { border-color: rgba(255, 93, 115, 0.55); box-shadow: 0 0 40px rgba(255, 93, 115, 0.14), 0 18px 50px rgba(0,0,0,0.35); }
.dd-card.dd-warn { border-color: rgba(255, 196, 87, 0.5); }
.dd-muted { color: var(--dd-muted); font-size: 0.92rem; }

.dd-verdict { font-family: var(--dd-display); font-weight: 700; font-size: clamp(1.7rem, 4.6vw, 2.6rem); letter-spacing: 0.02em; line-height: 1.1; }
.dd-same .dd-verdict { color: var(--dd-same); }
.dd-diff .dd-verdict { color: var(--dd-diff); }
.dd-warn .dd-verdict { color: #ffc457; font-size: 1.5rem; }
.dd-quip { font-size: 1.08rem; color: var(--dd-ink); }
.dd-pill { display: inline-block; margin-top: 8px; padding: 3px 12px; border-radius: 999px; font-size: 0.8rem; font-weight: 700;
  letter-spacing: 0.06em; text-transform: uppercase; background: rgba(94, 228, 255, 0.12); color: var(--dd-sonar); }
.dd-pill.dd-close { background: rgba(255, 196, 87, 0.14); color: #ffc457; }

.dd-pair { display: flex; align-items: center; justify-content: center; gap: 22px; margin: 22px 0 8px; flex-wrap: wrap; }
.dd-pair figure { margin: 0; }
.dd-pair img { width: 132px; height: 132px; border-radius: 18px; border: 2px solid var(--dd-line); object-fit: cover; display: block; }
.dd-pair figcaption { font-size: 0.8rem; color: var(--dd-muted); margin-top: 6px; }
.dd-link { width: 54px; height: 54px; border-radius: 50%; display: grid; place-items: center; font-size: 1.6rem; font-weight: 700; }
.dd-same .dd-link { background: rgba(61, 220, 151, 0.15); color: var(--dd-same); }
.dd-diff .dd-link { background: rgba(255, 93, 115, 0.15); color: var(--dd-diff); }

/* similarity meter */
.dd-meter { max-width: 560px; margin: 34px auto 6px; text-align: left; }
.dd-meter-track { position: relative; height: 14px; border-radius: 999px; background: rgba(255, 255, 255, 0.08); overflow: visible; }
.dd-meter-fill { height: 100%; border-radius: 999px; transition: width 0.8s ease; }
.dd-same .dd-meter-fill, .dd-meter-fill.dd-good { background: linear-gradient(90deg, #1b8f6a, var(--dd-same)); }
.dd-diff .dd-meter-fill, .dd-meter-fill.dd-bad { background: linear-gradient(90deg, #a3344a, var(--dd-diff)); }
.dd-meter-thr { position: absolute; top: -6px; bottom: -6px; width: 2px; background: #fff; box-shadow: 0 0 8px rgba(255,255,255,0.8); }
.dd-meter-thr span { position: absolute; top: -20px; left: 50%; transform: translateX(-50%); white-space: nowrap; font-size: 0.72rem; color: var(--dd-ink); }
.dd-meter-scale { display: flex; justify-content: space-between; font-size: 0.72rem; color: var(--dd-muted); margin-top: 6px; }

.dd-stats { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 10px; max-width: 560px; margin: 16px auto 4px; }
.dd-stat { background: rgba(255, 255, 255, 0.05); border-radius: 14px; padding: 10px 6px; }
.dd-stat b { display: block; font-family: var(--dd-display); font-size: 1.45rem; font-weight: 600; color: #fff; }
.dd-stat span { font-size: 0.74rem; color: var(--dd-muted); text-transform: uppercase; letter-spacing: 0.08em; }

.dd-details { max-width: 640px; margin: 18px auto 0; text-align: left; }
.dd-details h4 { font-family: var(--dd-display); font-weight: 600; font-size: 1rem; margin: 16px 0 8px; color: var(--dd-sonar); }
.dd-row { display: grid; grid-template-columns: 190px 1fr 56px; gap: 10px; align-items: center; font-size: 0.9rem; margin: 6px 0; }
.dd-bar { height: 8px; border-radius: 999px; background: rgba(255, 255, 255, 0.08); overflow: hidden; }
.dd-bar > div { height: 100%; border-radius: 999px; background: linear-gradient(90deg, #2a9bc0, var(--dd-sonar)); }
.dd-row .dd-num { text-align: right; font-variant-numeric: tabular-nums; }
.dd-chip { display: inline-block; padding: 2px 9px; margin: 2px 4px 2px 0; border-radius: 999px; font-size: 0.78rem;
  background: rgba(255, 196, 87, 0.13); color: #ffd58a; }
.dd-chip.dd-ok { background: rgba(61, 220, 151, 0.12); color: #8ff0c4; }
.dd-group { display: flex; gap: 14px; flex-wrap: wrap; justify-content: center; margin-top: 10px; }
.dd-group figure { margin: 0; text-align: center; }
.dd-group img { max-width: 290px; max-height: 220px; border-radius: 12px; border: 1px solid var(--dd-line); }
.dd-group figcaption { font-size: 0.8rem; color: var(--dd-muted); margin-top: 4px; }
.dd-note { margin-top: 16px; font-size: 0.8rem; color: var(--dd-muted); }

/* sea-of-faces list */
.dd-list { max-width: 620px; margin: 16px auto 0; text-align: left; }
.dd-item { display: grid; grid-template-columns: 52px 48px 1fr 56px 84px; gap: 10px; align-items: center; padding: 6px 8px; border-radius: 12px; font-size: 0.9rem; }
.dd-item:nth-child(odd) { background: rgba(255, 255, 255, 0.035); }
.dd-item img { width: 48px; height: 48px; border-radius: 10px; object-fit: cover; }
.dd-item .dd-num { text-align: right; font-variant-numeric: tabular-nums; }
.dd-tag-yes { color: var(--dd-same); font-weight: 700; font-size: 0.78rem; text-transform: uppercase; }
.dd-tag-no { color: var(--dd-muted); font-size: 0.78rem; }

/* ---------- sonar (idle + scanning) ---------- */
.dd-sonar { position: relative; width: 150px; height: 150px; margin: 6px auto 14px; border-radius: 50%;
  background: radial-gradient(circle, rgba(94, 228, 255, 0.10) 0%, rgba(94, 228, 255, 0.02) 60%, transparent 70%);
  border: 1px solid rgba(94, 228, 255, 0.25); overflow: hidden; }
.dd-sonar span { position: absolute; inset: 50%; width: 0; height: 0; border-radius: 50%; border: 2px solid var(--dd-sonar);
  transform: translate(-50%, -50%); animation: dd-ping 2.4s ease-out infinite; opacity: 0; }
.dd-sonar span:nth-child(2) { animation-delay: 0.8s; }
.dd-sonar span:nth-child(3) { animation-delay: 1.6s; }
.dd-sonar .dd-sweep { position: absolute; inset: 0; border-radius: 50%;
  background: conic-gradient(from 0deg, rgba(94, 228, 255, 0.45), rgba(94, 228, 255, 0) 70deg, transparent 360deg);
  animation: dd-spin 2.2s linear infinite; }
.dd-sonar b { position: absolute; inset: 0; display: grid; place-items: center; font-size: 2.4rem; }
.dd-card.dd-idle { animation: none; }
.dd-idle .dd-muted { color: #b5d3e3; }
.dd-idle .dd-sonar span, .dd-idle .dd-sonar .dd-sweep { animation-duration: 4.8s; }
.dd-idle .dd-sonar .dd-sweep { opacity: 0.45; }
@keyframes dd-ping { 0% { width: 0; height: 0; opacity: 0.9; } 100% { width: 150px; height: 150px; opacity: 0; } }
@keyframes dd-spin { to { transform: rotate(360deg); } }
.dd-scan .dd-title { font-family: var(--dd-display); font-size: 1.35rem; font-weight: 600; color: var(--dd-sonar); }
.dd-dots::after { content: ""; animation: dd-dots 1.4s steps(4) infinite; }
@keyframes dd-dots { 0% { content: ""; } 25% { content: "."; } 50% { content: ".."; } 75% { content: "..."; } }

/* ---------- dive log ---------- */
.dd-logstats { display: flex; gap: 12px; justify-content: center; flex-wrap: wrap; margin: 6px 0 10px; }
.dd-logstats div { background: var(--dd-panel); border: 1px solid var(--dd-line); border-radius: 14px; padding: 8px 18px; text-align: center; }
.dd-logstats b { display: block; font-family: var(--dd-display); font-size: 1.5rem; color: #fff; }
.dd-logstats span { font-size: 0.74rem; color: var(--dd-muted); text-transform: uppercase; letter-spacing: 0.08em; }

.dd-footer { text-align: center; color: var(--dd-muted); font-size: 0.82rem; padding: 26px 12px 8px; }
.dd-footer b { color: var(--dd-ink); font-weight: 600; }

@media (max-width: 640px) {
  .dd-split { flex-wrap: wrap !important; }
  .dd-split > * { min-width: 100% !important; flex: 1 1 100% !important; }
  .dd-hero { padding-top: 22px; }
  .dd-hero h1 { font-size: 2.1rem; }
  .dd-kicker { letter-spacing: 0.14em; font-size: 0.62rem; }
  .dd-card { padding: 20px 14px; }
  .dd-pair img { width: 104px; height: 104px; }
  #dive-btn, #sweep-btn { font-size: 1.15rem !important; }
  .dd-row { grid-template-columns: 1fr 44px; }
  .dd-row .dd-bar { grid-column: 1 / -1; grid-row: 2; }
  .dd-item { grid-template-columns: 44px 40px 1fr 50px; }
  .dd-item .dd-bar { display: none; }
  .dd-stat b { font-size: 1.15rem; }
}
@media (prefers-reduced-motion: reduce) {
  .dd-ocean i, .dd-ocean::before, #dive-btn, #sweep-btn, .dd-sonar span, .dd-sonar .dd-sweep, .dd-card { animation: none !important; }
}

## `tests/` — offline end-to-end test (synthetic data, fake models)

### `tests/test_end_to_end.py`
Offline tests: run the real scripts end-to-end on a synthetic dataset with a fake detector/embedder.

In [ ]:
%%writefile tests/test_end_to_end.py
"""Offline tests: run the real scripts end-to-end on a synthetic dataset with a fake detector/embedder.

No model downloads or GPU needed. Validates metrics maths, splitting/leakage guards, caching,
calibration, fine-tuning loop, ablation and the predict CLI.

    python tests/test_end_to_end.py
"""
import importlib.util
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import cv2
import numpy as np

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from pakface import augment, detect, embedders  # noqa: E402
from pakface.align import ARCFACE_TEMPLATE, align_face  # noqa: E402
from pakface.detect import Detection  # noqa: E402
from pakface.matching import Calibration, best_accuracy_threshold, best_f1_threshold  # noqa: E402
from pakface.metrics import compute_eer, confusion_metrics, full_report  # noqa: E402


# ---------------------------------------------------------------- fakes
class FakeDetector:
    def __init__(self, *a, **k):
        pass

    def detect(self, img):
        h, w = img.shape[:2]
        s = w / 112.0
        return Detection(np.array([0, 0, w, h], np.float32), ARCFACE_TEMPLATE * s, 0.9, "plain")

    def detect_all(self, img, min_rel_size=0.25):
        """Synthetic group photo = N square faces side by side (width = N x height)."""
        h, w = img.shape[:2]
        if w < 2 * h or w % h:
            return [self.detect(img)]
        s = h / 112.0
        return [Detection(np.array([i * h, 0, (i + 1) * h, h], np.float32),
                          ARCFACE_TEMPLATE * s + np.array([i * h, 0], np.float32), 0.9, "multi") for i in range(w // h)]


class FakeEmbedder(embedders.BaseEmbedder):
    """Embedding = 8x8 downsampled grayscale, centred. Identities are 8x8 patterns, so this separates them."""

    def __init__(self, name):
        self.name = name

    def embed_batch(self, crops):
        f = np.stack([cv2.resize(cv2.cvtColor(c, cv2.COLOR_BGR2GRAY), (8, 8), interpolation=cv2.INTER_AREA)
                      .astype(np.float32).ravel() for c in crops])
        f -= f.mean(1, keepdims=True)
        n = np.linalg.norm(f, axis=1, keepdims=True) + 1e-9
        return f / n, n[:, 0]


def fake_build_embedders(cfg):
    return [FakeEmbedder(m.name) for m in cfg.models]


def fake_load_adaface_hf(path, repo_id="", token=None):
    import torch.nn as nn

    class Tiny(nn.Module):
        def __init__(self):
            super().__init__()
            self.body = nn.Sequential(nn.Conv2d(3, 8, 5, stride=4, padding=2), nn.BatchNorm2d(8), nn.ReLU(),
                                      nn.AdaptiveAvgPool2d(4), nn.Flatten(), nn.Linear(128, 32))

        def forward(self, x):
            return self.body(x)

    import torch

    torch.manual_seed(0)
    return Tiny()


def load_script(name):
    spec = importlib.util.spec_from_file_location(name.replace(".py", ""), ROOT / "scripts" / name)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def run(mod, argv):
    old = sys.argv
    sys.argv = ["x"] + argv
    try:
        mod.main()
    finally:
        sys.argv = old


# ---------------------------------------------------------------- unit tests
def test_metrics():
    rng = np.random.default_rng(0)
    y = np.r_[np.ones(5000), np.zeros(5000)]
    s = np.r_[rng.normal(1, 1, 5000), rng.normal(-1, 1, 5000)]
    eer = compute_eer(s, y)
    # two unit-variance Gaussians 2 apart -> EER = Phi(-1) = 0.1587
    assert abs(eer - 0.1587) < 0.01, eer
    t, acc = best_accuracy_threshold(s, y)
    brute = max(((s > tt).astype(int) == y).mean() for tt in np.linspace(-3, 3, 2001))
    assert acc >= brute - 1e-3 and abs(t) < 0.15, (t, acc, brute)
    r = confusion_metrics(s, y, t)
    assert r["tp"] + r["fn"] == 5000 and abs(r["accuracy"] - acc) < 1e-9
    rep = full_report(s, y, t)
    assert 0.9 < rep["auc"] < 0.95 and rep["far"] == r["far"]
    tf, f1 = best_f1_threshold(s, y)
    brute_f1 = max(confusion_metrics(s, y, tt)["f1"] for tt in np.linspace(-3, 3, 2001))
    assert f1 >= brute_f1 - 1e-3 and abs(confusion_metrics(s, y, tf)["f1"] - f1) < 1e-9, (tf, f1, brute_f1)
    print("ok metrics: EER %.4f  best-acc %.4f @ %.3f" % (eer, acc, t))


def test_calibration_bins():
    rng = np.random.default_rng(1)
    q = rng.uniform(0, 1, 6000)
    y = rng.integers(0, 2, 6000)
    # low-quality pairs have compressed scores -> they need a lower threshold
    s = np.where(y == 1, 0.6, 0.0) * (0.3 + 0.7 * q) + rng.normal(0, 0.08, 6000)
    c = Calibration(use_bins=True).fit(s, y, q, n_bins=3)
    assert c.bin_thresholds[0] < c.bin_thresholds[-1], c.bin_thresholds
    acc_bins = (c.decide(s, q) == y).mean()
    acc_glob = ((s > c.global_threshold).astype(int) == y).mean()
    assert acc_bins >= acc_glob, (acc_bins, acc_glob)
    print(f"ok quality-binned thresholds {np.round(c.bin_thresholds, 3)}: acc {acc_glob:.4f} -> {acc_bins:.4f}")


def test_align_and_augment():
    rng = np.random.default_rng(2)
    img = rng.integers(0, 255, (112, 112, 3), dtype=np.uint8)
    out = align_face(img, ARCFACE_TEMPLATE)
    assert np.abs(out.astype(int) - img.astype(int)).mean() < 1.0  # identity transform
    big = rng.integers(0, 255, (480, 360, 3), dtype=np.uint8)
    for f in augment.PHOTOMETRIC + augment.OCCLUSIONS:
        o = f(img, rng) if f is not augment.grayscale else f(img)
        assert o.shape == img.shape and o.dtype == np.uint8, f.__name__
    for name, f in augment.STRESS_TESTS.items():
        o = f(big)
        assert o.shape == big.shape and o.dtype == np.uint8, name
    for _ in range(50):
        o = augment.train_augment(img, rng)
        assert o.shape == img.shape
    print("ok alignment + all augmentations / stress tests")


FORM_HEADER = [
    "Timestamp", "Email address", "Full name ", "Age group ", "Gender ",
    "Front face, normal light: looking at the camera, no glasses or cap ",
    "Side pose: face turned left and right (about halfway, both eyes still visible) ",
    "Glasses or sunglasses ", "Head covering you normally wear: hijab, dupatta, chador, turban, cap or topi ",
    "Face mask ", "Difficult lighting: night or dim room (no flash), harsh sunlight, or a window behind you ",
    "Low-quality photos: taken from far away or from above (like CCTV), blurry, old phone, or a WhatsApp-forwarded photo ",
    "Different look: older photo (2+ years ago), or with a different beard, moustache or hairstyle than now ",
    "I am 18+ and agree my photos may be used for this competition as described above.",
]


def test_google_form_import():
    import pandas as pd

    tmp = Path(tempfile.mkdtemp())
    rng = np.random.default_rng(4)
    files = tmp / "Face Collection (File responses)"
    q = FORM_HEADER[5:13]

    def upload(qi, fname, uploader, size=(300, 400)):
        d = files / (q[qi].strip().replace(":", "_") + " (File responses)")  # ':' -> '_' like a Windows zip
        d.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(d / f"{fname} - {uploader}.jpg"), rng.integers(0, 255, (*size, 3), dtype=np.uint8))
        return f"https://drive.google.com/open?id={fname}"

    def row(t, email, name, gender, ups, consent=True):
        cells = ["" for _ in q]
        for qi, links in ups.items():
            cells[qi] = ", ".join(links)
        return [t, email, name, "18-24", gender, *cells, FORM_HEADER[-1] if consent else ""]

    rows = [
        row("10/18/2026 10:00:00", "ali@x.pk", "Ali Khan", "Male",
            {0: [upload(0, "a1", "Ali Khan"), upload(0, "a2", "Ali Khan")], 2: [upload(2, "a3", "Ali Khan", (3000, 2400))]}),
        row("10/18/2026 11:00:00", "sara@x.pk", "Sara Ahmed", "Female",
            {0: [upload(0, "s1", "Sara Ahmed")], 3: [upload(3, "s2", "Sara Ahmed")], 4: [upload(4, "s3", "Sara Ahmed")]}),
        row("10/18/2026 12:00:00", "bilal@x.pk", "Bilal", "Male", {0: [upload(0, "b1", "Bilal"), upload(0, "b2", "Bilal")]},
            consent=False),
        row("10/18/2026 13:00:00", "zara@x.pk", "Zara Batool", "Female",
            {0: [upload(0, "z1", "Zara B")], 5: [upload(5, "z2", "Zara B")]}),  # Google name differs from typed name
        row("10/18/2026 14:00:00", "ALI@x.pk", "Ali Khan", "Male", {6: [upload(6, "a4", "Ali Khan")]}),  # 2nd submission
    ]
    pd.DataFrame(rows, columns=FORM_HEADER).to_csv(tmp / "responses.csv", index=False)
    imp = load_script("import_google_form.py")
    base = ["--csv", str(tmp / "responses.csv"), "--files", str(files), "--raw", str(tmp / "raw"),
            "--meta-dir", str(tmp / "meta"), "--code-map", str(tmp / "private" / "map.csv")]
    run(imp, base)
    own = tmp / "raw" / "own_collection"
    assert sorted(d.name for d in own.iterdir()) == ["P0001", "P0002"], "Zara unmatched, Bilal has no consent"
    assert len(list((own / "P0001").iterdir())) == 4, "both of Ali's submissions merge into one person"

    pd.DataFrame({"uploader": ["Zara B"], "email": ["zara@x.pk"]}).to_csv(tmp / "names.csv", index=False)
    run(imp, base + ["--name-map", str(tmp / "names.csv")])
    assert sorted(d.name for d in own.iterdir()) == ["P0001", "P0002", "P0003"]
    ids = pd.read_csv(tmp / "meta" / "identities.csv")
    ims = pd.read_csv(tmp / "meta" / "images.csv")
    text = (tmp / "meta" / "identities.csv").read_text() + (tmp / "meta" / "images.csv").read_text()
    assert "Ali" not in text and "@" not in text, "no names or emails outside the private map"
    assert set(ids["identity"]) == {"P0001", "P0002", "P0003"} and set(ids["gender"]) == {"male", "female"}
    assert ims.set_index("filename").loc["P0001_glasses_1.jpg", "glasses"] == "glasses"
    assert ims.set_index("filename").loc["P0002_mask_1.jpg", "mask"] == "mask"
    from PIL import Image

    with Image.open(own / "P0001" / "P0001_glasses_1.jpg") as im:
        assert max(im.size) == 2048 and not im.getexif(), "resized, metadata stripped"
    assert "Ali Khan" in (tmp / "private" / "map.csv").read_text()

    # Sara withdraws (response deleted): her photos and rows disappear, everyone else keeps their code
    pd.DataFrame([r for r in rows if r[2] != "Sara Ahmed"], columns=FORM_HEADER).to_csv(tmp / "responses.csv", index=False)
    run(imp, base + ["--name-map", str(tmp / "names.csv")])
    assert sorted(d.name for d in own.iterdir()) == ["P0001", "P0003"]
    assert "P0002" not in pd.read_csv(tmp / "meta" / "identities.csv")["identity"].tolist()
    assert not pd.read_csv(tmp / "meta" / "images.csv")["filename"].str.startswith("P0002").any()
    shutil.rmtree(tmp, ignore_errors=True)
    print("ok google form import: consent filter, stable codes, name map, withdrawal, no names in metadata")


def make_dataset(root: Path, n_ids=40, per_id=8, sources=("own_collection", "public_a")):
    rng = np.random.default_rng(3)
    k = 0
    rows = []
    for si, src in enumerate(sources):
        for i in range(n_ids // len(sources)):
            pattern = rng.integers(0, 255, (8, 8)).astype(np.uint8)
            ident = f"P{k:04d}" if src == "own_collection" else f"Person Name {k}"
            gender = ["male", "female"][k % 2]
            rows.append({"identity": ident, "gender": gender, "region": ["sindh", "punjab", "kpk", "balochistan"][k % 4]})
            k += 1
            d = root / "raw" / src / ident
            d.mkdir(parents=True)
            for j in range(per_id):
                face = cv2.resize(pattern, (112, 112), interpolation=cv2.INTER_NEAREST)
                face = np.clip(face.astype(np.float32) + rng.normal(0, 25, face.shape), 0, 255).astype(np.uint8)
                if j % 4 == 3:
                    face = (face * 0.25).astype(np.uint8)  # dark images -> exercise enhancement
                cv2.imwrite(str(d / f"img_{j}.jpg"), cv2.cvtColor(face, cv2.COLOR_GRAY2BGR))
    import pandas as pd

    pd.DataFrame(rows).to_csv(root / "identities.csv", index=False)


def test_end_to_end():
    tmp = Path(tempfile.mkdtemp())
    cwd = os.getcwd()
    try:
        make_dataset(tmp)
        os.chdir(tmp)
        detect.FaceDetector = FakeDetector
        embedders.build_embedders = fake_build_embedders

        from pakface.config import default_config

        cfg = default_config()
        cfg.ctx_id = -1
        os.makedirs("configs")
        cfg.to_json("configs/pipeline.json")

        run(load_script("01_prepare_data.py"), ["--raw", "raw", "--out", ".", "--identities", "identities.csv",
                                                 "--images", "none.csv", "--ctx", "-1", "--dedupe-dist", "0"])
        run(load_script("02_split.py"), ["--manifest", "manifest.csv", "--out", ".", "--genuine-per-id", "10"])
        import pandas as pd

        sp = pd.read_csv("splits.csv")
        assert set(sp[sp["split"] != "train"]["source"]) == {"own_collection"}
        assert sp.groupby("identity")["split"].nunique().max() == 1
        own = sp[sp["source"] == "own_collection"].groupby("identity")["split"].first()
        assert (own == "train").any() and (own != "train").any(), "own data must feed both fine-tuning and eval"

        try:
            import torch  # noqa: F401
            have_torch = True
        except Exception:
            have_torch = False
            print("   (torch not installed: skipping fine-tuning step)")
        if have_torch:
            ft = load_script("03_finetune.py")
            ft.load_adaface_hf = fake_load_adaface_hf
            run(ft, ["--splits", "splits.csv", "--pairs-val", "pairs_val.csv", "--epochs", "2", "--batch", "16",
                     "--workers", "0", "--min-images", "2"])
            assert Path("checkpoints/finetune_history.json").exists()
            assert Path("checkpoints/adaface_ir101_finetuned.pt").exists(), "fine-tuned model must always be saved"

        run(load_script("04_evaluate.py"), ["--pairs-val", "pairs_val.csv", "--pairs-test", "pairs_test.csv",
                                             "--splits", "splits.csv", "--ctx", "-1"])
        m = json.loads(Path("results/full_pipeline/metrics.json").read_text())
        print(f"   synthetic test accuracy {m['accuracy']:.4f}, EER {m['eer']:.4f}")
        assert m["accuracy"] > 0.9
        for f in ["test_roc.png", "test_det.png", "test_hist.png", "per_group.csv", "summary.md"]:
            assert Path("results/full_pipeline", f).exists(), f
        assert "gender" in Path("results/full_pipeline/summary.md").read_text()
        idm = m["identification"]
        print(f"   synthetic rank-1 {idm['rank1']:.4f}, DIR {idm['dir']:.4f}, FPIR {idm['fpir']:.4f}")
        assert idm["rank1"] > 0.9 and idm["rank5"] >= idm["rank1"]

        run(load_script("05_ablation.py"), ["--pairs-val", "pairs_val.csv", "--pairs-test", "pairs_test.csv",
                                             "--ctx", "-1"])
        ab = pd.read_csv("results/ablation/ablation.csv")
        assert len(ab) >= 9 and Path("results/ablation/stress_tests.csv").exists()

        pt = pd.read_csv("pairs_test.csv")
        pd.DataFrame({"img1": pt["path_a"], "img2": pt["path_b"], "is_same": pt["label"]}).to_csv("bench.csv", index=False)
        pr = load_script("predict.py")
        run(pr, ["--ctx", "-1", "pairs", "--csv", "bench.csv", "--out", "submission.csv"])
        sub = pd.read_csv("submission.csv")
        assert {"score", "prediction"} <= set(sub.columns) and len(sub) == len(pt)
        # identification mode: gallery = first 4 imgs per own id, probes = rest
        gal, prb = Path("gallery"), Path("probes")
        for d in Path("raw/own_collection").iterdir():
            imgs = sorted(d.iterdir())
            for i, p in enumerate(imgs):
                dst = (gal if i < 4 else prb) / d.name / p.name
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(p, dst)
        run(pr, ["--ctx", "-1", "identify", "--gallery", "gallery", "--probes", "probes", "--out", "ids.csv"])
        ids = pd.read_csv("ids.csv")
        acc = (ids["true_identity"] == ids["predicted_identity"]).mean()
        assert acc > 0.9, acc
        run(pr, ["--ctx", "-1", "one", "--img1", str(pt["path_a"][0]), "--img2", str(pt["path_b"][0])])

        # group photos: 3 different people side by side
        grp = Path("groups")
        grp.mkdir()
        own_ids = sorted(d.name for d in prb.iterdir())
        for g in range(4):
            row = [cv2.imread(str(sorted((prb / m).iterdir())[0])) for m in own_ids[g * 3:(g + 1) * 3]]
            cv2.imwrite(str(grp / f"group_{g}.png"), np.hstack(row))
        target = own_ids[4]  # middle face of group_1
        run(pr, ["--ctx", "-1", "search", "--query", str(gal / target), "--photos", "groups", "--out", "matches.csv",
                 "--draw", "found"])
        mt = pd.read_csv("matches.csv")
        assert mt.iloc[0]["photo"].endswith("group_1.png") and mt.iloc[0]["face"] == 1, mt.head()
        assert mt[mt["match"] == 1]["photo"].nunique() == 1 and Path("found/group_1_match.jpg").exists()
        gp = Path("probes_groups") / target
        gp.mkdir(parents=True)
        shutil.copy(grp / "group_1.png", gp)
        run(pr, ["--ctx", "-1", "identify", "--gallery", "gallery", "--probes", "probes_groups", "--out", "ids_g.csv"])
        ig = pd.read_csv("ids_g.csv")
        assert len(ig) == 3 and list(ig["predicted_identity"]) == own_ids[3:6], ig
        print("ok group photos: search finds the person, identify labels every face")
        print("ok end-to-end: prepare -> split -> finetune -> evaluate -> ablation -> predict")
    finally:
        os.chdir(cwd)
        shutil.rmtree(tmp, ignore_errors=True)


if __name__ == "__main__":
    test_metrics()
    test_calibration_bins()
    test_align_and_augment()
    test_google_form_import()
    test_end_to_end()
    print("\nALL TESTS PASSED")

## `docs/` — consent form and data-collection protocol

### `docs/consent_form.md`

In [ ]:
%%writefile docs/consent_form.md
# Participant Consent Form — Face Images for Research

**Project:** PakFace – Face Recognition for the Pakistani Context
**Event:** Nemo's Deep Dive, AI Odyssey (TechFest '26), AI Alliance, Dept. of CS&IT, NED University of Engineering & Technology
**Team:** ______________________  **Contact:** ______________________

## What we are doing
We are building and testing a face recognition system for a university competition. We would like to take
about 15–25 photos of your face (with and without glasses, cap, dupatta/hijab/turban or mask if you normally
wear them, in different lighting) using a mobile phone.

## How your photos will be used
- Only to train and evaluate our face recognition model for this competition and its presentation.
- Photos are stored under a **code** (e.g. P0123), never your name. The sheet linking code ↔ name is kept
  separately by one team member and destroyed after the competition.
- Photos will **not** be posted online, shared outside the team and the competition judges, or used commercially.
- Aggregated results (e.g. "accuracy for participants wearing hijab") may appear in our report and slides.
  **No identifiable photo of you will appear in slides unless you tick the box below.**
- All photos and derived data (embeddings) will be deleted by **30 November 2026**.

## Your rights
- Participation is voluntary. You can skip any photo condition (e.g. you may decline photos without a hijab).
- You can withdraw at any time before **22 October 2026** by telling any team member your code; your
  photos will be deleted and excluded from results.

## Optional information (helps us check the system is fair across groups — leave blank if you prefer)
| Gender | Age group (18–24 / 25–34 / 35–49 / 50+) | Home region / province | Skin tone (light / medium / dark, self-described) |
|---|---|---|---|
| | | | |

## Consent
- [ ] I am 18 years or older.
- [ ] I agree to have my photos taken and used as described above.
- [ ] (Optional) My photo may appear in the presentation slides as an example.

Participant code: ________  Name: ______________________  Signature: ____________  Date: ________

Team member: ______________________  Signature: ____________

*Participants under 18 must not be photographed for this dataset.*

### `docs/data_collection_protocol.md`

In [ ]:
%%writefile docs/data_collection_protocol.md
# Data Collection Protocol (own Pakistani dataset)

Your own consented collection is the most valuable data you have: it is the only data that (a) is truly
Pakistani, (b) has a real consent trail, which the presentation is judged on, and (c) is guaranteed NOT to be
inside the pretrained models' web-scraped training sets. Val/test splits are drawn from it by default.

## Target
- **50–80 people** minimum (more matters more than more photos per person). Aim for diversity across:
  - region / ethnicity: Sindhi, Punjabi, Pashtun, Baloch, Muhajir/Urdu-speaking, Kashmiri, Gilgit-Baltistan, Hazara …
    (Karachi + NED makes this realistic — ask friends, family, hostel residents, staff)
  - gender (target ≥ 40 % women), age groups (include parents / staff for 35+ and 50+), skin tones
- **15–25 photos per person** following the shot list below (≈ 10 minutes per person with two phones).

## Shot list per person (tick on the sheet)
| # | Condition | Notes |
|---|---|---|
| 1–3 | Frontal, good indoor light | neutral, smile, slight head turn |
| 4–5 | Outdoor daylight | one in direct sun (hard shadows) |
| 6–7 | Low light | corridor / evening, phone flash OFF |
| 8 | Backlit | person in front of a window |
| 9–10 | Head covering they normally wear | dupatta, hijab, niqab-off, turban, topi/cap — **only if they wear it** |
| 11 | Glasses / sunglasses | own glasses if they have them |
| 12 | Mask | surgical / cloth |
| 13–14 | Profile-ish (30–45°) | left and right |
| 15 | Far away (3–5 m) | gives genuinely low-resolution faces |
| 16 | Cheap / old phone or front camera | device diversity |
| 17 | CCTV-like | from above, 3 m away, through a doorway, or a screenshot of a video call |
| 18+ | Different day / outfit | if the person is available on a second day: very valuable (session variation) |

Two different phones per person if possible. Do not use beauty filters / portrait mode.

## Storage
```
data/raw/own_collection/P0001/IMG_xxx.jpg
data/raw/own_collection/P0002/...
```
- Folder name = participant code. Never names.
- Fill `data/identities.csv` (identity,gender,age_group,skin_tone,region) — copy from the form's optional fields.
- Optionally fill `data/images.csv` (filename,headwear,glasses,mask,lighting,device) — it powers the per-group
  breakdown that the judges explicitly asked for ("how the pipeline performs across different groups").
  Use values like headwear: none/hijab/dupatta/turban/cap; glasses: none/glasses/sunglasses; mask: none/mask;
  lighting: normal/low/backlit/outdoor; device: phone_a/phone_b/cctv_like.

## Ethics checklist for the slides
- Signed consent for every participant, adults only, code-based storage, deletion date, withdrawal route.
- Public datasets used only under their research licences; celebrity images used for training only,
  never for reported test numbers (they may be in the pretrained models' training data).
- Fairness: report per-group FAR/FRR, not just overall accuracy.

## `scripts/` — runnable stages

### `scripts/check_submission.py`
Check a Face Embedding Challenge submission (.npz) the way the teacher grades it, before uploading.

In [ ]:
%%writefile scripts/check_submission.py
"""Check a Face Embedding Challenge submission (.npz) the way the teacher grades it, before uploading.

    python scripts/check_submission.py group05.npz --data <data_dir> [--key practice_key.csv]

Format rules from the student pack README: keys train_ids, train_emb [N,D], test_ids, test_emb [N,D]; same D;
no NaNs; ID arrays exactly as make_submission.py writes them (sorted os.listdir of train/ and test/).
Scoring: each test embedding is matched to the closest train embedding by cosine similarity; score = top-1
accuracy, top-5 breaks ties. Scores are only computed when the answer key is given (practice).
"""
import argparse
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

KEYS = ("train_ids", "train_emb", "test_ids", "test_emb")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("npz")
    ap.add_argument("--data", default="", help="folder with train/ and test/: checks the ID arrays against it")
    ap.add_argument("--key", default="", help="answer CSV with test_id,person: prints top-1 / top-5 accuracy")
    args = ap.parse_args()

    z = np.load(args.npz, allow_pickle=False)
    errors, warnings = [], []
    missing = [k for k in KEYS if k not in z.files]
    if missing:
        sys.exit(f"FAIL: missing arrays {missing} (has {z.files})")
    tr_ids, tr, te_ids, te = (z[k] for k in KEYS)
    print(f"train_emb {tr.shape} {tr.dtype} | test_emb {te.shape} {te.dtype} | extra arrays: "
          f"{[k for k in z.files if k not in KEYS] or 'none'}")

    if tr.ndim != 2 or te.ndim != 2:
        errors.append("embeddings must be 2-D [N, D]")
    elif tr.shape[1] != te.shape[1]:
        errors.append(f"train D={tr.shape[1]} != test D={te.shape[1]}")
    if len(tr_ids) != len(tr) or len(te_ids) != len(te):
        errors.append("number of IDs and embeddings differ")
    for name, e in (("train_emb", tr), ("test_emb", te)):
        if not np.issubdtype(e.dtype, np.floating):
            errors.append(f"{name} is {e.dtype}, not float")
        elif not np.isfinite(e).all():
            errors.append(f"{name} has {int((~np.isfinite(e)).sum())} NaN/inf values")
        else:
            n = np.linalg.norm(e, axis=1)
            if (n < 1e-6).any():
                errors.append(f"{name} has {int((n < 1e-6).sum())} all-zero vectors (cosine undefined)")
            elif not np.allclose(n, 1, atol=1e-3):
                warnings.append(f"{name} is not unit length (norms {n.min():.3f}-{n.max():.3f}): fine for cosine, "
                                "but a plain dot-product grader would rank differently")

    if args.data:
        for name, ids in (("train", tr_ids), ("test", te_ids)):
            listing = sorted(os.listdir(Path(args.data) / name))  # exactly what make_submission.py uses
            if list(ids) != listing:
                extra = sorted(set(listing) - set(ids))
                errors.append(f"{name}_ids differ from sorted(os.listdir({name}/))"
                              + (f"; not in the npz: {extra[:5]}" if extra else ""))

    for w in warnings:
        print("WARNING:", w)
    if errors:
        for e in errors:
            print("FAIL:", e)
        sys.exit(1)
    print("OK: format matches the submission rules")

    if args.key:
        key = pd.read_csv(args.key)
        truth = dict(zip(key["test_id"].astype(str), key["person"].astype(str)))
        known = set(truth.values())
        person = np.array([s if s in known else re.sub(r"[_\-\s]*\d+$", "", s)
                           for s in (Path(str(i)).stem for i in tr_ids)])
        trn = tr / np.linalg.norm(tr, axis=1, keepdims=True)
        ten = te / np.linalg.norm(te, axis=1, keepdims=True)
        order = np.argsort(-(ten @ trn.T), axis=1)
        y = np.array([truth[str(i)] for i in te_ids])
        top1 = np.mean(person[order[:, 0]] == y)
        top5 = np.mean([t in person[o[:5]] for t, o in zip(y, order)])
        print(f"top-1 accuracy: {top1 * 100:.2f}%   top-5 accuracy: {top5 * 100:.2f}%   ({len(y)} test images)")


if __name__ == "__main__":
    main()

### `scripts/eval_student_pack.py`
Evaluate the pipeline on the teacher's student pack (Face Embedding Challenge).

In [ ]:
%%writefile scripts/eval_student_pack.py
"""Evaluate the pipeline on the teacher's student pack (Face Embedding Challenge).

The pack has <data>/train (one photo per person, named by person) and <data>/test (anonymous IDs).
"Training" here is enrolment: every train image is embedded with the frozen pipeline and becomes the
gallery. Each test image is then assigned to its closest train image by cosine similarity, which is exactly
how the teacher scores the submission (top-1 accuracy, top-5 breaks ties).

    python scripts/eval_student_pack.py --data student_pack/student_pack/student_pack/practice \
        --key student_pack/student_pack/student_pack/practice_key.csv --group group05

--key (test_id,person) prints accuracy / precision / recall / F1; without it (lab day) only predictions
are written. --group also saves <group>.npz in the format make_submission.py produces, ready to upload.

Group photos (--multi-face auto, the default): a test image with 2+ faces is split into faces and the face that
best matches an enrolled person becomes that image's embedding (the submission needs one vector per image).
Train images are always treated as single-person enrolment photos.
"""
import argparse
import os
import re
import sys
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from sklearn.metrics import accuracy_score, precision_recall_fscore_support  # noqa: E402

from pakface.config import PipelineConfig, default_config  # noqa: E402
from pakface.data import IMG_EXT  # noqa: E402
from pakface.matching import Calibration, fused_embeddings  # noqa: E402
from pakface.pipeline import FacePipeline  # noqa: E402


def person_of(train_id, known):
    """Train images are named by person: 'maryam_nawaz.jpg', possibly 'maryam_nawaz_2.jpg' with several photos."""
    stem = Path(train_id).stem
    return stem if stem in known else re.sub(r"[_\-\s]*\d+$", "", stem)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="folder containing train/ and test/")
    ap.add_argument("--key", default="", help="answer CSV with test_id,person (practice_key.csv)")
    ap.add_argument("--group", default="", help="also save <group>.npz in the submission format")
    ap.add_argument("--out", default="student_pack_predictions.csv")
    ap.add_argument("--config", default="configs/pipeline.json")
    ap.add_argument("--calibration", default="results/calibration.json")
    ap.add_argument("--cache", default="cache/student_pack")
    ap.add_argument("--ctx", type=int, default=0, help="GPU id, -1 for CPU")
    ap.add_argument("--multi-face", choices=["auto", "off"], default="auto",
                    help="auto: test images with 2+ faces use the face that best matches an enrolled person")
    ap.add_argument("--min-face-rel", type=float, default=0.25,
                    help="ignore faces smaller than this fraction of the image's biggest face")
    args = ap.parse_args()

    # same file lists and order as make_submission.py, so the .npz IDs match what the teacher expects
    tr_dir, te_dir = Path(args.data) / "train", Path(args.data) / "test"
    tr_ids = sorted(f for f in os.listdir(tr_dir) if Path(f).suffix.lower() in IMG_EXT)
    te_ids = sorted(f for f in os.listdir(te_dir) if Path(f).suffix.lower() in IMG_EXT)
    tr_paths = [str((tr_dir / f).resolve()) for f in tr_ids]
    te_paths = [str((te_dir / f).resolve()) for f in te_ids]
    print(f"train: {len(tr_ids)} images | test: {len(te_ids)} images")

    cfg = PipelineConfig.from_json(args.config) if os.path.exists(args.config) else default_config()
    cfg.ctx_id = args.ctx
    cfg.cache_dir = args.cache
    calib = Calibration.load(args.calibration) if os.path.exists(args.calibration) else None
    weights = calib.weights if calib and calib.weights else None

    pipe = FacePipeline(cfg)
    if args.multi_face == "auto":
        faces, _ = pipe.expand_faces(te_paths, args.min_face_rel)
    else:
        faces = {p: [p] for p in te_paths}
    store = pipe.build_store(tr_paths + [k for p in te_paths for k in faces[p]])
    tr_emb = fused_embeddings(store, tr_paths, weights).astype(np.float32)
    te_emb, n_group = [], 0
    for p in te_paths:
        e = fused_embeddings(store, faces[p], weights)
        n_group += len(e) > 1
        te_emb.append(e[(e @ tr_emb.T).max(1).argmax()])   # face closest to any enrolled person
    te_emb = np.stack(te_emb).astype(np.float32)
    if n_group:
        print(f"{n_group} test images are group photos (scored by their best-matching face)")

    if args.group:
        np.savez(f"{args.group}.npz", train_ids=np.array(tr_ids), train_emb=tr_emb,
                 test_ids=np.array(te_ids), test_emb=te_emb)
        print(f"saved {args.group}.npz {tr_emb.shape} {te_emb.shape}")

    key = pd.read_csv(args.key) if args.key else None
    known = set(key["person"]) if key is not None else set()
    tr_person = np.array([person_of(i, known) for i in tr_ids])

    sims = te_emb @ tr_emb.T                       # cosine: fused vectors are unit length
    order = np.argsort(-sims, axis=1)
    pred = tr_person[order[:, 0]]
    top5 = [list(dict.fromkeys(tr_person[o]))[:5] for o in order]   # 5 closest distinct people
    out = pd.DataFrame({"test_id": te_ids, "predicted": pred,
                        "score": np.round(sims[np.arange(len(te_ids)), order[:, 0]], 4),
                        "second_best": [t[1] if len(t) > 1 else "" for t in top5]})

    if key is not None:
        truth = dict(zip(key["test_id"].astype(str), key["person"].astype(str)))
        missing = [t for t in te_ids if t not in truth]
        if missing:
            raise SystemExit(f"{len(missing)} test images not in {args.key}, e.g. {missing[:3]}")
        y = np.array([truth[t] for t in te_ids])
        out.insert(1, "true", y)
        out["correct"] = (out["true"] == out["predicted"]).map({True: "yes", False: "NO"})
        labels = sorted(set(y) | set(pred))
        p, r, f1, _ = precision_recall_fscore_support(y, pred, labels=labels, average="macro", zero_division=0)
        pw, rw, f1w, _ = precision_recall_fscore_support(y, pred, labels=labels, average="weighted", zero_division=0)
        top5_acc = np.mean([t in c for t, c in zip(y, top5)])
        print("\n" + out.to_string(index=False))
        print("\n================ RESULTS ================")
        print(f"Test images         : {len(y)}  ({len(set(tr_person))} enrolled people)")
        print(f"Top-1 accuracy      : {accuracy_score(y, pred) * 100:6.2f} %")
        print(f"Top-5 accuracy      : {top5_acc * 100:6.2f} %")
        print(f"Precision (macro)   : {p * 100:6.2f} %    (weighted {pw * 100:6.2f} %)")
        print(f"Recall    (macro)   : {r * 100:6.2f} %    (weighted {rw * 100:6.2f} %)")
        print(f"F1 score  (macro)   : {f1 * 100:6.2f} %    (weighted {f1w * 100:6.2f} %)")
        print("=========================================")
    else:
        print("\n" + out.to_string(index=False))
        print("\n(no --key given, so no metrics: predictions only)")

    out.to_csv(args.out, index=False)
    print(f"wrote {args.out}")


if __name__ == "__main__":
    main()

# Part 2 — Install, test, download models

Installs dependencies (building `insightface` takes ~2-3 min). If Colab asks to restart the session, do it, then re-run **Setup** and continue from here.

In [ ]:
!pip install -q -r requirements.txt

Offline end-to-end test of all the plumbing (synthetic faces, fake models — no downloads needed).

In [ ]:
!python tests/test_end_to_end.py

In [ ]:
!python scripts/00_download_models.py

# Part 3 — Data
Each source below can be **any** of:
* a **Kaggle dataset** id `"owner/dataset-name"` (the part after `kaggle.com/datasets/` in its URL) —
  downloaded straight into this runtime, **no Google Drive needed**;
* a **folder**, e.g. a dataset attached on Kaggle with *Add Input* (`/kaggle/input/...`) or a Drive folder;
* a **.zip** file.

**Our own photos come from the Google Form**: set `FORM_CSV` to the responses CSV (responses spreadsheet →
File → Download → CSV) and `FORM_FILES` to the form's *File responses* folder in Drive. The importer keeps only
consented responses, gives each person a code (P0001 …), converts photos to JPEG (location metadata removed)
and records each photo's condition (glasses, head covering, mask, lighting, …) from the question it was
uploaded to. Names and emails go only to a private code map in your Drive.

| Source | Inside | Used for |
|---|---|---|
| `FORM_CSV` + `FORM_FILES` | the Google Form export (above) | train + **val/test** (only consented Pakistani data is evaluated on) |
| `OWN_SRC` | alternative: `P0001/*.jpg …` (one folder per person) + optional `identities.csv`, `images.csv` | same |
| `IMFDB_SRC` | `<Actor>/*.jpg` (any nesting) | train |
| `HALFFACE_SRC` | halfFace `P_XXX/*.jpg` | train (masks, phones) |

**Kaggle downloads on Colab** need your Kaggle API key once: kaggle.com → Settings → API → *Create New Token*,
then in Colab click the 🔑 *Secrets* icon and add `KAGGLE_USERNAME` and `KAGGLE_KEY` (values from the downloaded
`kaggle.json`). On Kaggle itself nothing is needed.

**Own photos are personal data**: keep them in your private Google Drive (Google Form uploads already are) or a
**private** Kaggle dataset — never public. Empty sources are skipped.

In [ ]:
IMFDB_SRC    = ""   # e.g. "owner/imfdb-dataset" (Kaggle), "/kaggle/input/imfdb", or ".../imfdb.zip"
HALFFACE_SRC = ""
FORM_CSV     = ""   # e.g. "/content/drive/MyDrive/Face Collection (Responses).csv"
FORM_FILES   = ""   # e.g. "/content/drive/MyDrive/Face Collection (File responses)"  (Kaggle: a private dataset)
OWN_SRC      = ""   # only if you already have P0001/... folders instead of the form export
# private code <-> name map: in Drive on Colab so person codes stay the same across sessions
CODE_MAP = "/content/drive/MyDrive/pakface_private/form_code_map.csv" if IN_COLAB else "data/private/form_code_map.csv"

if IN_COLAB and any(str(s).startswith("/content/drive") for s in
                    [IMFDB_SRC, HALFFACE_SRC, OWN_SRC, FORM_CSV, FORM_FILES] + ([CODE_MAP] if FORM_CSV else [])):
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import glob, shutil, subprocess, zipfile
import pandas as pd

IMG = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif")

def unzip(zp, name):
    dst = f"/content/datasets/{name}" if IN_COLAB else os.path.join(PROJECT, "datasets", name)
    if not os.path.exists(dst):
        with zipfile.ZipFile(zp) as z:
            z.extractall(dst)
    return dst

def kaggle_download(slug):
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    if IN_COLAB and not os.environ.get("KAGGLE_KEY"):
        try:
            from google.colab import userdata
            os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
        except Exception as e:
            print("no Kaggle secrets found (🔑 Secrets -> KAGGLE_USERNAME, KAGGLE_KEY):", type(e).__name__)
    return kagglehub.dataset_download(slug)

def fetch(src, name):
    """Kaggle id / folder / zip -> local folder, or None if src is empty."""
    if not src:
        return None
    if str(src).lower().endswith(".zip"):
        return unzip(src, name)
    if os.path.isdir(src):
        return src
    if src.count("/") == 1 and not src.startswith(("/", ".")):
        path = kaggle_download(src)
        zips = glob.glob(os.path.join(path, "*.zip"))
        return unzip(zips[0], name) if zips and not any(os.path.isdir(p) for p in glob.glob(os.path.join(path, "*"))) else path
    raise SystemExit(f"{name}: {src} is not a Kaggle id 'owner/dataset', a folder, or a .zip")

def identity_root(d):
    """Deepest folder whose sub-folders hold the images (one sub-folder per person)."""
    def is_root(p):
        subs = [s for s in glob.glob(os.path.join(p, "*")) if os.path.isdir(s)]
        return len(subs) >= 2 and all(any(f.lower().endswith(IMG) for f in os.listdir(s)) for s in subs[:5])
    for p, _, _ in os.walk(d):
        if is_root(p):
            return p
    raise SystemExit(f"no <person>/<images> folders found under {d}")

def merge_csv(src, dst, key):
    new = pd.read_csv(src)
    if os.path.exists(dst):
        new = pd.concat([pd.read_csv(dst), new]).drop_duplicates(subset=[key], keep="last")
    new.to_csv(dst, index=False)

src = fetch(IMFDB_SRC, "imfdb")
if src:
    root = identity_root(src)
    !python scripts/import_public.py --format imfdb --src "{root}" --name imfdb --copy
else:
    print("skip IMFDB: IMFDB_SRC is empty")

src = fetch(HALFFACE_SRC, "halfface")
if src:
    root = identity_root(src)
    !python scripts/import_public.py --format halfface --src "{root}" --name halfface --copy
else:
    print("skip halfFace: HALFFACE_SRC is empty")

if FORM_CSV:
    files_dir = fetch(FORM_FILES, "form_files")
    !python scripts/import_google_form.py --csv "{FORM_CSV}" --files "{files_dir}" --code-map "{CODE_MAP}"

src = fetch(OWN_SRC, "own_collection")
if src:
    root = identity_root(src)
    shutil.copytree(root, "data/raw/own_collection", dirs_exist_ok=True)
    for name, key in [("identities.csv", "identity"), ("images.csv", None)]:
        hits = glob.glob(os.path.join(src, "**", name), recursive=True)
        if hits:
            merge_csv(hits[0], f"data/{name}", key or pd.read_csv(hits[0]).columns[0])
    print("own_collection:", len(os.listdir("data/raw/own_collection")), "people")
elif not FORM_CSV:
    print("skip own collection: FORM_CSV / OWN_SRC are empty  <- needed for real val/test numbers "
          "(without it, 02_split falls back to a practice split of the public data)")

for s in sorted(glob.glob("data/raw/*")):
    print(f"{os.path.basename(s):16s} {len(os.listdir(s)):5d} identities")

# Part 4 — Train and evaluate
### Sanity check
GPU use, AdaFace channel order (RGB vs BGR, written to the config), quick EER per backbone.

In [ ]:
!python scripts/00_sanity_check.py --data data/raw --ids 40

### Prepare: de-duplicate, detect, align, quality-score every image

In [ ]:
!python scripts/01_prepare_data.py
print(open('data/report_prepare.txt').read())

### Identity-disjoint split
40% of own-collection people go to training (fine-tuning must use our own data); the rest are split val/test. Public datasets are training-only. No person is in two splits.

In [ ]:
!python scripts/02_split.py --eval-sources own_collection

### Baseline: pretrained models, full pipeline
Thresholds and fusion weights are fitted on validation people only (maximising F1, the ranking metric); every metric is reported on unseen test people.

In [ ]:
from IPython.display import Image, Markdown, display

def show(name):
    display(Markdown(open(f"results/{name}/summary.md").read()))
    for f in ["test_roc.png", "test_det.png", "test_hist.png"]:
        display(Image(f"results/{name}/{f}", width=480))

!python scripts/04_evaluate.py --name pretrained
show("pretrained")

### Ablation and stress tests
How much each pipeline stage contributes, and robustness to blur, low resolution, low light, JPEG, CCTV-style and grayscale degradations.

In [ ]:
!python scripts/05_ablation.py
display(Markdown(open('results/ablation/ablation.md').read()))

### Fine-tuning on our Pakistani training identities
Partial unfreezing, frozen BatchNorm, distillation to the original model, heavy degradation + occlusion augmentation (masks, hijab/dupatta, caps, sunglasses). The best epoch is always saved; we ship it only if it beats the pretrained model.

In [ ]:
import json
!python scripts/03_finetune.py --epochs 6 --batch 64
hist = json.load(open("checkpoints/finetune_history.json"))
print({k: v for k, v in hist.items() if k != "epochs"})

In [ ]:
CKPT = "checkpoints/adaface_ir101_finetuned.pt"
!python scripts/05_ablation.py --finetuned {CKPT}
display(Markdown(open("results/ablation/ablation.md").read()))

If the fine-tuned model wins on the **test** split, switch the pipeline to it and re-evaluate. This also rewrites `results/calibration.json`, which the prediction step uses.

In [ ]:
USE_FINETUNED = hist["beat_pretrained"]   # or set True/False by hand after reading the ablation
cfg = json.load(open("configs/pipeline.json"))
for m in cfg["models"]:
    if m["name"] == "adaface_ir101":
        m["finetuned_ckpt"] = CKPT if USE_FINETUNED else ""
json.dump(cfg, open("configs/pipeline.json", "w"), indent=2)
NAME = "finetuned" if USE_FINETUNED else "final_pretrained"
!python scripts/04_evaluate.py --name {NAME}
show(NAME)

### Per-group fairness and hardest errors

In [ ]:
display(pd.read_csv(f"results/{NAME}/per_group.csv"))
display(pd.read_csv(f"results/{NAME}/worst_errors.csv").head(10))

### Save the outputs
Colab deletes the runtime's files when the session ends. This bundles results, the fine-tuned checkpoint and the config into `pakface_outputs.zip`: downloaded on Colab (also copied to Drive if it is mounted), kept as notebook output on Kaggle.

In [ ]:
OUT = os.path.join(PROJECT, "outputs")
for d in ["results", "checkpoints", "configs"]:
    if os.path.exists(d):
        shutil.copytree(d, f"{OUT}/{d}", dirs_exist_ok=True)
for f in ["data/report_prepare.txt", "data/splits.csv"]:
    if os.path.exists(f):
        os.makedirs(f"{OUT}/data", exist_ok=True)
        shutil.copy(f, f"{OUT}/data/")
bundle = shutil.make_archive(os.path.join("/kaggle/working" if IN_KAGGLE else PROJECT, "pakface_outputs"), "zip", OUT)
print("bundle:", bundle)
if IN_COLAB:
    if os.path.isdir("/content/drive/MyDrive"):
        shutil.copy(bundle, "/content/drive/MyDrive/")
        print("copied to Drive: MyDrive/pakface_outputs.zip")
    from google.colab import files
    files.download(bundle)

# Part 5 — Hidden benchmark and demos
Group photos are handled automatically: an image with 2+ faces is split into one entry per face.
Add `--multi-face off` (before the mode) only if every image is known to be single-person.

**Verification pairs** — the CSV needs two image columns (img1/img2, path_a/path_b, … or the first two columns).

In [ ]:
BENCH_CSV, BENCH_ROOT = "/content/bench/pairs.csv", "/content/bench/images"
!python scripts/predict.py pairs --csv {BENCH_CSV} --root {BENCH_ROOT} --out submission.csv
display(pd.read_csv("submission.csv").head())

**Identification** — `gallery/<person>/*.jpg` and a folder of probe images.

In [ ]:
!python scripts/predict.py identify --gallery /content/bench/gallery --probes /content/bench/probes --out identification.csv --open-set
display(pd.read_csv("identification.csv").head())

**Search** — find one person across group / classroom / CCTV photos; matches are drawn in green.

In [ ]:
!python scripts/predict.py search --query /content/demo/person --photos /content/demo/photos --out matches.csv --draw found
for f in sorted(glob.glob("found/*.jpg"))[:4]:
    display(Image(f, width=640))

**Try two of your own images** (Colab: an upload button appears).

In [ ]:
if IN_COLAB:
    from google.colab import files
    up = list(files.upload())
    if len(up) >= 2:
        !python scripts/predict.py one --img1 "{up[0]}" --img2 "{up[1]}"
else:
    print("upload widget is Colab-only; use: !python scripts/predict.py one --img1 a.jpg --img2 b.jpg")

### Live demo app: Nemo's Deep Dive face match
Face Match (live capture or upload two faces → same person or not, with the evidence), Sea of Faces (box one
person in a group photo) and a session Dive Log. Open the **public https link** printed below (the webcam works
through it, also from a phone). The cell keeps running while the app is up: stop it to shut the app down.

In [ ]:
!python app.py --share